# V4 Strategy Institutional Report (Presentation)

Production-safe, read-only institutional monitoring notebook.
Data is sourced from live dashboard tables and runtime signal/trade history only.


In [0]:
%pip install -q joblib


In [0]:
%pip install -q fredapi


In [0]:
import json
from datetime import datetime, timedelta

import importlib.util
import subprocess
import sys
import warnings

import joblib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.backends.backend_pdf import PdfPages
import math
import requests
import os
import re
import textwrap
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 250)
pd.set_option('display.float_format', '{:,.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
_AVAILABLE_FONT_NAMES = {font.name for font in font_manager.fontManager.ttflist}
if 'Times New Roman' in _AVAILABLE_FONT_NAMES:
    REPORT_MATPLOTLIB_FONT = 'Times New Roman'
elif 'DejaVu Serif' in _AVAILABLE_FONT_NAMES:
    REPORT_MATPLOTLIB_FONT = 'DejaVu Serif'
else:
    REPORT_MATPLOTLIB_FONT = 'DejaVu Sans'
plt.rcParams.update({
    'font.family': REPORT_MATPLOTLIB_FONT,
    'font.serif': [REPORT_MATPLOTLIB_FONT],
    'font.sans-serif': [REPORT_MATPLOTLIB_FONT],
    'axes.titlesize': 15,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 120,
})

try:
    display
except NameError:
    def display(obj):
        print(obj)

try:
    displayHTML
except NameError:
    def displayHTML(html):
        print(html)

DEFAULT_REPORT_OUTPUT_PATH = '/Volumes/pawan_quant_research_1/v4/market_data/reports/institutional_dashboard/'


def _get_widget_value(name, default):
    if 'dbutils' not in globals():
        return os.environ.get(name.upper(), default)
    try:
        dbutils.widgets.text(name, default)
    except Exception:
        pass
    try:
        value = dbutils.widgets.get(name)
        return value if value else default
    except Exception:
        return default


REPORT_MODE = _get_widget_value('report_mode', 'interactive').strip().lower()
REPORT_VARIANT = _get_widget_value('report_variant', 'overall').strip().lower() or 'overall'
SHOW_CATALYST_DEBUG = str(_get_widget_value('show_catalyst_debug', 'false')).strip().lower() in {'1', 'true', 'yes', 'y', 'debug'}
REPORT_TIMEZONE = _get_widget_value('timezone', 'Asia/Singapore').strip() or 'Asia/Singapore'
REPORT_OUTPUT_PATH = _get_widget_value('output_path', DEFAULT_REPORT_OUTPUT_PATH).strip() or DEFAULT_REPORT_OUTPUT_PATH
REPORT_TZ = ZoneInfo(REPORT_TIMEZONE)
REPORT_TZ_LABEL = 'SGT' if REPORT_TIMEZONE == 'Asia/Singapore' else REPORT_TIMEZONE
REPORT_VARIANT = 'circulation' if REPORT_VARIANT == 'circulation' else 'overall'


CATALOG = 'pawan_quant_research_1'
SCHEMA = 'v4'

FEED_TABLE = f'{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_feed'
HEALTH_TABLE = f'{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_health_rollup'
PERF_TABLE = f'{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_performance_rollup'
REGIME_SIGNAL_TABLE = f'{CATALOG}.{SCHEMA}.bronze_rt_regime_signal'
SILVER_RT_TABLE = f'{CATALOG}.{SCHEMA}.silver_rt_1h'
SILVER_HIST_TABLE = f'{CATALOG}.{SCHEMA}.silver_bt_master_grid_1h'
BT_MACRO_TABLE = f'{CATALOG}.{SCHEMA}.bronze_bt_macro_events'
LIVE_MACRO_TABLE = f'{CATALOG}.{SCHEMA}.bronze_rt_macro_events_live'
BT_MARKET_TABLE = f'{CATALOG}.{SCHEMA}.bronze_bt_market_1h'
RT_MARKET_TABLE = f'{CATALOG}.{SCHEMA}.bronze_rt_market_1h'
BT_VIX_TABLE = f'{CATALOG}.{SCHEMA}.bronze_bt_vix_1h'
BT_VIX3M_TABLE = f'{CATALOG}.{SCHEMA}.bronze_bt_vix3m_1h'
RT_VIX_TABLE = f'{CATALOG}.{SCHEMA}.bronze_rt_vix_1d'
TRADE_LOG_TABLE = f'{CATALOG}.{SCHEMA}.bronze_rt_saxo_trade_log'
V4_STRATEGY_EQUITY_TABLE = f'{CATALOG}.{SCHEMA}.gold_saxo_v4_strategy_equity_curve'
V4_STRATEGY_LEDGER_TABLE = f'{CATALOG}.{SCHEMA}.gold_saxo_v4_strategy_trade_ledger'
MODEL_REGISTRY_TABLE = f'{CATALOG}.{SCHEMA}.model_registry_current'

LIVE_MODEL_PATH = f'/Volumes/{CATALOG}/{SCHEMA}/market_data/models/bt_fred_1d_xgboost_regime_live.joblib'
REPORT_XGBOOST_VERSION = '2.1.4'
LOOKBACK_HOURS = 360
TOP_FEATURES = 8
FEATURE_PLOT_DAYS = 30
ROLLING_WINDOW_HOURS = 48
PERF_WINDOW_FALLBACK_HOURS = 48
REGIME_WINDOW_FALLBACK_HOURS = 48
FEATURE_WINDOW_FALLBACK_HOURS = 120
HEALTH_WINDOW_FALLBACK_HOURS = 48
WINDOW_MIN_ROWS = 3
PRIMARY_SYMBOL = 'SPY'
VIX_Z_WINDOW_HOURS = 8760
BATCH_Z_HISTORY_ROWS = 8760
LIVE_Z_HISTORY_ROWS = VIX_Z_WINDOW_HOURS - 1
PRICE_CONTEXT_HOURS = 72
EVENT_WINDOW_HOURS = 48
RECENT_TRACKING_FEATURES = ['VIX_Inversion_Z', 'VIXCLS_z', 'vix3m_z', 'PAYEMS_z', 'UNRATE_z', 'Price_Mom_24H']
MARKET_FRESH_HOURS = 36
MARKET_WARN_HOURS = 72
Z_EXACT_TOL = 1e-6
Z_CLOSE_TOL = 1e-4
RATIO_EXACT_TOL = 1e-8
RATIO_CLOSE_TOL = 1e-5
LIVE_MACRO_OVERRIDE_INDICATORS = {'CPIAUCSL', 'UMCSENT'}
DISPLAY_TZ = REPORT_TZ
DISPLAY_TZ_LABEL = REPORT_TZ_LABEL
NEW_YORK_TZ = ZoneInfo('America/New_York')
MONITORING_START_DATE = '2026-04-20'
MONITORING_START_TZ = ZoneInfo('Asia/Kolkata')
MONITORING_START_TS = pd.Timestamp(MONITORING_START_DATE, tz=MONITORING_START_TZ)
MONITORING_START_TS_UTC = MONITORING_START_TS.tz_convert('UTC').tz_localize(None)
MONITORING_START_SQL = MONITORING_START_TS_UTC.strftime('%Y-%m-%d %H:%M:%S')
V4_STRATEGY_PERFORMANCE_START_DATE = '2026-04-20'
V4_STRATEGY_PERFORMANCE_START_SQL = f'{V4_STRATEGY_PERFORMANCE_START_DATE} 00:00:00'
V4_STRATEGY_COST_FOOTNOTE = (
    'V4 strategy-only performance is reconstructed from broker-native SPY/QQQ fills and derived CASH stance transitions. '
    'Other Saxo account activity is excluded. The 5 bps line models a USD-denominated Saxo account trading US ETFs without repeated currency conversion. '
    'Captured-cost performance includes broker-reported conversion cost only and is not confirmed all-in transaction cost.'
)

ALFRED_LOOKBACK_DAYS = 365
ALFRED_SERIES_LOOKBACK_DAYS = 365
ALFRED_RETRY_ATTEMPTS = 3
ALFRED_RETRY_DELAY_SECONDS = 2
ALFRED_PUBLIC_EXPORT_CSV_PATH = '/tmp/alfred_macro_public_export.csv'
ALFRED_EXPORT_LOOKBACK_DAYS = 365
FRED_API_BASE_URL = 'https://api.stlouisfed.org/fred'
FRED_API_TIMEOUT_SECONDS = 30
FRED_FALLBACK_API_KEY = ''

PUBLIC_MACRO_SERIES_FEATURE_MAP = {
    'CPIAUCSL': 'CPI',
    'PAYEMS': 'Payrolls',
    'UNRATE': 'Unemployment',
    'INDPRO': 'Industrial Production',
    'GDPC1': 'GDP',
    'PCEPI': 'Core PCE',
    'PCE': 'PCE',
}

REQUIRED_PUBLIC_MACRO_SERIES = ['CPIAUCSL', 'PAYEMS', 'UNRATE', 'INDPRO', 'GDPC1']
OPTIONAL_PUBLIC_MACRO_SERIES = ['PCEPI', 'PCE']
RAW_FEATURE_SPECS = {
    'CPIAUCSL_z': {'type': 'macro', 'indicator_id': 'CPIAUCSL', 'live_override': False},
    'PCEPI_z': {'type': 'macro', 'indicator_id': 'PCEPI', 'live_override': False},
    'PCE_z': {'type': 'macro', 'indicator_id': 'PCE', 'live_override': False},
    'PAYEMS_z': {'type': 'macro', 'indicator_id': 'PAYEMS', 'live_override': False},
    'UNRATE_z': {'type': 'macro', 'indicator_id': 'UNRATE', 'live_override': False},
    'GDPC1_z': {'type': 'macro', 'indicator_id': 'GDPC1', 'live_override': False},
    'INDPRO_z': {'type': 'macro', 'indicator_id': 'INDPRO', 'live_override': False},
    'UMCSENT_z': {'type': 'macro', 'indicator_id': 'UMCSENT', 'live_override': True},
    'VIXCLS_z': {'type': 'rt_vix', 'series': 'VIXCLS', 'source_field': 'close'},
    'vix3m_z': {'type': 'rt_vix', 'series': 'vix3m', 'source_field': 'close'},
    'VIX_Inversion_Z': {'type': 'derived', 'source_table': RT_VIX_TABLE, 'source_field': 'close', 'note': 'Derived from RT VIX spot and VIX 3M inputs; direct raw scalar not displayed'},
    'Price_Mom_24H': {'type': 'derived', 'source_table': f'{RT_MARKET_TABLE} + historical SPY context', 'source_field': 'close', 'note': 'Derived from realtime SPY close and 24H lag context; direct raw scalar not displayed'},
    'CPI_Event_Active': {'type': 'derived', 'source_table': SILVER_HIST_TABLE, 'source_field': 'CPI_Event_Active', 'note': 'Derived event flag carried forward from silver history'},
    'NFP_Event_Active': {'type': 'derived', 'source_table': SILVER_HIST_TABLE, 'source_field': 'NFP_Event_Active', 'note': 'Derived event flag carried forward from silver history'},
}
RAW_FIELD_CANDIDATES = {
    'CPIAUCSL_z': [('silver', 'CPIAUCSL'), ('silver', 'CPIAUCSL_raw')],
    'PCEPI_z': [('silver', 'PCEPI'), ('silver', 'PCEPI_raw')],
    'PCE_z': [('silver', 'PCE'), ('silver', 'PCE_raw')],
    'PAYEMS_z': [('silver', 'PAYEMS'), ('silver', 'PAYEMS_raw')],
    'UNRATE_z': [('silver', 'UNRATE'), ('silver', 'UNRATE_raw')],
    'GDPC1_z': [('silver', 'GDPC1'), ('silver', 'GDPC1_raw')],
    'INDPRO_z': [('silver', 'INDPRO'), ('silver', 'INDPRO_raw')],
    'UMCSENT_z': [('silver', 'UMCSENT'), ('silver', 'UMCSENT_raw')],
    'VIXCLS_z': [('silver', 'VIXCLS'), ('silver', 'VIXCLS_raw')],
    'vix3m_z': [('silver', 'vix3m'), ('silver', 'VIX3M'), ('silver', 'vix3m_raw')],
    'VIX_Inversion_Z': [('silver', 'VIX_Inversion'), ('silver', 'vix_inversion'), ('silver', 'VIX_Inversion_Raw')],
    'Price_Mom_24H': [('silver', 'Price_Mom_24H'), ('silver', 'Price_Mom_24H_raw')],
    'CPI_Event_Active': [('silver', 'CPI_Event_Active')],
    'NFP_Event_Active': [('silver', 'NFP_Event_Active')],
}

FEATURES = [
    'CPIAUCSL_z',
    'PCEPI_z',
    'PCE_z',
    'PAYEMS_z',
    'UNRATE_z',
    'GDPC1_z',
    'INDPRO_z',
    'UMCSENT_z',
    'VIXCLS_z',
    'vix3m_z',
    'VIX_Inversion_Z',
    'Price_Mom_24H',
    'CPI_Event_Active',
    'NFP_Event_Active',
]

FEATURE_LABELS = {
    'CPIAUCSL_z': 'CPI',
    'PCEPI_z': 'Core PCE',
    'PCE_z': 'PCE',
    'PAYEMS_z': 'Payrolls',
    'UNRATE_z': 'Unemployment',
    'GDPC1_z': 'GDP',
    'INDPRO_z': 'Industrial Production',
    'UMCSENT_z': 'Sentiment',
    'VIXCLS_z': 'VIX Spot',
    'vix3m_z': 'VIX 3M',
    'VIX_Inversion_Z': 'VIX Inversion',
    'Price_Mom_24H': '24H Price Momentum',
    'CPI_Event_Active': 'CPI Event',
    'NFP_Event_Active': 'NFP Event',
}

REGIME_COLOR = {'Expansion': '#2E8B57', 'Slowdown': '#D4A017', 'Recession': '#C0392B'}
REGIME_NAME_MAP = {0: 'Expansion', 1: 'Slowdown', 2: 'Recession'}
PROBABILITY_COLUMNS = {0: 'prob_expansion', 1: 'prob_slowdown', 2: 'prob_recession'}
PROBABILITY_LABELS = {'prob_expansion': 'Expansion', 'prob_slowdown': 'Slowdown', 'prob_recession': 'Recession'}
HISTORICAL_MIX_COLUMNS = {'pct_expansion': 'Expansion', 'pct_slowdown': 'Slowdown', 'pct_recession': 'Recession'}
FEATURE_MONITORING_GROUPS = {
    'Inflation': ['CPIAUCSL_z', 'PCEPI_z', 'PCE_z'],
    'Growth': ['GDPC1_z', 'INDPRO_z'],
    'Employment': ['PAYEMS_z', 'UNRATE_z'],
    'Sentiment': ['UMCSENT_z'],
    'Volatility': ['VIXCLS_z', 'vix3m_z', 'VIX_Inversion_Z'],
    'Price Action': ['Price_Mom_24H'],
    'Event Flags': ['CPI_Event_Active', 'NFP_Event_Active'],
}

def resolve_fred_api_key():
    # Retain optional environment/secret lookup for local safety, but keep
    # an explicit fallback key for this notebook's canonical FRED/ALFRED lookup flow.
    key = os.environ.get('FRED_API_KEY', '').strip()
    if key:
        return key

    if 'dbutils' in globals():
        scope_candidates = ['api', 'secrets', 'fred', 'v4', 'market_data', 'production']
        key_candidates = ['fred_api_key', 'fred', 'FRED_API_KEY', 'fred_key']
        for scope in scope_candidates:
            for key_name in key_candidates:
                try:
                    candidate = dbutils.secrets.get(scope=scope, key=key_name)
                    if isinstance(candidate, str) and candidate.strip():
                        return candidate.strip()
                except Exception:
                    pass

    return FRED_FALLBACK_API_KEY


# FRED_API_KEY = resolve_fred_api_key()
FRED_API_KEY = "f52bdf7341f23d0458820ecd806178db"

try:
    from fredapi import Fred  # type: ignore
    _fred_client = Fred(api_key=FRED_API_KEY) if FRED_API_KEY else None
except Exception:
    _fred_client = None

_ALFRED_RELEASE_CACHE = {}

def _safe_float(value):
    if value is None:
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    text = str(value).replace(',', '').strip()
    if not text or text == 'N/A':
        return np.nan
    try:
        parsed = float(text)
        return parsed if parsed == parsed else np.nan
    except Exception:
        return np.nan


def _to_float(value):
    if value is None:
        return np.nan
    if isinstance(value, pd.Series):
        value = value.dropna()
        return float(value.iloc[0]) if not value.empty else np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value) if value == value else np.nan
    try:
        parsed = float(value)
        return parsed if parsed == parsed else np.nan
    except Exception:
        return np.nan


def _is_missing(value):
    if value is None:
        return True
    try:
        missing = pd.isna(value)
    except Exception:
        return False
    if isinstance(missing, (bool, int, float)):
        return bool(missing)
    return False


def _as_text(value, default='N/A'):
    return default if _is_missing(value) else str(value)


def fmt_money(value):
    value = _to_float(value)
    return 'N/A' if pd.isna(value) else f'${value:,.2f}'


def fmt_pct(value):
    value = _to_float(value)
    return 'N/A' if pd.isna(value) else f'{value:+,.2%}'


def fmt_percent_points(value):
    value = _to_float(value)
    return 'N/A' if pd.isna(value) else f'{value:.1f}%'


def fmt_optional_currency(value):
    value = _to_float(value)
    if pd.isna(value):
        return 'N/A'
    return f'{value:,.2f}'


def fmt_optional_percent(value):
    value = _to_float(value)
    if pd.isna(value):
        return 'N/A'
    return f'{value:+.2%}'


def fmt_signed_money(value):
    value = _to_float(value)
    if pd.isna(value):
        return 'N/A'
    sign = '+' if value >= 0 else '-'
    return f'{sign}${abs(value):,.2f}'


def fmt_flag(value):
    if _is_missing(value):
        return 'N/A'
    if isinstance(value, str):
        normalized = value.strip().lower()
        if normalized in {'true', 'yes', 'y', '1'}:
            return 'Yes'
        if normalized in {'false', 'no', 'n', '0'}:
            return 'No'
    return 'Yes' if bool(value) else 'No'


def fmt_raw_snapshot_value(value):
    if _is_missing(value):
        return 'N/A'
    if isinstance(value, str):
        return value
    numeric = _to_float(value)
    if pd.notna(numeric):
        if numeric in (0.0, 1.0):
            return fmt_flag(int(numeric))
        return f'{numeric:,.4f}'
    return str(value)


def fmt_minutes(value):
    value = _to_float(value)
    return 'N/A' if pd.isna(value) else f'{value:.0f}'


def fmt_freshness_value(value):
    value = _to_float(value)
    if pd.isna(value):
        return 'N/A'
    if value <= 0:
        return 'current'
    if value < 120:
        return f'{value:.0f} min'
    if value < 2880:
        return f'{value / 60.0:.1f}h'
    return f'{value / 1440.0:.1f}d'


def fmt_freshness_summary(row):
    market_value = fmt_freshness_value(row.get('freshness_market_minutes'))
    macro_value = fmt_freshness_value(row.get('freshness_macro_minutes'))
    audit_minutes = _to_float(row.get('freshness_audit_minutes'))
    audit_value = 'current' if pd.notna(audit_minutes) and audit_minutes <= 1 else fmt_freshness_value(audit_minutes)
    return f'Market: {market_value} | Macro: {macro_value} | Audit: {audit_value}'


def fmt_audit_summary(row):
    audit_ts = fmt_timestamp(row.get('latest_audit_ts'))
    action = _as_text(row.get('last_execution_action'))
    if action == 'N/A':
        return audit_ts
    return f'{audit_ts} | Last action: {action}'


def to_display_ist_timestamp(value):
    ts = pd.to_datetime(value, errors='coerce')
    if pd.isna(ts):
        return pd.NaT
    if getattr(ts, 'tzinfo', None) is None:
        return ts.tz_localize('UTC').tz_convert(DISPLAY_TZ)
    return ts.tz_convert(DISPLAY_TZ)


def to_display_ist_series(values):
    ts = pd.to_datetime(values, errors='coerce')
    if isinstance(ts, pd.Series):
        try:
            if ts.dt.tz is None:
                return ts.dt.tz_localize('UTC').dt.tz_convert(DISPLAY_TZ)
            return ts.dt.tz_convert(DISPLAY_TZ)
        except Exception:
            return ts
    if isinstance(ts, pd.DatetimeIndex):
        if ts.tz is None:
            return ts.tz_localize('UTC').tz_convert(DISPLAY_TZ)
        return ts.tz_convert(DISPLAY_TZ)
    return to_display_ist_timestamp(ts)


def fmt_timestamp(value):
    ts = to_display_ist_timestamp(value)
    return 'N/A' if pd.isna(ts) else f"{ts.strftime('%Y-%m-%d %H:%M:%S')} {DISPLAY_TZ_LABEL}"


def _ordinal_day(value):
    try:
        day = int(value)
    except Exception:
        return str(value)
    default_suffix = 'th'
    if 10 <= day % 100 <= 20:
        suffix = default_suffix
    else:
        suffix_map = {1: 'st', 2: 'nd', 3: 'rd'}
        suffix = suffix_map.get(day % 10, default_suffix)
    return f'{day}{suffix}'


def fmt_timestamp_long(value, timezone=None, label=None):
    ts = pd.to_datetime(value, errors='coerce')
    if pd.isna(ts):
        return 'N/A'
    if getattr(ts, 'tzinfo', None) is None:
        ts = ts.tz_localize('UTC')
    target_tz = timezone or DISPLAY_TZ
    ts = ts.tz_convert(target_tz)
    tz_label = label or ts.strftime('%Z')
    return f"{_ordinal_day(ts.day)} {ts.strftime('%B %Y, %H:%M')} {tz_label}"


def fmt_timestamp_pair(value):
    ts = pd.to_datetime(value, errors='coerce')
    if pd.isna(ts):
        return 'N/A'
    if getattr(ts, 'tzinfo', None) is None:
        ts = ts.tz_localize('UTC')
    ny_ts = ts.tz_convert(NEW_YORK_TZ)
    ny_label = 'EDT' if ny_ts.dst() != pd.Timedelta(0) else 'EST'
    display_text = fmt_timestamp_long(ts, DISPLAY_TZ, DISPLAY_TZ_LABEL)
    ny_text = fmt_timestamp_long(ts, NEW_YORK_TZ, ny_label)
    return f'{display_text}\n{ny_text}'


def _split_paired_timestamp_text(value):
    text = _as_text(value)
    if text == 'N/A':
        return 'N/A', 'N/A'
    if '\n' in text:
        parts = [part.strip() for part in text.split('\n', 1)]
        if len(parts) == 2:
            return parts[0], parts[1]
    if ' | ' in text:
        parts = [part.strip() for part in text.split(' | ', 1)]
        if len(parts) == 2:
            return parts[0], parts[1]
    return text, 'N/A'


def _format_paired_timestamp_cell(value):
    sgt_text, ny_text = _split_paired_timestamp_text(value)
    if sgt_text != 'N/A' and ny_text != 'N/A':
        return f'{sgt_text}\n{ny_text}'
    return fmt_timestamp_pair(value)


def _safe_split_session_timestamp(value):
    ts = pd.to_datetime(value, errors='coerce')
    if pd.isna(ts):
        return 'N/A', 'N/A'
    if getattr(ts, 'tzinfo', None) is None:
        ts = ts.tz_localize('UTC')
    sgt = fmt_timestamp_long(ts, DISPLAY_TZ, DISPLAY_TZ_LABEL)
    ny_ts = ts.tz_convert(NEW_YORK_TZ)
    ny_label = 'EDT' if ny_ts.dst() != pd.Timedelta(0) else 'EST'
    edt = fmt_timestamp_long(ts, NEW_YORK_TZ, ny_label)
    return sgt, edt


def fmt_date(value):
    ts = pd.to_datetime(value, errors='coerce')
    return 'N/A' if pd.isna(ts) else ts.strftime('%Y-%m-%d')


def format_duration(delta):
    if delta is None or pd.isna(delta):
        return 'N/A'
    if not isinstance(delta, pd.Timedelta):
        delta = pd.Timedelta(delta)
    total_minutes = max(int(delta.total_seconds() // 60), 0)
    hours, minutes = divmod(total_minutes, 60)
    days, hours = divmod(hours, 24)
    if days > 0:
        return f'{days}d {hours}h {minutes}m'
    return f'{hours}h {minutes}m'


def next_regular_market_open(now_ny):
    session_open = now_ny.normalize() + pd.Timedelta(hours=9, minutes=30)
    session_close = now_ny.normalize() + pd.Timedelta(hours=16)
    if now_ny.dayofweek < 5 and now_ny < session_open:
        return session_open
    if now_ny.dayofweek < 5 and session_open <= now_ny < session_close:
        return session_open
    next_open = now_ny.normalize() + pd.Timedelta(days=1)
    while next_open.dayofweek >= 5:
        next_open += pd.Timedelta(days=1)
    return next_open + pd.Timedelta(hours=9, minutes=30)


def build_market_status_snapshot():
    now_ny = pd.Timestamp.now(tz=NEW_YORK_TZ)
    now_display = now_ny.tz_convert(DISPLAY_TZ)
    session_open = now_ny.normalize() + pd.Timedelta(hours=9, minutes=30)
    session_close = now_ny.normalize() + pd.Timedelta(hours=16)
    market_is_open = now_ny.dayofweek < 5 and session_open <= now_ny < session_close
    if market_is_open:
        time_label = 'Time until close'
        time_value = format_duration(session_close - now_ny)
        status = 'Open'
    else:
        next_open = next_regular_market_open(now_ny)
        time_label = 'Time until next open'
        time_value = format_duration(next_open - now_ny)
        status = 'Closed'
    return {
        'New York time': fmt_timestamp_long(now_ny, NEW_YORK_TZ, 'EDT' if now_ny.dst() != pd.Timedelta(0) else 'EST'),
        DISPLAY_TZ_LABEL: fmt_timestamp_long(now_display, DISPLAY_TZ, DISPLAY_TZ_LABEL),
        'Market status': status,
        time_label: time_value,
        'Status basis': 'Regular US equity session, Monday to Friday, 09:30-16:00 New York time',
    }


def apply_ist_axis(ax, xlabel=None, maxticks=None):
    if xlabel is None:
        xlabel = f'Timestamp ({DISPLAY_TZ_LABEL})'
    locator = mdates.AutoDateLocator(tz=DISPLAY_TZ, maxticks=maxticks) if maxticks else mdates.AutoDateLocator(tz=DISPLAY_TZ)
    formatter = mdates.ConciseDateFormatter(locator, tz=DISPLAY_TZ)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)
    ax.set_xlabel(xlabel)
    ax.tick_params(axis='x', rotation=0)


def apply_dual_time_axes(ax, bottom_label=None, top_label='Timestamp (EDT)', maxticks=None, top_labelsize=8):
    apply_ist_axis(ax, xlabel=bottom_label, maxticks=maxticks)
    top_ax = ax.twiny()
    top_ax.set_xlim(ax.get_xlim())
    locator = mdates.AutoDateLocator(tz=NEW_YORK_TZ, maxticks=maxticks) if maxticks else mdates.AutoDateLocator(tz=NEW_YORK_TZ)
    formatter = mdates.ConciseDateFormatter(locator, tz=NEW_YORK_TZ)
    top_ax.xaxis.set_major_locator(locator)
    top_ax.xaxis.set_major_formatter(formatter)
    top_ax.set_xlabel(top_label)
    top_ax.tick_params(axis='x', rotation=0, labelsize=top_labelsize)
    return top_ax


def latest_completed_session_bounds(reference_ts=None):
    now_ny = pd.Timestamp.now(tz=NEW_YORK_TZ) if reference_ts is None else pd.Timestamp(reference_ts).tz_convert(NEW_YORK_TZ)
    session_day = now_ny.normalize()
    current_session_close = session_day + pd.Timedelta(hours=16)
    if now_ny < current_session_close:
        session_day -= pd.Timedelta(days=1)
    while session_day.dayofweek >= 5:
        session_day -= pd.Timedelta(days=1)
    session_start = session_day + pd.Timedelta(hours=9, minutes=30)
    session_end = session_day + pd.Timedelta(hours=16)
    return session_start, session_end


def filter_to_completed_session(frame, timestamp_col):
    if frame is None or getattr(frame, 'empty', True) or timestamp_col not in frame.columns:
        return frame.copy() if frame is not None else pd.DataFrame()
    session_start_ny, session_end_ny = latest_completed_session_bounds()
    ts = pd.to_datetime(frame[timestamp_col], errors='coerce')
    if getattr(ts.dt, 'tz', None) is None:
        ts = ts.dt.tz_localize(DISPLAY_TZ)
    ts_ny = ts.dt.tz_convert(NEW_YORK_TZ)
    mask = (ts_ny >= session_start_ny) & (ts_ny <= session_end_ny)
    return frame.loc[mask].copy()




OPEN_CONTEXT_TARGET_LAG = pd.Timedelta(hours=1)
OPEN_CONTEXT_EXACT_TOLERANCE = pd.Timedelta(minutes=2)
OPEN_CONTEXT_REASONABLE_TOLERANCE = pd.Timedelta(minutes=90)


def _append_open_context_row(session_frame, full_frame, timestamp_col, session_start_ny):
    session_frame = session_frame.copy() if session_frame is not None else pd.DataFrame()
    if timestamp_col not in getattr(full_frame, 'columns', []):
        if not session_frame.empty:
            session_frame['Context'] = 'Session row'
        return session_frame
    full = full_frame.copy()
    ts = pd.to_datetime(full[timestamp_col], errors='coerce')
    if getattr(ts.dt, 'tz', None) is None:
        ts = ts.dt.tz_localize(DISPLAY_TZ)
    ts_ny = ts.dt.tz_convert(NEW_YORK_TZ)
    full['_open_context_ts_ny'] = ts_ny
    if session_frame.empty:
        session_frame = pd.DataFrame(columns=[c for c in full_frame.columns if c != '_open_context_ts_ny'])
    session_frame = session_frame.copy()
    session_frame['Context'] = 'Session row'
    session_ts = pd.to_datetime(session_frame[timestamp_col], errors='coerce') if timestamp_col in session_frame.columns else pd.Series(dtype='datetime64[ns]')
    if len(session_ts) and getattr(session_ts.dt, 'tz', None) is None:
        session_ts = session_ts.dt.tz_localize(DISPLAY_TZ)
    session_ts_ny = session_ts.dt.tz_convert(NEW_YORK_TZ) if len(session_ts) else pd.Series(dtype='datetime64[ns, America/New_York]')
    first_session_ts_ny = session_ts_ny.dropna().min() if len(session_ts_ny.dropna()) else session_start_ny
    target_ts_ny = first_session_ts_ny - OPEN_CONTEXT_TARGET_LAG
    candidates = full[(full['_open_context_ts_ny'] < first_session_ts_ny) & full['_open_context_ts_ny'].notna()].copy()
    if candidates.empty:
        return session_frame
    candidates['_open_context_abs_delta'] = (candidates['_open_context_ts_ny'] - target_ts_ny).abs()
    exact_candidates = candidates[candidates['_open_context_abs_delta'] <= OPEN_CONTEXT_EXACT_TOLERANCE].copy()
    if exact_candidates.empty:
        before_target = candidates[candidates['_open_context_ts_ny'] <= target_ts_ny].copy()
        if not before_target.empty and before_target['_open_context_abs_delta'].min() <= OPEN_CONTEXT_REASONABLE_TOLERANCE:
            open_row = before_target.sort_values('_open_context_ts_ny').tail(1)
            open_context_label = 'Nearest open context'
        else:
            nearest_candidates = candidates[candidates['_open_context_abs_delta'] <= OPEN_CONTEXT_REASONABLE_TOLERANCE].copy()
            if nearest_candidates.empty:
                return session_frame
            open_row = nearest_candidates.sort_values('_open_context_abs_delta').head(1)
            open_context_label = 'Nearest open context'
    else:
        open_row = exact_candidates.sort_values('_open_context_abs_delta').head(1)
        open_context_label = 'Open context (-1h)'
    open_row = open_row.drop(columns=['_open_context_ts_ny', '_open_context_abs_delta'], errors='ignore')
    if open_row.empty:
        return session_frame
    open_row = open_row.copy()
    open_row['Context'] = open_context_label
    if timestamp_col in session_frame.columns:
        existing_ts = pd.to_datetime(session_frame[timestamp_col], errors='coerce')
        open_ts = pd.to_datetime(open_row[timestamp_col].iloc[0], errors='coerce')
        duplicate_mask = existing_ts.eq(open_ts)
        if duplicate_mask.any():
            session_frame.loc[duplicate_mask, 'Context'] = open_context_label
            return session_frame
    return pd.concat([session_frame, open_row], ignore_index=True, sort=False)

def _to_naive_utc(value):
    if isinstance(value, pd.Series):
        ts = pd.to_datetime(value, errors='coerce')
        if ts.dtype == 'datetime64[ns]':
            return ts
        if pd.api.types.is_datetime64tz_dtype(ts.dtype):
            return ts.dt.tz_convert('UTC').dt.tz_localize(None)
        return ts

    if isinstance(value, pd.DatetimeIndex):
        ts = pd.to_datetime(value, errors='coerce')
        if ts.tz is not None:
            return ts.tz_convert('UTC').tz_localize(None)
        return ts

    ts = pd.to_datetime(value, errors='coerce')
    if pd.isna(ts):
        return pd.NaT
    if getattr(ts, 'tzinfo', None) is not None:
        return ts.tz_convert('UTC').tz_localize(None)
    return ts

def _coerce_public_timestamp(value):
    return _to_naive_utc(value)


def _coerce_public_series_timestamps(values):
    return _to_naive_utc(values)

def _coerce_alfred_timestamps(series):
    for column in ['observation_date', 'realtime_start', 'realtime_end']:
        if column in series.columns:
            series[column] = _to_naive_utc(series[column])
        else:
            series[column] = pd.NaT
    return series


def _ensure_alfred_columns(vintages, source_label='ALFRED'):
    if vintages is None:
        return pd.DataFrame(columns=['series_id', 'observation_date', 'realtime_start', 'realtime_end', 'public_value', 'public_source'])

    vintages = vintages.copy()
    if 'date' in vintages.columns and 'observation_date' not in vintages.columns:
        vintages = vintages.rename(columns={'date': 'observation_date'})

    if 'realtime_start' not in vintages.columns:
        vintages['realtime_start'] = pd.NaT
    if 'realtime_end' not in vintages.columns:
        vintages['realtime_end'] = pd.NaT

    if 'value' not in vintages.columns:
        if 'series_value' in vintages.columns:
            vintages['value'] = vintages['series_value']
        else:
            vintages['value'] = np.nan

    vintages['public_value'] = pd.to_numeric(vintages['value'], errors='coerce')
    vintages['public_source'] = source_label

    return vintages


def _load_alfred_releases(series_id):
    if series_id in _ALFRED_RELEASE_CACHE:
        return _ALFRED_RELEASE_CACHE[series_id]

    empty_df = pd.DataFrame(columns=['series_id', 'observation_date', 'realtime_start', 'realtime_end', 'public_value', 'public_source'])
    retry_attempts = globals().get('ALFRED_RETRY_ATTEMPTS', 3)
    retry_delay_seconds = globals().get('ALFRED_RETRY_DELAY_SECONDS', 2)

    if not series_id:
        _ALFRED_RELEASE_CACHE[series_id] = empty_df
        return empty_df

    if _fred_client is not None:
        last_error = None
        for attempt in range(retry_attempts):
            try:
                vintages = _fred_client.get_series_all_releases(series_id)
                if vintages is None or getattr(vintages, 'empty', True):
                    break
                if 'realtime_end' not in vintages.columns:
                    vintages['realtime_end'] = pd.NaT
                if 'realtime_start' in vintages.columns:
                    vintages['realtime_start'] = pd.to_datetime(vintages['realtime_start'], errors='coerce')
                else:
                    vintages['realtime_start'] = pd.NaT
                if 'observation_date' not in vintages.columns and 'date' in vintages.columns:
                    vintages = vintages.rename(columns={'date': 'observation_date'})
                if 'value' not in vintages.columns and 'series_value' in vintages.columns:
                    vintages['value'] = vintages['series_value']
                if 'value' in vintages.columns:
                    vintages['public_value'] = pd.to_numeric(vintages['value'], errors='coerce')
                else:
                    vintages['public_value'] = np.nan
                vintages['public_source'] = 'ALFRED'
                if 'observation_date' not in vintages.columns:
                    break
                vintages = _ensure_alfred_columns(vintages, source_label='ALFRED')
                vintages = _coerce_alfred_timestamps(vintages)
                vintages['series_id'] = series_id
                result = vintages[['series_id', 'observation_date', 'realtime_start', 'realtime_end', 'public_value', 'public_source']]
                result = result.dropna(subset=['observation_date']).sort_values(['observation_date', 'realtime_start'], ascending=[True, True]).reset_index(drop=True)
                _ALFRED_RELEASE_CACHE[series_id] = result
                return result
            except Exception as exc:
                last_error = exc
                if attempt + 1 < retry_attempts:
                    import time
                    time.sleep(retry_delay_seconds * (2 ** attempt))
        if last_error is not None:
            print(f'ALFRED fetch failed for {series_id}: {last_error}')

    if FRED_API_KEY:
        try:
            today = pd.Timestamp.utcnow().normalize()
            start = today - pd.Timedelta(days=ALFRED_LOOKBACK_DAYS)
            payload = requests.get(
                f"{FRED_API_BASE_URL}/series/observations",
                params={
                    'series_id': series_id,
                    'api_key': FRED_API_KEY,
                    'file_type': 'json',
                    'observation_start': start.strftime('%Y-%m-%d'),
                    'observation_end': today.strftime('%Y-%m-%d'),
                    'sort_order': 'desc',
                    'limit': 12000,
                },
                timeout=FRED_API_TIMEOUT_SECONDS,
            )
            payload.raise_for_status()
            payload_json = payload.json()
            observations = payload_json.get('observations', [])
            if not observations:
                _ALFRED_RELEASE_CACHE[series_id] = empty_df
                return empty_df

            df = pd.DataFrame(observations)
            if df.empty or 'date' not in df.columns:
                _ALFRED_RELEASE_CACHE[series_id] = empty_df
                return empty_df

            df = df.rename(columns={'date': 'observation_date'})
            df = _ensure_alfred_columns(df, source_label='FRED')
            if 'value' in df.columns:
                df['public_value'] = pd.to_numeric(df['value'], errors='coerce')
            else:
                df['public_value'] = np.nan
            df['series_id'] = series_id
            df['realtime_start'] = pd.NaT
            df['realtime_end'] = pd.NaT
            df['public_source'] = 'FRED'
            df = _coerce_alfred_timestamps(df)
            df = df[['series_id', 'observation_date', 'realtime_start', 'realtime_end', 'public_value', 'public_source']]
            result = df.dropna(subset=['observation_date']).sort_values('observation_date', ascending=False).reset_index(drop=True)
            _ALFRED_RELEASE_CACHE[series_id] = result
            return result
        except Exception as exc:
            print(f'FRED fallback fetch failed for {series_id}: {exc}')

    _ALFRED_RELEASE_CACHE[series_id] = empty_df
    return empty_df


def fetch_alfred_public_lookup(series_id, lookback_days=ALFRED_SERIES_LOOKBACK_DAYS):
    history = _load_alfred_releases(series_id)
    if history.empty:
        return history

    cutoff = _to_naive_utc(pd.Timestamp.utcnow() - pd.Timedelta(days=lookback_days))
    selected = history[history['observation_date'] >= cutoff].copy()
    return selected.sort_values(['observation_date', 'realtime_start'], ascending=[False, False]).reset_index(drop=True)




def _evaluate_alfred_observation_selection(feature, series_id, feature_label, model_feature_ts):
    if pd.isna(model_feature_ts):
        return {
            'Feature': feature_label,
            'series_id': series_id,
            'selected_release_timestamp': pd.NaT,
            'selected_observation_date': pd.NaT,
            'max_observation_date_at_release': pd.NaT,
            'selected_value': np.nan,
            'latest_value_at_release': np.nan,
            'selected_matches_latest_observation_date': False,
            'selected_matches_latest_value': False,
            'selection_status': 'unresolved',
            'note': 'No model timestamp available to trace lineage.',
        }

    silver_bt_row = load_silver_bt_macro_row(model_feature_ts, feature)
    if silver_bt_row.empty:
        return {
            'Feature': feature_label,
            'series_id': series_id,
            'selected_release_timestamp': pd.NaT,
            'selected_observation_date': pd.NaT,
            'max_observation_date_at_release': pd.NaT,
            'selected_value': np.nan,
            'latest_value_at_release': np.nan,
            'selected_matches_latest_observation_date': False,
            'selected_matches_latest_value': False,
            'selection_status': 'unresolved',
            'note': 'No carry-forward silver_bt row was available for current model timestamp.',
        }

    silver_bt_ts = silver_bt_row.get('timestamp')
    upstream_release = load_macro_release_asof(series_id, silver_bt_ts)
    selected_release_ts = pd.to_datetime(upstream_release.get('source_ts'), errors='coerce')
    selected_value = _to_float(upstream_release.get('raw_value'))
    if pd.isna(selected_release_ts) or pd.isna(selected_value):
        return {
            'Feature': feature_label,
            'series_id': series_id,
            'selected_release_timestamp': pd.NaT,
            'selected_observation_date': pd.NaT,
            'max_observation_date_at_release': pd.NaT,
            'selected_value': selected_value,
            'latest_value_at_release': np.nan,
            'selected_matches_latest_observation_date': False,
            'selected_matches_latest_value': False,
            'selection_status': 'unresolved',
            'note': 'Could not locate upstream macro release selected by silver_bt lineage.',
        }

    alfred_df = _load_alfred_releases(series_id)
    if alfred_df.empty:
        return {
            'Feature': feature_label,
            'series_id': series_id,
            'selected_release_timestamp': selected_release_ts,
            'selected_observation_date': pd.NaT,
            'max_observation_date_at_release': pd.NaT,
            'selected_value': selected_value,
            'latest_value_at_release': np.nan,
            'selected_matches_latest_observation_date': False,
            'selected_matches_latest_value': False,
            'selection_status': 'unresolved',
            'note': 'No ALFRED/FRED vintage rows were available for this series.',
        }

    alfred_df = alfred_df.copy()
    alfred_df['observation_date'] = _to_naive_utc(alfred_df['observation_date'])
    alfred_df['realtime_start'] = _to_naive_utc(alfred_df['realtime_start'])
    alfred_df['realtime_end'] = _to_naive_utc(alfred_df['realtime_end'])
    alfred_df['public_value'] = pd.to_numeric(alfred_df['public_value'], errors='coerce')
    alfred_df = alfred_df.dropna(subset=['observation_date', 'public_value'])
    if alfred_df.empty:
        return {
            'Feature': feature_label,
            'series_id': series_id,
            'selected_release_timestamp': selected_release_ts,
            'selected_observation_date': pd.NaT,
            'max_observation_date_at_release': pd.NaT,
            'selected_value': selected_value,
            'latest_value_at_release': np.nan,
            'selected_matches_latest_observation_date': False,
            'selected_matches_latest_value': False,
            'selection_status': 'unresolved',
            'note': 'ALFRED/FRED rows were returned without usable public values.',
        }

    release_rows = alfred_df.copy()
    has_release_window = release_rows['realtime_start'].notna()
    if has_release_window.any():
        release_rows = release_rows[has_release_window]
        release_rows = release_rows[(release_rows['realtime_start'] <= selected_release_ts)]
        release_rows = release_rows[(release_rows['realtime_end'].isna()) | (selected_release_ts < release_rows['realtime_end'])]

    if release_rows.empty:
        return {
            'Feature': feature_label,
            'series_id': series_id,
            'selected_release_timestamp': selected_release_ts,
            'selected_observation_date': pd.NaT,
            'max_observation_date_at_release': pd.NaT,
            'selected_value': selected_value,
            'latest_value_at_release': np.nan,
            'selected_matches_latest_observation_date': False,
            'selected_matches_latest_value': False,
            'selection_status': 'no multi-observation release found',
            'note': 'No ALFRED/FRED rows matched this selected release bucket.',
        }

    if release_rows.shape[0] == 1:
        selected_obs = release_rows['observation_date'].iloc[0]
        latest_obs = release_rows['observation_date'].max()
        latest_public = release_rows['public_value'].iloc[0]
        selected_matches_obs = bool(pd.notna(selected_obs) and pd.notna(latest_obs) and selected_obs == latest_obs)
        selected_matches_val = bool(not pd.isna(selected_value) and not pd.isna(latest_public) and abs(selected_value - latest_public) <= 1e-9)

        if selected_matches_obs and selected_matches_val:
            status = 'latest observation selected'
            note = 'Latest observation date and value match ALFRED latest row.'
        elif (not selected_matches_obs) and selected_matches_val:
            status = 'earlier observation selected but public value still valid'
            note = 'Selected observation is older but value matches ALFRED release bucket.'
        elif selected_matches_obs and (not selected_matches_val):
            status = 'latest observation date but value mismatch'
            note = 'Latest observation date selected but value differs from ALFRED latest value.'
        else:
            status = 'value and observation mismatch vs live audit expectation'
            note = 'Selected observation and value both differ from the latest-observation expectation in the ALFRED release bucket.'

        return {
            'Feature': feature_label,
            'series_id': series_id,
            'selected_release_timestamp': selected_release_ts,
            'selected_observation_date': selected_obs,
            'max_observation_date_at_release': latest_obs,
            'selected_value': selected_value,
            'latest_value_at_release': latest_public,
            'selected_matches_latest_observation_date': selected_matches_obs,
            'selected_matches_latest_value': selected_matches_val,
            'selection_status': status,
            'note': note,
        }

    max_obs = release_rows['observation_date'].max()
    latest_rows = release_rows[release_rows['observation_date'] == max_obs]
    latest_value = pd.to_numeric(latest_rows['public_value'], errors='coerce').max()

    matching_rows = release_rows[pd.to_numeric(release_rows['public_value'], errors='coerce').between(selected_value - 1e-9, selected_value + 1e-9)]
    if matching_rows.empty:
        return {
            'Feature': feature_label,
            'series_id': series_id,
            'selected_release_timestamp': selected_release_ts,
            'selected_observation_date': pd.NaT,
            'max_observation_date_at_release': max_obs,
            'selected_value': selected_value,
            'latest_value_at_release': latest_value,
            'selected_matches_latest_observation_date': False,
            'selected_matches_latest_value': False,
            'selection_status': 'unresolved',
            'note': 'Selected raw value is not present in the matching ALFRED/FRED release bucket.',
        }

    if matching_rows['observation_date'].nunique() > 1:
        return {
            'Feature': feature_label,
            'series_id': series_id,
            'selected_release_timestamp': selected_release_ts,
            'selected_observation_date': pd.NaT,
            'max_observation_date_at_release': max_obs,
            'selected_value': selected_value,
            'latest_value_at_release': latest_value,
            'selected_matches_latest_observation_date': False,
            'selected_matches_latest_value': bool(not pd.isna(selected_value) and not pd.isna(latest_value) and abs(selected_value - latest_value) <= 1e-9),
            'selection_status': 'ambiguous duplicate values',
            'note': 'Multiple observation dates share same value; cannot uniquely identify selection.',
        }

    selected_obs = matching_rows['observation_date'].iloc[0]
    selected_matches_latest_observation_date = bool(pd.notna(selected_obs) and pd.notna(max_obs) and selected_obs == max_obs)
    selected_matches_latest_value = bool(not pd.isna(selected_value) and not pd.isna(latest_value) and abs(selected_value - latest_value) <= 1e-9)

    if selected_matches_latest_observation_date and selected_matches_latest_value:
        status = 'latest observation selected'
        note = 'Latest observation date and value match ALFRED latest row.'
    elif (not selected_matches_latest_observation_date) and selected_matches_latest_value:
        status = 'earlier observation selected but public value still valid'
        note = 'Selected observation is older but value matches ALFRED release bucket.'
    elif selected_matches_latest_observation_date and (not selected_matches_latest_value):
        status = 'latest observation date but value mismatch'
        note = 'Latest observation date selected but value differs from ALFRED latest value.'
    else:
        status = 'value and observation mismatch vs live audit expectation'
        note = 'Selected observation and value both differ from the latest-observation expectation in the ALFRED release bucket.'

    return {
        'Feature': feature_label,
        'series_id': series_id,
        'selected_release_timestamp': selected_release_ts,
        'selected_observation_date': selected_obs,
        'max_observation_date_at_release': max_obs,
        'selected_value': selected_value,
        'latest_value_at_release': latest_value,
        'selected_matches_latest_observation_date': selected_matches_latest_observation_date,
        'selected_matches_latest_value': selected_matches_latest_value,
        'selection_status': status,
        'note': note,
    }


def _as_naive_utc_timestamp(value):
    ts = pd.to_datetime(value, errors='coerce')
    if pd.isna(ts):
        return pd.NaT
    if getattr(ts, 'tzinfo', None) is not None:
        return ts.tz_convert(ZoneInfo('UTC')).tz_localize(None)
    return ts


def _normalize_timestamp_series(values):
    if isinstance(values, pd.Series):
        ts = pd.to_datetime(values, errors='coerce')
        if pd.api.types.is_datetime64_any_dtype(ts.dtype):
            try:
                if pd.api.types.is_datetime64tz_dtype(ts.dtype):
                    return ts.dt.tz_convert('UTC').dt.tz_localize(None)
            except Exception:
                return ts
        return ts
    return values.apply(_as_naive_utc_timestamp) if hasattr(values, 'apply') else values



def table_exists(name):
    return spark.catalog.tableExists(name)


def query_to_pandas(sql, empty_columns=None):
    try:
        return spark.sql(sql).toPandas()
    except Exception as exc:
        print(f'Query failed: {exc}')
        return pd.DataFrame(columns=empty_columns or [])


def latest_row(table_name, order_col):
    if not table_exists(table_name):
        return pd.Series(dtype='object')
    rows = query_to_pandas(f'SELECT * FROM {table_name} ORDER BY {order_col} DESC LIMIT 1')
    return rows.iloc[0] if not rows.empty else pd.Series(dtype='object')


def latest_rows(table_name, order_col, limit=LOOKBACK_HOURS):
    if not table_exists(table_name):
        return pd.DataFrame()
    q = f'''
        SELECT *
        FROM (
            SELECT *
            FROM {table_name}
            ORDER BY {order_col} DESC
            LIMIT {int(limit)}
        ) t
        ORDER BY {order_col} ASC
    '''
    return query_to_pandas(q)


def table_columns(table_name):
    schema = query_to_pandas(f'DESCRIBE {table_name}', empty_columns=['col_name'])
    if schema.empty or 'col_name' not in schema.columns:
        return []
    return [str(c).strip() for c in schema['col_name'].tolist()]


def load_model_metadata():
    default_version = 'bootstrap_live_model'
    default_path = LIVE_MODEL_PATH
    if table_exists(MODEL_REGISTRY_TABLE):
        rows = query_to_pandas(
            f'''
                SELECT model_version, model_path
                FROM {MODEL_REGISTRY_TABLE}
                ORDER BY promoted_ts DESC
                LIMIT 1
            '''
        )
        if not rows.empty and not _is_missing(rows.iloc[0].get('model_path')):
            return _as_text(rows.iloc[0].get('model_version'), default_version), rows.iloc[0].get('model_path')
    return default_version, default_path


def ensure_xgboost_runtime():
    if importlib.util.find_spec('xgboost') is None:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', f'xgboost=={REPORT_XGBOOST_VERSION}', '--quiet'],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
    import xgboost  # noqa: F401
    return xgboost


def load_model():
    model_version, model_path = load_model_metadata()
    try:
        ensure_xgboost_runtime()
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', message='.*serialized model.*', category=UserWarning)
            model = joblib.load(model_path)
        return model, model_version, model_path
    except Exception as exc:
        raise RuntimeError(f'Model load failed for institutional report ({model_path}): {exc}') from exc


def feature_importance(model):
    if model is None or not hasattr(model, 'feature_importances_'):
        raise RuntimeError('Model-derived feature importance is unavailable because the promoted model did not expose feature_importances_.')
    try:
        importance = np.array(model.feature_importances_, dtype=float)
    except Exception as exc:
        raise RuntimeError(f'Model-derived feature importance could not be read: {exc}') from exc

    finite_importance = importance[np.isfinite(importance)] if importance.ndim == 1 else np.array([])
    total_importance = np.nansum(importance) if importance.ndim == 1 else np.nan
    is_equal_weight_fallback = finite_importance.size > 1 and np.nanmax(finite_importance) > 0 and np.isclose(np.nanstd(finite_importance), 0)
    if importance.ndim != 1 or importance.size != len(FEATURES) or finite_importance.size == 0 or np.isclose(total_importance, 0) or is_equal_weight_fallback:
        raise RuntimeError('Model-derived feature importance is invalid or equal-weight; refusing to render misleading importance values.')

    frame = pd.DataFrame({'feature': FEATURES, 'importance': importance})
    frame['importance_pct'] = frame['importance'] / frame['importance'].sum()
    frame['label'] = frame['feature'].map(FEATURE_LABELS)
    return frame.sort_values('importance', ascending=False).reset_index(drop=True)


def top_named_value_from_row(row, columns):
    best_label = 'N/A'
    best_value = np.nan
    for c, label in columns.items():
        val = _to_float(row.get(c))
        if pd.notna(val) and (pd.isna(best_value) or val > best_value):
            best_label = label
            best_value = val
    return best_label, best_value


def historical_mix_text(row):
    mix_parts = []
    for column, label in HISTORICAL_MIX_COLUMNS.items():
        value = _to_float(row.get(column))
        if pd.notna(value):
            mix_parts.append(f'{label}: {fmt_percent_points(value)}')
    return ' | '.join(mix_parts) if mix_parts else 'N/A'


def render_probability_and_regime_window(probability_frame, regime_frame, probability_title, regime_title):
    fig, (axp, axr) = plt.subplots(2, 1, figsize=(16, 7.8), sharex=True, gridspec_kw={'height_ratios': [2.1, 1.15]}, constrained_layout=False)
    fig.subplots_adjust(top=0.90, bottom=0.16, hspace=0.28)
    probability_plot_columns = [column for column in PROBABILITY_LABELS.keys() if column in probability_frame.columns and probability_frame[column].notna().any()]
    if probability_plot_columns and not probability_frame.empty:
        for column in probability_plot_columns:
            axp.step(probability_frame['timestamp'], probability_frame[column] * 100, where='post', linewidth=1.7, label=PROBABILITY_LABELS[column])
        axp.set_title(probability_title, fontsize=15, fontweight='bold', loc='left')
        axp.set_ylabel('Probability (%)')
        axp.set_ylim(0, 100)
        axp.grid(alpha=0.25)
        axp.legend(loc='upper left', ncol=3)
    else:
        axp.axis('off')
        axp.text(0.5, 0.5, 'Model probability history is unavailable because predict_proba could not be computed.', ha='center', va='center')

    sig_plot = regime_frame[['timestamp', 'regime_int']].dropna().copy() if not regime_frame.empty else pd.DataFrame()
    if not sig_plot.empty:
        sig_plot['regime_step'] = pd.to_numeric(sig_plot['regime_int'], errors='coerce')
        sig_plot = sig_plot.sort_values('timestamp').reset_index(drop=True)
        band_colors = {0: '#DFF3E4', 1: '#FFF1C7', 2: '#F7D6D0'}
        line_colors = {0: '#2E8B57', 1: '#D4A017', 2: '#C0392B'}
        for regime_code, color in band_colors.items():
            mask = sig_plot['regime_step'] == regime_code
            if mask.any():
                axr.fill_between(sig_plot['timestamp'], regime_code - 0.42, regime_code + 0.42, where=mask, step='post', color=color, alpha=0.95)
        axr.step(sig_plot['timestamp'], sig_plot['regime_step'], where='post', linewidth=1.6, color='#34495E')
        if not sig_plot.empty:
            latest_regime = int(sig_plot['regime_step'].iloc[-1]) if pd.notna(sig_plot['regime_step'].iloc[-1]) else None
            if latest_regime in line_colors:
                axr.scatter(sig_plot['timestamp'].iloc[-1], sig_plot['regime_step'].iloc[-1], s=34, color=line_colors[latest_regime], zorder=5)
        axr.set_ylim(-0.5, 2.5)
        axr.set_ylabel('Regime')
        axr.set_title(regime_title, fontsize=15, fontweight='bold', loc='left')
        apply_ist_axis(axr)
        axr.set_yticks([0, 1, 2])
        axr.set_yticklabels(['Expansion', 'Slowdown', 'Recession'])
        axr.grid(axis='x', alpha=0.18)
        axr.grid(axis='y', alpha=0.08)
    else:
        axr.axis('off')
        axr.text(0.5, 0.5, 'No signal history available.', ha='center', va='center')

    plt.show()


def render_feature_monitoring_scope(frame, title):
    if frame.empty or 'timestamp' not in frame.columns:
        print(f'No feature monitoring data available for {title}.')
        return

    available_groups = []
    for group_name, features in FEATURE_MONITORING_GROUPS.items():
        present = [feature for feature in features if feature in frame.columns]
        if present:
            available_groups.append((group_name, present))

    if not available_groups:
        print(f'No grouped feature history available for {title}.')
        return

    n_cols = 2
    n_rows = math.ceil(len(available_groups) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(17, 4.8 * n_rows), sharex=True, constrained_layout=False)
    fig.subplots_adjust(top=0.84, bottom=0.13, hspace=0.62, wspace=0.24)
    axes = np.atleast_1d(axes).ravel()
    palette = ['#1F4E79', '#B03A2E', '#7D6608', '#117A65', '#6C3483', '#2E86C1']

    for axis, (group_name, features) in zip(axes, available_groups):
        for idx, feature in enumerate(features):
            series = pd.to_numeric(frame[feature], errors='coerce')
            axis.plot(frame['timestamp'], series, linewidth=1.9, alpha=0.96, label=FEATURE_LABELS.get(feature, feature), color=palette[idx % len(palette)])
        axis.axhline(0, color='#566573', linestyle=':', linewidth=1.0, alpha=0.6)
        axis.set_title(group_name, fontsize=14, fontweight='bold', loc='left', pad=10)
        axis.grid(alpha=0.22)
        axis.legend(loc='upper center', bbox_to_anchor=(0.5, 1.30), ncol=max(1, min(3, len(features))), fontsize=7.5, frameon=False, handlelength=1.8, columnspacing=1.2)
        if group_name == 'Event Flags':
            axis.set_ylim(-0.1, 1.1)
            axis.set_yticks([0, 1])
            axis.set_ylabel('Flag')
        elif group_name == 'Price Action':
            axis.set_ylabel('Proxy')
        else:
            axis.set_ylabel('z-score')
        apply_dual_time_axes(axis, bottom_label=f'Timestamp ({DISPLAY_TZ_LABEL})', maxticks=5, top_labelsize=7)

    for axis in axes[len(available_groups):]:
        axis.axis('off')

    fig.suptitle(title, fontsize=17, fontweight='bold', x=0.01, ha='left', y=1.01)
    plt.show()


def compute_probabilities(frame, model):
    probability_df = pd.DataFrame(index=frame.index)
    for column in PROBABILITY_COLUMNS.values():
        probability_df[column] = np.nan

    if frame.empty or model is None or not hasattr(model, 'predict_proba'):
        return probability_df

    feature_input = frame.reindex(columns=FEATURES).apply(pd.to_numeric, errors='coerce').fillna(0.0)
    try:
        probs = model.predict_proba(feature_input)
        classes = list(getattr(model, 'classes_', [0, 1, 2]))
        for idx, class_id in enumerate(classes):
            try:
                column_name = PROBABILITY_COLUMNS.get(int(class_id))
            except Exception:
                column_name = None
            if column_name and idx < probs.shape[1]:
                probability_df[column_name] = probs[:, idx]
    except Exception as exc:
        print(f'Probability computation unavailable: {exc}')

    return probability_df


PROBABILITY_COLUMN_ALIASES = {
    'prob_expansion': [
        'prob_expansion', 'expansion_probability', 'expansion_prob', 'p_expansion',
        'regime_prob_expansion', 'prob_regime_0', 'prob_0', 'class_0_probability',
    ],
    'prob_slowdown': [
        'prob_slowdown', 'slowdown_probability', 'slowdown_prob', 'p_slowdown',
        'regime_prob_slowdown', 'prob_regime_1', 'prob_1', 'class_1_probability',
    ],
    'prob_recession': [
        'prob_recession', 'recession_probability', 'recession_prob', 'p_recession',
        'regime_prob_recession', 'prob_contraction', 'contraction_prob', 'contraction_probability', 'prob_regime_2', 'prob_2', 'class_2_probability',
    ],
}


def _resolve_probability_columns(columns):
    columns = list(columns or [])
    normalized = {str(col).lower(): col for col in columns}
    resolved = {}
    for target, aliases in PROBABILITY_COLUMN_ALIASES.items():
        for alias in aliases:
            source = normalized.get(alias.lower())
            if source is not None:
                resolved[target] = source
                break
        if target not in resolved:
            label_token = PROBABILITY_LABELS[target].lower()
            for col in columns:
                col_text = str(col).lower()
                if label_token in col_text and ('prob' in col_text or 'probability' in col_text):
                    resolved[target] = col
                    break
    return resolved


def _normalize_probability_columns(frame):
    if frame.empty:
        return frame
    frame = frame.copy()
    for column in PROBABILITY_LABELS.keys():
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors='coerce')
            finite = frame[column].dropna()
            if not finite.empty and finite.abs().max() > 1.5:
                frame[column] = frame[column] / 100.0
    return frame


def load_probability_history_from_table(table_name, timestamp_col, lookback_rows=LOOKBACK_HOURS):
    empty = pd.DataFrame(columns=['timestamp'] + list(PROBABILITY_LABELS.keys()))
    if not table_exists(table_name):
        return empty
    columns = table_columns(table_name)
    if timestamp_col not in columns:
        return empty
    resolved = _resolve_probability_columns(columns)
    if not resolved:
        return empty
    select_exprs = [f'`{timestamp_col}` AS timestamp']
    for target, source in resolved.items():
        select_exprs.append(f'`{source}` AS `{target}`')
    query = f'''
        SELECT {', '.join(select_exprs)}
        FROM (
            SELECT *
            FROM {table_name}
            ORDER BY `{timestamp_col}` DESC
            LIMIT {int(lookback_rows)}
        ) t
        ORDER BY timestamp ASC
    '''
    frame = query_to_pandas(query, empty_columns=['timestamp'] + list(resolved.keys()))
    if frame.empty or 'timestamp' not in frame.columns:
        return empty
    frame['timestamp'] = to_display_ist_series(frame['timestamp'])
    for column in PROBABILITY_LABELS.keys():
        if column not in frame.columns:
            frame[column] = np.nan
    return _normalize_probability_columns(frame[['timestamp'] + list(PROBABILITY_LABELS.keys())])


def combine_probability_histories(*frames):
    usable = []
    for frame in frames:
        if frame is None or frame.empty or 'timestamp' not in frame.columns:
            continue
        subset = frame[['timestamp'] + [column for column in PROBABILITY_LABELS.keys() if column in frame.columns]].copy()
        for column in PROBABILITY_LABELS.keys():
            if column not in subset.columns:
                subset[column] = np.nan
        subset = _normalize_probability_columns(subset[['timestamp'] + list(PROBABILITY_LABELS.keys())])
        usable.append(subset)
    if not usable:
        return pd.DataFrame(columns=['timestamp'] + list(PROBABILITY_LABELS.keys()))
    combined = pd.concat(usable, ignore_index=True)
    combined = combined.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    probability_columns = list(PROBABILITY_LABELS.keys())
    if combined.empty:
        return combined
    combined['_has_probability'] = combined[probability_columns].notna().any(axis=1)
    combined = combined.sort_values(['timestamp', '_has_probability']).drop_duplicates(subset=['timestamp'], keep='last')
    combined = combined.drop(columns=['_has_probability']).sort_values('timestamp').reset_index(drop=True)
    return combined


def probability_summary_text(row):
    if row is None or len(row) == 0:
        return None
    parts = []
    for column, label in PROBABILITY_LABELS.items():
        value = _to_float(row.get(column))
        if pd.notna(value):
            parts.append(f'{label}: {fmt_pct(value)}')
    return ', '.join(parts) if parts else None


def detect_table_columns(frame, candidates):
    cols = set(frame.columns) if hasattr(frame, 'columns') else set(frame)
    return [c for c in candidates if c in cols]


def regime_label(name, code):
    text = _as_text(name, '')
    if text:
        return text
    code_value = _to_float(code)
    if pd.isna(code_value):
        return 'Unknown'
    return REGIME_NAME_MAP.get(int(code_value), f'Regime {int(code_value)}')


def build_regime_intervals(signal_frame):
    if signal_frame.empty or 'timestamp' not in signal_frame.columns:
        return pd.DataFrame(columns=['start_ts', 'end_ts', 'regime_name', 'regime_int', 'position'])

    frame = signal_frame.copy()
    frame = frame.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    if frame.empty:
        return pd.DataFrame(columns=['start_ts', 'end_ts', 'regime_name', 'regime_int', 'position'])

    name_series = frame['regime_name'] if 'regime_name' in frame.columns else pd.Series(index=frame.index, dtype='object')
    code_series = frame['regime_int'] if 'regime_int' in frame.columns else pd.Series(index=frame.index, dtype='float64')
    frame['regime_name'] = [regime_label(name, code) for name, code in zip(name_series, code_series)]

    change_mask = frame['regime_name'].ne(frame['regime_name'].shift())
    if 'regime_int' in frame.columns:
        regime_codes = pd.to_numeric(frame['regime_int'], errors='coerce')
        change_mask = change_mask | regime_codes.ne(regime_codes.shift())
    if 'position' in frame.columns:
        positions = frame['position'].fillna('')
        change_mask = change_mask | positions.ne(positions.shift().fillna(''))

    interval_cols = ['timestamp', 'regime_name'] + detect_table_columns(frame, ['regime_int', 'position'])
    intervals = frame.loc[change_mask, interval_cols].copy()
    intervals = intervals.rename(columns={'timestamp': 'start_ts'})
    intervals['end_ts'] = intervals['start_ts'].shift(-1)
    return intervals.reset_index(drop=True)


def select_live_monitoring_window(frame, timestamp_col, fallback_hours=48, min_rows=WINDOW_MIN_ROWS):
    if frame.empty or timestamp_col not in frame.columns:
        return (frame.copy() if hasattr(frame, 'copy') else pd.DataFrame()), pd.NaT, 'unavailable'

    working = frame.copy()
    working[timestamp_col] = pd.to_datetime(working[timestamp_col], errors='coerce')
    working = working.dropna(subset=[timestamp_col]).sort_values(timestamp_col).reset_index(drop=True)
    if working.empty:
        return working, pd.NaT, 'unavailable'

    anchor_ts = pd.to_datetime(working[timestamp_col].iloc[-1], errors='coerce')
    today_window_start = anchor_ts.normalize()
    today_window = working[working[timestamp_col] >= today_window_start].copy()
    if len(today_window) >= min_rows:
        return today_window, today_window_start, 'today'

    fallback_start = anchor_ts - pd.Timedelta(hours=int(fallback_hours))
    fallback_window = working[working[timestamp_col] >= fallback_start].copy()
    if not fallback_window.empty:
        return fallback_window, fallback_start, f'last {int(fallback_hours)}h'

    tail_window = working.tail(max(min_rows, 1)).copy()
    tail_start = tail_window[timestamp_col].iloc[0] if not tail_window.empty else pd.NaT
    return tail_window, tail_start, 'latest rows'


def format_monitoring_window_note(window_mode, window_start):
    start_text = fmt_timestamp(window_start)
    if window_mode == 'today':
        return f'today / current live-day window (from {start_text})'
    if window_mode == 'unavailable':
        return 'current live window unavailable'
    return f'{window_mode} fallback window (from {start_text})'


def select_fixed_monitoring_window(frame, timestamp_col, start_ts):
    if frame.empty or timestamp_col not in frame.columns:
        return (frame.copy() if hasattr(frame, 'copy') else pd.DataFrame()), pd.NaT, 'fixed_start'

    working = frame.copy()
    working[timestamp_col] = pd.to_datetime(working[timestamp_col], errors='coerce')
    working = working.dropna(subset=[timestamp_col]).sort_values(timestamp_col).reset_index(drop=True)
    if working.empty:
        return working, pd.NaT, 'fixed_start'

    fixed_start = pd.to_datetime(start_ts, errors='coerce')
    if pd.isna(fixed_start):
        return working, pd.NaT, 'fixed_start'

    fixed_window = working[working[timestamp_col] >= fixed_start].copy()
    return fixed_window.reset_index(drop=True), fixed_start, 'fixed_start'


def format_fixed_monitoring_note(window_start):
    if pd.isna(window_start):
        return 'post-release monitoring window unavailable'
    return f'post-release monitoring window from {window_start.strftime("%Y-%m-%d")} onward'


def select_rolling_window(frame, timestamp_col, hours=ROLLING_WINDOW_HOURS):
    if frame.empty or timestamp_col not in frame.columns:
        return (frame.copy() if hasattr(frame, 'copy') else pd.DataFrame()), pd.NaT, f'last {int(hours)}h'

    working = frame.copy()
    working[timestamp_col] = pd.to_datetime(working[timestamp_col], errors='coerce')
    working = working.dropna(subset=[timestamp_col]).sort_values(timestamp_col).reset_index(drop=True)
    if working.empty:
        return working, pd.NaT, f'last {int(hours)}h'

    anchor_ts = pd.to_datetime(working[timestamp_col].iloc[-1], errors='coerce')
    rolling_start = anchor_ts - pd.Timedelta(hours=int(hours))
    rolling_window = working[working[timestamp_col] >= rolling_start].copy()
    return rolling_window.reset_index(drop=True), rolling_start, f'last {int(hours)}h'


def format_rolling_window_note(window_start, hours=ROLLING_WINDOW_HOURS):
    if pd.isna(window_start):
        return f'rolling last {int(hours)} hours window unavailable'
    return f'rolling last {int(hours)} hours (from {fmt_timestamp(window_start)})'


def compute_running_drawdown(index_series):
    series = pd.to_numeric(index_series, errors='coerce')
    running_peak = series.cummax()
    return series / running_peak - 1.0


def fmt_model_input_value(value):
    value = _to_float(value)
    if pd.isna(value):
        return 'N/A'
    if float(value).is_integer() and abs(value) <= 1:
        return str(int(value))
    return f'{value:+.3f}'


def fmt_report_number(value, column_name=''):
    value = _to_float(value)
    if pd.isna(value):
        return 'N/A'

    column_name = _as_text(column_name).lower()
    if 'importance' in column_name:
        return f'{value:.2%}'
    if 'z-score' in column_name or 'z score' in column_name or column_name.endswith('_z'):
        return f'{value:+.3f}'
    if 'lag (hours)' in column_name:
        return f'{value:.1f}'
    if any(token in column_name for token in ('return', 'alpha', 'pnl', 'p&l', 'equity', 'cash', 'balance', 'drawdown', 'position')):
        return f'{value:+,.2f}'
    if abs(value) < 1 and abs(value) > 0:
        return f'{value:.4f}'
    return f'{value:,.2f}'



def fmt_report_text(value, max_chars=240):
    text = _as_text(value)
    if not text:
        return 'N/A'
    text = text.replace('\n', ' ').strip()
    if len(text) <= max_chars:
        return text
    return f'{text[: max_chars - 1]}…'

def fmt_validation_diff(value):
    value = _to_float(value)
    if pd.isna(value):
        return 'N/A'
    if value == 0:
        return '0'
    return f'{value:.6f}'


def sql_timestamp(value):
    ts = pd.to_datetime(value, errors='coerce')
    return None if pd.isna(ts) else ts.strftime('%Y-%m-%d %H:%M:%S')


def first_row(frame):
    return frame.iloc[0] if isinstance(frame, pd.DataFrame) and not frame.empty else pd.Series(dtype='object')


def fmt_freshness(source_ts, reference_ts):
    src = pd.to_datetime(source_ts, errors='coerce')
    ref = pd.to_datetime(reference_ts, errors='coerce')
    if pd.isna(src) or pd.isna(ref):
        return 'N/A', np.nan
    delta_minutes = max((ref - src).total_seconds() / 60.0, 0.0)
    if delta_minutes < 180:
        return f'{delta_minutes:.0f} min', delta_minutes
    if delta_minutes < 2880:
        return f'{delta_minutes / 60.0:.1f} h', delta_minutes
    return f'{delta_minutes / 1440.0:.1f} d', delta_minutes


def classify_market_freshness(source_ts, reference_ts):
    _, lag_minutes = fmt_freshness(source_ts, reference_ts)
    if pd.isna(lag_minutes):
        return 'Unresolved'
    if lag_minutes <= MARKET_FRESH_HOURS * 60:
        return 'Fresh'
    if lag_minutes <= MARKET_WARN_HOURS * 60:
        return 'Warning'
    return 'Stale'


def classify_macro_freshness(source_row):
    return 'Fresh' if not getattr(source_row, 'empty', True) else 'Unresolved'


def proof_tolerances(feature):
    if feature in {'Price_Mom_24H'}:
        return RATIO_EXACT_TOL, RATIO_CLOSE_TOL
    if feature in {'CPI_Event_Active', 'NFP_Event_Active'}:
        return 0.0, 0.0
    return Z_EXACT_TOL, Z_CLOSE_TOL


def evaluate_proof(feature, model_input_value, reconstructed_value):
    model_val = _to_float(model_input_value)
    recon_val = _to_float(reconstructed_value)
    if pd.isna(model_val) or pd.isna(recon_val):
        return np.nan, 'Derived / unresolved'
    if feature in {'CPI_Event_Active', 'NFP_Event_Active'}:
        diff = 0.0 if int(round(model_val)) == int(round(recon_val)) else 1.0
        status = 'Exact match to current model row' if diff == 0.0 else 'Mismatch vs current model row'
        return diff, status
    diff = abs(model_val - recon_val)
    exact_tol, close_tol = proof_tolerances(feature)
    if diff <= exact_tol:
        return diff, 'Exact match to current model row'
    if diff <= close_tol:
        return diff, 'Close match to current model row'
    return diff, 'Mismatch vs current model row'


def macro_gating_filter(indicator_id, live_columns):
    if indicator_id == 'UMCSENT' and {'release_stage', 'validation_status'}.issubset(live_columns):
        return "AND release_stage = 'final' AND validation_status = 'validated'"
    return ''


def load_live_macro_source(indicator_id, feature_ts):
    feature_ts_sql = sql_timestamp(feature_ts)
    if indicator_id not in LIVE_MACRO_OVERRIDE_INDICATORS or feature_ts_sql is None or not table_exists(LIVE_MACRO_TABLE):
        return pd.Series(dtype='object')
    live_columns = set(table_columns(LIVE_MACRO_TABLE))
    required = {'indicator_id', 'observed_value', 'scheduled_release_ts', 'status'}
    if not required.issubset(live_columns):
        return pd.Series(dtype='object')
    gating_filter = macro_gating_filter(indicator_id, live_columns)
    provider_field = 'source_provider' if 'source_provider' in live_columns else 'NULL'
    release_stage_field = 'release_stage' if 'release_stage' in live_columns else 'NULL'
    validation_field = 'validation_status' if 'validation_status' in live_columns else 'NULL'
    live_sql = f"""
        SELECT indicator_id,
               observed_value AS raw_value,
               scheduled_release_ts AS source_ts,
               scheduled_release_ts AS effective_ts,
               '{LIVE_MACRO_TABLE}' AS source_table,
               'observed_value' AS source_field,
               status,
               {provider_field} AS source_provider,
               {release_stage_field} AS release_stage,
               {validation_field} AS validation_status
        FROM {LIVE_MACRO_TABLE}
        WHERE indicator_id = '{indicator_id}'
          AND observed_value IS NOT NULL
          AND status IN ('observed', 'reconciled_observed')
          {gating_filter}
          AND scheduled_release_ts <= TIMESTAMP('{feature_ts_sql}')
        ORDER BY scheduled_release_ts DESC, ingestion_ts DESC
        LIMIT 1
    """
    return first_row(query_to_pandas(live_sql))


def load_bt_macro_source(indicator_id, feature_ts):
    feature_ts_sql = sql_timestamp(feature_ts)
    if feature_ts_sql is None or not table_exists(BT_MACRO_TABLE):
        return pd.Series(dtype='object')

    bt_columns = set(table_columns(BT_MACRO_TABLE))
    value_field = 'actual' if 'actual' in bt_columns else 'observed_value' if 'observed_value' in bt_columns else None
    if value_field is None or not {'indicator_id', 'timestamp'}.issubset(bt_columns):
        return pd.Series(dtype='object')

    bt_sql = f"""
        SELECT indicator_id,
               {value_field} AS raw_value,
               timestamp AS source_ts,
               date_trunc('hour', timestamp + INTERVAL 2 HOUR) AS effective_ts,
               '{BT_MACRO_TABLE}' AS source_table,
               '{value_field}' AS source_field
        FROM {BT_MACRO_TABLE}
        WHERE indicator_id = '{indicator_id}'
          AND {value_field} IS NOT NULL
          AND date_trunc('hour', timestamp + INTERVAL 2 HOUR) <= TIMESTAMP('{feature_ts_sql}')
        ORDER BY effective_ts DESC, timestamp DESC
        LIMIT 1
    """
    return first_row(query_to_pandas(bt_sql))


def load_silver_bt_macro_row(feature_ts, z_feature):
    feature_ts_sql = sql_timestamp(feature_ts)
    if feature_ts_sql is None or not table_exists(SILVER_HIST_TABLE):
        return pd.Series(dtype='object')

    bt_columns = set(table_columns(SILVER_HIST_TABLE))
    if z_feature not in bt_columns:
        return pd.Series(dtype='object')

    silver_bt_sql = f"""
        SELECT
            timestamp,
            {z_feature} AS z_value
        FROM {SILVER_HIST_TABLE}
        WHERE timestamp <= TIMESTAMP('{feature_ts_sql}')
        ORDER BY timestamp DESC
        LIMIT 1
    """
    return first_row(query_to_pandas(silver_bt_sql, empty_columns=['timestamp', 'z_value']))

def load_macro_release_asof(indicator_id, reference_ts):
    ref_ts_sql = sql_timestamp(reference_ts)
    if ref_ts_sql is None or not table_exists(BT_MACRO_TABLE):
        return pd.Series(dtype='object')

    bt_columns = set(table_columns(BT_MACRO_TABLE))
    value_field = 'actual' if 'actual' in bt_columns else 'observed_value' if 'observed_value' in bt_columns else None
    if value_field is None or not {'indicator_id', 'timestamp'}.issubset(bt_columns):
        return pd.Series(dtype='object')

    release_sql = f"""
        SELECT
            {value_field} AS raw_value,
            timestamp AS source_ts,
            date_trunc('hour', timestamp + INTERVAL 2 HOUR) AS effective_ts
        FROM {BT_MACRO_TABLE}
        WHERE indicator_id = '{indicator_id}'
          AND {value_field} IS NOT NULL
          AND date_trunc('hour', timestamp + INTERVAL 2 HOUR) <= TIMESTAMP('{ref_ts_sql}')
        ORDER BY effective_ts DESC, timestamp DESC
        LIMIT 1
    """
    return first_row(query_to_pandas(release_sql, empty_columns=['raw_value', 'source_ts', 'effective_ts']))



def load_rt_vix_source(series_name, feature_ts):
    feature_ts_sql = sql_timestamp(feature_ts)
    if feature_ts_sql is None or not table_exists(RT_VIX_TABLE):
        return pd.Series(dtype='object')
    vix_columns = set(table_columns(RT_VIX_TABLE))
    if not {'series', 'timestamp', 'close'}.issubset(vix_columns):
        return pd.Series(dtype='object')
    vix_sql = f"""
        SELECT '{series_name}' AS indicator_id,
               close AS raw_value,
               timestamp AS source_ts,
               timestamp AS effective_ts,
               '{RT_VIX_TABLE}' AS source_table,
               'close' AS source_field
        FROM {RT_VIX_TABLE}
        WHERE series = '{series_name}'
          AND timestamp <= TIMESTAMP('{feature_ts_sql}')
        ORDER BY timestamp DESC
        LIMIT 1
    """
    return first_row(query_to_pandas(vix_sql))


def load_rt_market_source(feature_ts):
    feature_ts_sql = sql_timestamp(feature_ts)
    if feature_ts_sql is None or not table_exists(RT_MARKET_TABLE):
        return pd.Series(dtype='object')
    market_sql = f"""
        SELECT close AS raw_value,
               timestamp AS source_ts,
               '{RT_MARKET_TABLE}' AS source_table,
               'close' AS source_field
        FROM {RT_MARKET_TABLE}
        WHERE symbol = '{PRIMARY_SYMBOL}'
          AND timestamp <= TIMESTAMP('{feature_ts_sql}')
        ORDER BY timestamp DESC
        LIMIT 1
    """
    return first_row(query_to_pandas(market_sql))


def resolve_vix_history_value_column(table_name):
    columns = set(table_columns(table_name))
    if 'close' in columns:
        return 'close'
    if 'vix3m' in columns:
        return 'vix3m'
    return None


def load_history_values(table_name, value_field, end_ts, limit_rows, extra_filters=''):
    end_ts_sql = sql_timestamp(end_ts)
    if end_ts_sql is None or not table_exists(table_name) or not value_field:
        return []
    history_sql = f"""
        SELECT timestamp, {value_field} AS value
        FROM {table_name}
        WHERE timestamp < TIMESTAMP('{end_ts_sql}')
          {extra_filters}
          AND {value_field} IS NOT NULL
        ORDER BY timestamp DESC
        LIMIT {int(limit_rows)}
    """
    history_pdf = query_to_pandas(history_sql, empty_columns=['timestamp', 'value'])
    if history_pdf.empty:
        return []
    history_pdf['timestamp'] = pd.to_datetime(history_pdf['timestamp'], errors='coerce')
    history_pdf['value'] = pd.to_numeric(history_pdf['value'], errors='coerce')
    history_pdf = history_pdf.dropna(subset=['timestamp', 'value']).sort_values('timestamp')
    return history_pdf['value'].astype(float).tolist()


def load_macro_release_history(indicator_id, feature_ts, history_hours):
    feature_ts = pd.to_datetime(feature_ts, errors='coerce')
    if pd.isna(feature_ts) or not table_exists(BT_MACRO_TABLE):
        return pd.DataFrame(columns=['source_ts', 'effective_ts', 'raw_value'])
    bt_columns = set(table_columns(BT_MACRO_TABLE))
    value_field = 'actual' if 'actual' in bt_columns else 'observed_value' if 'observed_value' in bt_columns else None
    if value_field is None or not {'indicator_id', 'timestamp'}.issubset(bt_columns):
        return pd.DataFrame(columns=['source_ts', 'effective_ts', 'raw_value'])
    history_start = feature_ts - pd.Timedelta(hours=int(history_hours))
    feature_ts_sql = sql_timestamp(feature_ts)
    history_start_sql = sql_timestamp(history_start)
    release_sql = f"""
        SELECT timestamp AS source_ts,
               date_trunc('hour', timestamp + INTERVAL 2 HOUR) AS effective_ts,
               {value_field} AS raw_value
        FROM {BT_MACRO_TABLE}
        WHERE indicator_id = '{indicator_id}'
          AND {value_field} IS NOT NULL
          AND date_trunc('hour', timestamp + INTERVAL 2 HOUR) <= TIMESTAMP('{feature_ts_sql}')
          AND date_trunc('hour', timestamp + INTERVAL 2 HOUR) >= TIMESTAMP('{history_start_sql}')
        ORDER BY effective_ts ASC, source_ts ASC
    """
    anchor_sql = f"""
        SELECT timestamp AS source_ts,
               date_trunc('hour', timestamp + INTERVAL 2 HOUR) AS effective_ts,
               {value_field} AS raw_value
        FROM {BT_MACRO_TABLE}
        WHERE indicator_id = '{indicator_id}'
          AND {value_field} IS NOT NULL
          AND date_trunc('hour', timestamp + INTERVAL 2 HOUR) < TIMESTAMP('{history_start_sql}')
        ORDER BY effective_ts DESC, source_ts DESC
        LIMIT 1
    """
    release_pdf = query_to_pandas(release_sql, empty_columns=['source_ts', 'effective_ts', 'raw_value'])
    anchor_pdf = query_to_pandas(anchor_sql, empty_columns=['source_ts', 'effective_ts', 'raw_value'])
    history_pdf = pd.concat([anchor_pdf, release_pdf], ignore_index=True)
    if history_pdf.empty:
        return history_pdf
    history_pdf['source_ts'] = pd.to_datetime(history_pdf['source_ts'], errors='coerce')
    history_pdf['effective_ts'] = pd.to_datetime(history_pdf['effective_ts'], errors='coerce')
    history_pdf['raw_value'] = pd.to_numeric(history_pdf['raw_value'], errors='coerce')
    history_pdf = history_pdf.dropna(subset=['effective_ts', 'raw_value']).sort_values(['effective_ts', 'source_ts'])
    history_pdf = history_pdf.drop_duplicates(subset=['effective_ts'], keep='last')
    return history_pdf.reset_index(drop=True)


def build_macro_hourly_history(indicator_id, feature_ts, history_hours, include_current_hour=False):
    feature_ts = pd.to_datetime(feature_ts, errors='coerce')
    if pd.isna(feature_ts):
        return []
    release_history = load_macro_release_history(indicator_id, feature_ts, history_hours)
    if release_history.empty:
        return []
    end_ts = feature_ts if include_current_hour else feature_ts - pd.Timedelta(hours=1)
    start_ts = feature_ts - pd.Timedelta(hours=int(history_hours))
    if pd.isna(end_ts) or end_ts < start_ts:
        return []
    hourly_index = pd.date_range(start=start_ts, end=end_ts, freq='h')
    hourly_frame = pd.DataFrame({'timestamp': hourly_index})
    release_history = release_history[['effective_ts', 'raw_value']].rename(columns={'effective_ts': 'timestamp'})
    aligned = pd.merge_asof(hourly_frame, release_history.sort_values('timestamp'), on='timestamp', direction='backward')
    aligned['raw_value'] = pd.to_numeric(aligned['raw_value'], errors='coerce').ffill()
    aligned = aligned.dropna(subset=['raw_value'])
    return aligned['raw_value'].astype(float).tolist()


def compute_zscore_from_values(values):
    cleaned = [float(v) for v in values if pd.notna(v)]
    if not cleaned:
        return np.nan
    series = pd.Series(cleaned, dtype='float64')
    current_value = float(series.iloc[-1])
    mean_val = float(series.mean())
    std_val = float(series.std(ddof=1)) if len(series) > 1 else 0.0
    if math.isnan(std_val) or std_val == 0.0:
        return 0.0
    return (current_value - mean_val) / std_val


def build_feature_result(feature, label, model_input_value, source_value, source_ts, source_table, source_feature, reconstructed_value, freshness, freshness_status, note, lineage_source='N/A', lineage_reference_ts=None, lineage_reference_value=None, live_override_ts=None, live_override_value=None, upstream_source_table=None, upstream_source_field=None, upstream_source_ts=None, upstream_source_value=None, first_divergence='N/A', resolution_status='N/A'):
    diff, proof_status = evaluate_proof(feature, model_input_value, reconstructed_value)
    return {
        'feature': feature,
        'Feature': label,
        'Model input value': fmt_model_input_value(model_input_value),
        'Lineage reference table': _as_text(source_table),
        'Lineage reference feature': _as_text(source_feature),
        'Lineage reference timestamp': fmt_timestamp(source_ts),
        'Lineage reference value': fmt_model_input_value(source_value),
        'Upstream raw source table': _as_text(upstream_source_table),
        'Upstream raw source field': _as_text(upstream_source_field),
        'Upstream raw source timestamp': fmt_timestamp(upstream_source_ts),
        'Upstream raw source value': fmt_raw_snapshot_value(upstream_source_value),
        'Reconstructed value': fmt_model_input_value(reconstructed_value),
        'Validation diff': fmt_validation_diff(diff),
        'Proof status': proof_status,
        'Freshness': freshness,
        'Freshness status': freshness_status,
        'Lineage source type': lineage_source,
        'Live override timestamp': fmt_timestamp(live_override_ts),
        'Live override value': fmt_raw_snapshot_value(live_override_value),
        'Lineage reference timestamp/row': fmt_timestamp(lineage_reference_ts),
        'Lineage reference value/row': fmt_model_input_value(lineage_reference_value),
        'First point of divergence': _as_text(first_divergence),
        'Resolution status': _as_text(resolution_status),
        'Note': note,
    }


def unresolved_feature_result(feature, model_input_value, source_table, source_feature, source_field='N/A', note='Derived / unresolved', lineage_source='N/A', lineage_reference_ts=None, lineage_reference_value=None, live_override_ts=None, live_override_value=None, upstream_source_table='N/A', upstream_source_field='N/A', upstream_source_ts=None, upstream_source_value=None, first_divergence='N/A', resolution_status='N/A'):
    return {
        'feature': feature,
        'Feature': FEATURE_LABELS.get(feature, feature),
        'Model input value': fmt_model_input_value(model_input_value),
        'Lineage reference table': _as_text(source_table),
        'Lineage reference feature': _as_text(source_feature),
        'Lineage reference timestamp': 'N/A',
        'Lineage reference value': 'N/A',
        'Upstream raw source table': _as_text(upstream_source_table),
        'Upstream raw source field': _as_text(upstream_source_field),
        'Upstream raw source timestamp': fmt_timestamp(upstream_source_ts),
        'Upstream raw source value': fmt_raw_snapshot_value(upstream_source_value),
        'Reconstructed value': 'N/A',
        'Validation diff': 'N/A',
        'Proof status': 'Derived / unresolved',
        'Freshness': 'N/A',
        'Freshness status': 'Unresolved',
        'Lineage source type': lineage_source,
        'Live override timestamp': fmt_timestamp(live_override_ts),
        'Live override value': fmt_raw_snapshot_value(live_override_value),
        'Lineage reference timestamp/row': fmt_timestamp(lineage_reference_ts),
        'Lineage reference value/row': fmt_model_input_value(lineage_reference_value),
        'First point of divergence': _as_text(first_divergence),
        'Resolution status': _as_text(resolution_status),
        'Note': note,
    }


def validate_macro_feature(feature, model_input_value, feature_ts):
    spec = RAW_FEATURE_SPECS.get(feature, {})
    indicator_id = spec.get('indicator_id')
    if not indicator_id:
        return unresolved_feature_result(feature, model_input_value, SILVER_HIST_TABLE, feature, 'source not mapped', 'Missing macro spec mapping.')

    silver_bt_row = load_silver_bt_macro_row(feature_ts, feature)
    if silver_bt_row.empty:
        return unresolved_feature_result(feature, model_input_value, SILVER_HIST_TABLE, feature, 'carry-forward row', 'No carry-forward silver_bt row loaded for current model timestamp.')

    silver_bt_ts = silver_bt_row.get('timestamp')
    silver_bt_z = _to_float(silver_bt_row.get('z_value'))
    upstream_release = load_macro_release_asof(indicator_id, silver_bt_ts)

    source_type = 'silver_bt_master_grid_1h carry-forward'
    source_ts = silver_bt_ts
    source_feature = feature
    freshness, _ = fmt_freshness(source_ts, feature_ts)

    if pd.isna(silver_bt_z):
        return unresolved_feature_result(
            feature,
            model_input_value,
            SILVER_HIST_TABLE,
            source_feature,
            'z_value',
            'silver_bt carry-forward row has null z-score.',
            lineage_source=source_type,
            lineage_reference_ts=silver_bt_ts,
            lineage_reference_value=np.nan,
            upstream_source_table=BT_MACRO_TABLE,
            upstream_source_field='actual / observed_value',
            upstream_source_ts=upstream_release.get('source_ts'),
            upstream_source_value=upstream_release.get('raw_value'),
            first_divergence='silver_bt row returned null z-score',
            resolution_status='Unresolved before reconstruction',
        )

    model_diff_vs_bt = abs(_to_float(model_input_value) - silver_bt_z)

    live_row = load_live_macro_source(indicator_id, feature_ts) if spec.get('live_override') else pd.Series(dtype='object')
    if not live_row.empty:
        live_value = _to_float(live_row.get('raw_value'))
        if not pd.isna(live_value):
            history_values = build_macro_hourly_history(indicator_id, feature_ts, BATCH_Z_HISTORY_ROWS, include_current_hour=False)
            if history_values:
                live_recon = compute_zscore_from_values(history_values + [live_value])
                _, live_status = evaluate_proof(feature, model_input_value, live_recon)
                if live_status in {'Exact match to current model row', 'Close match to current model row'}:
                    source_type = 'RT live override used'
                    source_ts = live_row.get('source_ts')
                    freshness, _ = fmt_freshness(source_ts, feature_ts)
                    return build_feature_result(
                        feature,
                        FEATURE_LABELS.get(feature, feature),
                        model_input_value,
                        live_value,
                        source_ts,
                        live_row.get('source_table'),
                        live_row.get('source_field'),
                        live_recon,
                        freshness,
                        classify_market_freshness(source_ts, feature_ts),
                        'Live override value validated against model row and accepted as source.',
                        lineage_source=source_type,
                        lineage_reference_ts=silver_bt_ts,
                        lineage_reference_value=silver_bt_z,
                        live_override_ts=source_ts,
                        live_override_value=live_value,
                        upstream_source_table=BT_MACRO_TABLE,
                        upstream_source_field='actual / observed_value',
                        upstream_source_ts=upstream_release.get('source_ts'),
                        upstream_source_value=upstream_release.get('raw_value'),
                        first_divergence='None (proven live override match)',
                        resolution_status='Proven via live override reconstruction against raw macro release path',
                    )

    first_divergence = 'None (proven carry-forward match)' if model_diff_vs_bt <= Z_CLOSE_TOL else 'silver_rt_1h value differs from carry-forward silver_bt row'
    resolution_status = 'Proven via silver_bt lineage' if model_diff_vs_bt <= Z_CLOSE_TOL else 'Mismatch before model-input lineage boundary'

    return build_feature_result(
        feature,
        FEATURE_LABELS.get(feature, feature),
        model_input_value,
        silver_bt_z,
        source_ts,
        SILVER_HIST_TABLE,
        source_feature,
        silver_bt_z,
        freshness,
        'Fresh' if model_diff_vs_bt <= Z_CLOSE_TOL else 'Unresolved',
        'Prefer SILVER_BT row first (production lineage). Upstream raw check is diagnostic.',
        lineage_source=source_type,
        lineage_reference_ts=silver_bt_ts,
        lineage_reference_value=silver_bt_z,
        live_override_ts=pd.NaT,
        live_override_value=np.nan,
        upstream_source_table=BT_MACRO_TABLE,
        upstream_source_field='actual / observed_value',
        upstream_source_ts=upstream_release.get('source_ts'),
        upstream_source_value=upstream_release.get('raw_value'),
        first_divergence=first_divergence,
        resolution_status=resolution_status,
    )
def validate_rt_vix_feature(feature, model_input_value, feature_ts):
    spec = RAW_FEATURE_SPECS.get(feature, {})
    source_row = load_rt_vix_source(spec.get('series'), feature_ts)
    history_table = BT_VIX_TABLE if feature == 'VIXCLS_z' else BT_VIX3M_TABLE
    history_field = 'close' if feature == 'VIXCLS_z' else resolve_vix_history_value_column(history_table)
    if source_row.empty or not history_field:
        return unresolved_feature_result(feature, model_input_value, RT_VIX_TABLE, feature, spec.get('source_field', 'close'), 'RT VIX source or matching batch history path is unavailable.')
    history_values = load_history_values(history_table, history_field, feature_ts, LIVE_Z_HISTORY_ROWS)
    if not history_values:
        return unresolved_feature_result(feature, model_input_value, history_table, feature, history_field, 'Insufficient VIX history to reconstruct RT z-score.')
    reconstructed_value = compute_zscore_from_values(history_values + [float(source_row.get('raw_value'))])
    freshness, _ = fmt_freshness(source_row.get('source_ts'), feature_ts)
    note = 'Current raw input comes from bronze_rt_vix_1d; z-score history matches the RT enrichment batch window.'
    return build_feature_result(feature, FEATURE_LABELS.get(feature, feature), model_input_value, source_row.get('raw_value'), source_row.get('source_ts'), source_row.get('source_table'), source_row.get('source_field'), reconstructed_value, freshness, classify_market_freshness(source_row.get('source_ts'), feature_ts), note)


def load_market_history(feature_ts, lookback_hours):
    feature_ts = pd.to_datetime(feature_ts, errors='coerce')
    if pd.isna(feature_ts) or not table_exists(BT_MARKET_TABLE) or not table_exists(RT_MARKET_TABLE):
        return pd.Series(dtype='float64')
    history_start = feature_ts - pd.Timedelta(hours=lookback_hours)
    history_start_sql = sql_timestamp(history_start)
    feature_ts_sql = sql_timestamp(feature_ts)
    union_sql = f"""
        SELECT timestamp, close, source_rank
        FROM (
            SELECT timestamp, close, 0 AS source_rank
            FROM {BT_MARKET_TABLE}
            WHERE symbol = '{PRIMARY_SYMBOL}'
              AND timestamp >= TIMESTAMP('{history_start_sql}')
              AND timestamp <= TIMESTAMP('{feature_ts_sql}')
            UNION ALL
            SELECT timestamp, close, 1 AS source_rank
            FROM {RT_MARKET_TABLE}
            WHERE symbol = '{PRIMARY_SYMBOL}'
              AND timestamp >= TIMESTAMP('{history_start_sql}')
              AND timestamp <= TIMESTAMP('{feature_ts_sql}')
        ) t
        ORDER BY timestamp ASC, source_rank ASC
    """
    union_pdf = query_to_pandas(union_sql, empty_columns=['timestamp', 'close', 'source_rank'])
    anchor_sql = f"""
        SELECT timestamp, close, source_rank
        FROM (
            SELECT timestamp, close, 0 AS source_rank
            FROM {BT_MARKET_TABLE}
            WHERE symbol = '{PRIMARY_SYMBOL}'
              AND timestamp < TIMESTAMP('{history_start_sql}')
            UNION ALL
            SELECT timestamp, close, 1 AS source_rank
            FROM {RT_MARKET_TABLE}
            WHERE symbol = '{PRIMARY_SYMBOL}'
              AND timestamp < TIMESTAMP('{history_start_sql}')
        ) t
        ORDER BY timestamp DESC, source_rank DESC
        LIMIT 1
    """
    anchor_pdf = query_to_pandas(anchor_sql, empty_columns=['timestamp', 'close', 'source_rank'])
    if union_pdf.empty and anchor_pdf.empty:
        return pd.Series(dtype='float64')
    if not anchor_pdf.empty:
        anchor_row = anchor_pdf.iloc[0]
        anchor_add = pd.DataFrame([{'timestamp': pd.Timestamp(history_start), 'close': anchor_row.get('close'), 'source_rank': anchor_row.get('source_rank')}])
        union_pdf = pd.concat([anchor_add, union_pdf], ignore_index=True)
    union_pdf['timestamp'] = pd.to_datetime(union_pdf['timestamp'], errors='coerce')
    union_pdf['close'] = pd.to_numeric(union_pdf['close'], errors='coerce')
    union_pdf = union_pdf.dropna(subset=['timestamp', 'close']).sort_values(['timestamp', 'source_rank'])
    union_pdf = union_pdf.drop_duplicates(subset=['timestamp'], keep='last')
    hourly_index = pd.date_range(start=history_start, end=feature_ts, freq='h')
    market_series = union_pdf.set_index('timestamp')['close'].astype(float).reindex(hourly_index).ffill()
    if market_series.isna().any():
        market_series = market_series.bfill()
    return market_series


def validate_price_momentum_feature(model_input_value, feature_ts):
    source_row = load_rt_market_source(feature_ts)
    market_series = load_market_history(feature_ts, PRICE_CONTEXT_HOURS)
    feature_ts = pd.to_datetime(feature_ts, errors='coerce')
    lag_ts = feature_ts - pd.Timedelta(hours=24) if pd.notna(feature_ts) else pd.NaT
    if source_row.empty or market_series.empty or pd.isna(feature_ts) or lag_ts not in market_series.index:
        return unresolved_feature_result('Price_Mom_24H', model_input_value, RT_MARKET_TABLE, 'Price_Mom_24H', 'close', 'Unable to reconstruct RT market history path for 24H momentum.')
    lag_close = market_series.loc[lag_ts]
    current_close = _to_float(source_row.get('raw_value'))
    if pd.isna(lag_close) or pd.isna(current_close) or float(lag_close) == 0.0:
        return unresolved_feature_result('Price_Mom_24H', model_input_value, RT_MARKET_TABLE, 'Price_Mom_24H', 'close', 'Momentum reconstruction missing current or 24H lag close.')
    reconstructed_value = current_close / float(lag_close) - 1.0
    freshness, _ = fmt_freshness(source_row.get('source_ts'), feature_ts)
    note = 'Recomputed from the same RT market close plus 24H lag context used by RT enrichment.'
    return build_feature_result('Price_Mom_24H', FEATURE_LABELS.get('Price_Mom_24H', 'Price_Mom_24H'), model_input_value, current_close, source_row.get('source_ts'), source_row.get('source_table'), source_row.get('source_field'), reconstructed_value, freshness, classify_market_freshness(source_row.get('source_ts'), feature_ts), note)


def validate_vix_inversion_feature(model_input_value, feature_ts):
    vix_result = validate_rt_vix_feature('VIXCLS_z', np.nan, feature_ts)
    vix3m_result = validate_rt_vix_feature('vix3m_z', np.nan, feature_ts)
    vix_recon = _to_float(vix_result.get('Reconstructed value'))
    vix3m_recon = _to_float(vix3m_result.get('Reconstructed value'))
    if pd.isna(vix_recon) or pd.isna(vix3m_recon):
        return unresolved_feature_result('VIX_Inversion_Z', model_input_value, RT_VIX_TABLE, 'VIX_Inversion_Z', 'close (VIXCLS, vix3m)', 'Unable to reconstruct both RT VIX component z-scores.')
    source_row_vix = load_rt_vix_source('VIXCLS', feature_ts)
    source_row_vix3m = load_rt_vix_source('vix3m', feature_ts)
    source_ts = max(pd.to_datetime(source_row_vix.get('source_ts'), errors='coerce'), pd.to_datetime(source_row_vix3m.get('source_ts'), errors='coerce'))
    freshness, _ = fmt_freshness(source_ts, feature_ts)
    reconstructed_value = vix_recon - vix3m_recon
    raw_pair = 'N/A'
    if not source_row_vix.empty and not source_row_vix3m.empty:
        raw_pair = f"VIX {fmt_raw_snapshot_value(source_row_vix.get('raw_value'))} / VIX3M {fmt_raw_snapshot_value(source_row_vix3m.get('raw_value'))}"
    note = 'Derived from reconstructed RT VIX spot and VIX 3M z-scores.'
    return build_feature_result('VIX_Inversion_Z', FEATURE_LABELS.get('VIX_Inversion_Z', 'VIX_Inversion_Z'), model_input_value, raw_pair, source_ts, RT_VIX_TABLE, 'close (VIXCLS, vix3m)', reconstructed_value, freshness, classify_market_freshness(source_ts, feature_ts), note)


def validate_event_feature(feature, model_input_value, feature_ts, raw_column, indicator_id):
    note = 'Derived event-pulse flag not proven here; dashboard keeps it unresolved unless directly reconstructible.'
    return unresolved_feature_result(
        feature,
        model_input_value,
        SILVER_HIST_TABLE,
        feature,
        'N/A',
        note,
        lineage_source='Derived flag feature',
        resolution_status='Unresolved (event flag reconstruction skipped)',
    )

def validate_feature_against_model_row(feature, model_input_value, feature_ts):
    spec = RAW_FEATURE_SPECS.get(feature, {})
    feature_type = spec.get('type')
    if feature_type == 'macro':
        return validate_macro_feature(feature, model_input_value, feature_ts)
    if feature_type == 'rt_vix':
        return validate_rt_vix_feature(feature, model_input_value, feature_ts)
    if feature == 'VIX_Inversion_Z':
        return validate_vix_inversion_feature(model_input_value, feature_ts)
    if feature == 'Price_Mom_24H':
        return validate_price_momentum_feature(model_input_value, feature_ts)
    if feature == 'CPI_Event_Active':
        return validate_event_feature(feature, model_input_value, feature_ts, 'CPIAUCSL', 'CPIAUCSL')
    if feature == 'NFP_Event_Active':
        return validate_event_feature(feature, model_input_value, feature_ts, 'PAYEMS', 'PAYEMS')
    return unresolved_feature_result(feature, model_input_value, spec.get('source_table', 'N/A'), feature, spec.get('source_field', 'N/A'), spec.get('note', 'Direct raw source not displayed'))


latest_feed = latest_row(FEED_TABLE, 'feed_ts')
latest_health = latest_row(HEALTH_TABLE, 'rollup_ts')
latest_perf = latest_row(PERF_TABLE, 'rollup_ts')
execution_mode = _as_text(latest_feed.get('execution_mode', ''), '') if hasattr(latest_feed, 'get') else ''
if not execution_mode:
    execution_mode = _as_text(latest_perf.get('execution_mode', ''), '')
safe_execution_mode = execution_mode.replace("'", "''") if execution_mode else ''
mode_filter = '' if not safe_execution_mode else f"WHERE execution_mode = '{safe_execution_mode}'"

perf_filter_clauses = [f"rollup_ts >= TIMESTAMP('{MONITORING_START_SQL}')"]
if safe_execution_mode:
    perf_filter_clauses.insert(0, f"execution_mode = '{safe_execution_mode}'")
perf_where_clause = '' if not perf_filter_clauses else 'WHERE ' + ' AND '.join(perf_filter_clauses)

perf_history = query_to_pandas(f"""
    SELECT *
    FROM {PERF_TABLE}
    {perf_where_clause}
    ORDER BY rollup_ts ASC
""")

health_history = query_to_pandas(f"""
    SELECT *
    FROM {HEALTH_TABLE}
    {mode_filter}
    ORDER BY rollup_ts DESC
    LIMIT {LOOKBACK_HOURS}
""")

signal_history = query_to_pandas(f"""
    SELECT *
    FROM (
      SELECT timestamp, regime_name, CAST(regime_int AS INT) AS regime_int, position
      FROM {REGIME_SIGNAL_TABLE}
      ORDER BY timestamp DESC
      LIMIT {LOOKBACK_HOURS}
    ) t
    ORDER BY timestamp ASC
""")

silver_cols = table_columns(SILVER_RT_TABLE)
if silver_cols:
    requested = [f for f in FEATURES if f in silver_cols]
    query_cols = ', '.join(['timestamp'] + requested)
    feature_history = query_to_pandas(
        f"SELECT {query_cols} FROM {SILVER_RT_TABLE} WHERE timestamp >= TIMESTAMP('{MONITORING_START_SQL}') ORDER BY timestamp ASC"
    )
else:
    feature_history = pd.DataFrame(columns=['timestamp'] + FEATURES)
if 'timestamp' in feature_history.columns:
    feature_history['timestamp'] = to_display_ist_series(feature_history['timestamp'])
    feature_history = feature_history.sort_values('timestamp').reset_index(drop=True)
    for col in [c for c in feature_history.columns if c != 'timestamp']:
        feature_history[col] = pd.to_numeric(feature_history[col], errors='coerce')

trade_history = query_to_pandas(
    f"SELECT * FROM {TRADE_LOG_TABLE} WHERE trade_ts >= TIMESTAMP('{MONITORING_START_SQL}') ORDER BY trade_ts ASC",
    empty_columns=['trade_ts']
)
if 'trade_ts' in trade_history.columns:
    trade_history['trade_ts'] = to_display_ist_series(trade_history['trade_ts'])
    if 'signal_ts' in trade_history.columns:
        trade_history['signal_ts'] = to_display_ist_series(trade_history['signal_ts'])
    trade_history = trade_history.sort_values('trade_ts').reset_index(drop=True)

v4_strategy_equity_history = pd.DataFrame(columns=['timestamp'])
if table_exists(V4_STRATEGY_EQUITY_TABLE):
    v4_strategy_equity_history = query_to_pandas(
        f"""
        SELECT
            timestamp,
            strategy_stance,
            strategy_cash,
            strategy_position_symbol,
            strategy_position_qty,
            strategy_position_market_value,
            strategy_equity_gross,
            strategy_equity_net,
            cumulative_transaction_cost,
            strategy_return_gross,
            strategy_return_net,
            source
        FROM {V4_STRATEGY_EQUITY_TABLE}
        WHERE timestamp >= TIMESTAMP('{V4_STRATEGY_PERFORMANCE_START_SQL}')
        ORDER BY timestamp ASC
        """,
        empty_columns=['timestamp'],
    )
if 'timestamp' in v4_strategy_equity_history.columns:
    v4_strategy_equity_history['timestamp'] = to_display_ist_series(v4_strategy_equity_history['timestamp'])
    for c in [
        'strategy_cash',
        'strategy_position_qty',
        'strategy_position_market_value',
        'strategy_equity_gross',
        'strategy_equity_net',
        'cumulative_transaction_cost',
        'strategy_return_gross',
        'strategy_return_net',
    ]:
        if c in v4_strategy_equity_history.columns:
            v4_strategy_equity_history[c] = pd.to_numeric(v4_strategy_equity_history[c], errors='coerce')
    v4_strategy_equity_history = v4_strategy_equity_history.sort_values('timestamp').reset_index(drop=True)

v4_strategy_trade_history = pd.DataFrame(columns=['broker_event_ts'])
if table_exists(V4_STRATEGY_LEDGER_TABLE):
    v4_strategy_ledger_columns = set(table_columns(V4_STRATEGY_LEDGER_TABLE))
    v4_trade_ts_col = 'broker_event_ts' if 'broker_event_ts' in v4_strategy_ledger_columns else 'event_ts'

    def _v4_ledger_expr(column_name, default_expr, alias=None):
        alias = alias or column_name
        if column_name in v4_strategy_ledger_columns:
            return f'{column_name} AS {alias}'
        return f'{default_expr} AS {alias}'

    v4_trade_select_exprs = [
        f'{v4_trade_ts_col} AS broker_event_ts',
        _v4_ledger_expr('strategy_stance_before', 'CAST(NULL AS STRING)'),
        _v4_ledger_expr('strategy_stance_after', 'CAST(NULL AS STRING)'),
        _v4_ledger_expr('symbol', 'CAST(NULL AS STRING)'),
        _v4_ledger_expr('side', 'CAST(NULL AS STRING)'),
        _v4_ledger_expr('quantity', 'CAST(NULL AS DOUBLE)'),
        _v4_ledger_expr('price', 'CAST(NULL AS DOUBLE)'),
        _v4_ledger_expr('gross_notional', 'CAST(NULL AS DOUBLE)'),
        _v4_ledger_expr('total_transaction_cost', 'CAST(0.0 AS DOUBLE)'),
        'CAST(gross_notional AS DOUBLE) * 0.0005 AS modeled_5bps_cost' if 'gross_notional' in v4_strategy_ledger_columns else 'CAST(NULL AS DOUBLE) AS modeled_5bps_cost',
        _v4_ledger_expr('cost_observation_quality', 'CAST(NULL AS STRING)'),
        _v4_ledger_expr('all_in_cost_confirmed', 'CAST(false AS BOOLEAN)'),
        _v4_ledger_expr('trade_id', 'CAST(NULL AS STRING)'),
        _v4_ledger_expr('position_id', 'CAST(NULL AS STRING)'),
    ]
    v4_trade_symbol_filter = "UPPER(symbol) IN ('SPY', 'QQQ')" if 'symbol' in v4_strategy_ledger_columns else '1 = 0'
    v4_trade_strategy_filter = 'CAST(is_v4_strategy_trade AS BOOLEAN) = true' if 'is_v4_strategy_trade' in v4_strategy_ledger_columns else '1 = 0'
    v4_strategy_trade_history = query_to_pandas(
        f"""
        SELECT
            {', '.join(v4_trade_select_exprs)}
        FROM {V4_STRATEGY_LEDGER_TABLE}
        WHERE {v4_trade_strategy_filter}
          AND {v4_trade_symbol_filter}
        ORDER BY {v4_trade_ts_col} ASC
        """,
        empty_columns=['broker_event_ts'],
    )
if 'broker_event_ts' in v4_strategy_trade_history.columns:
    v4_strategy_trade_history['broker_event_ts'] = to_display_ist_series(v4_strategy_trade_history['broker_event_ts'])
    for c in ['quantity', 'price', 'gross_notional', 'total_transaction_cost', 'modeled_5bps_cost']:
        if c in v4_strategy_trade_history.columns:
            v4_strategy_trade_history[c] = pd.to_numeric(v4_strategy_trade_history[c], errors='coerce')
    v4_strategy_trade_history['symbol'] = v4_strategy_trade_history.get('symbol', pd.Series(dtype='object')).astype(str).str.upper()
    v4_strategy_trade_history['side'] = v4_strategy_trade_history.get('side', pd.Series(dtype='object')).astype(str).str.upper()
    v4_strategy_trade_history = v4_strategy_trade_history.sort_values('broker_event_ts').reset_index(drop=True)

model, model_version, model_path = load_model()
importance_frame = feature_importance(model)
feature_importance_available = bool(importance_frame['importance_pct'].notna().any())
importance_values = pd.to_numeric(importance_frame.get('importance_pct'), errors='coerce').dropna()
if not feature_importance_available or importance_values.empty or np.isclose(float(importance_values.std(ddof=0)), 0.0):
    raise RuntimeError('Model-derived feature importance validation failed; refusing to render missing or equal-weight importance values.')
importance_value_map = dict(zip(importance_frame['feature'], importance_frame['importance']))
importance_pct_map = dict(zip(importance_frame['feature'], importance_frame['importance_pct']))
model_probability_history = pd.DataFrame(columns=['timestamp'] + list(PROBABILITY_LABELS.keys()))
if not feature_history.empty and 'timestamp' in feature_history.columns:
    model_probability_history = pd.concat(
        [
            feature_history[['timestamp']].reset_index(drop=True),
            compute_probabilities(feature_history, model).reset_index(drop=True),
        ],
        axis=1,
    )
    model_probability_history['timestamp'] = to_display_ist_series(model_probability_history['timestamp'])

feed_probability_history = load_probability_history_from_table(FEED_TABLE, 'feed_ts')
signal_probability_history = load_probability_history_from_table(REGIME_SIGNAL_TABLE, 'timestamp')
perf_probability_history = load_probability_history_from_table(PERF_TABLE, 'rollup_ts')
probability_history = combine_probability_histories(
    model_probability_history,
    feed_probability_history,
    signal_probability_history,
    perf_probability_history,
)

perf_history = perf_history.sort_values('rollup_ts').reset_index(drop=True) if not perf_history.empty else perf_history
if not perf_history.empty:
    perf_history['rollup_ts'] = to_display_ist_series(perf_history['rollup_ts'])
    for c in ['account_equity', 'account_cash', 'drawdown_pct', 'regime_drawdown_pct', 'alpha_pct', 'strategy_return_1h', 'alpha_1h', 'strategy_return_1d', 'alpha_1d']:
        if c in perf_history.columns:
            perf_history[c] = pd.to_numeric(perf_history[c], errors='coerce')

if not health_history.empty:
    health_history['rollup_ts'] = to_display_ist_series(health_history['rollup_ts'])
    health_history = health_history.sort_values('rollup_ts').reset_index(drop=True)

if not signal_history.empty:
    signal_history['timestamp'] = to_display_ist_series(signal_history['timestamp'])
    signal_history = signal_history.sort_values('timestamp').reset_index(drop=True)
    signal_history['regime_int'] = pd.to_numeric(signal_history['regime_int'], errors='coerce')
    signal_history['regime_name'] = [regime_label(name, code) for name, code in zip(signal_history.get('regime_name', pd.Series(index=signal_history.index, dtype='object')), signal_history.get('regime_int', pd.Series(index=signal_history.index, dtype='float64')))]

regime_history = signal_history.copy()
regime_intervals = build_regime_intervals(signal_history)
latest_probability_rows = probability_history.dropna(subset=list(PROBABILITY_LABELS.keys()), how='all') if not probability_history.empty else pd.DataFrame()
latest_probability = latest_probability_rows.iloc[-1] if not latest_probability_rows.empty else pd.Series(dtype='float64')
completed_session_start_ny, completed_session_end_ny = latest_completed_session_bounds()
completed_session_label = f"{fmt_timestamp_long(completed_session_start_ny, NEW_YORK_TZ, 'EDT' if completed_session_start_ny.dst() != pd.Timedelta(0) else 'EST')} to {fmt_timestamp_long(completed_session_end_ny, NEW_YORK_TZ, 'EDT' if completed_session_end_ny.dst() != pd.Timedelta(0) else 'EST')}"
signal_completed_session_history = filter_to_completed_session(signal_history, 'timestamp')
trade_completed_session_history = filter_to_completed_session(trade_history, 'trade_ts')
signal_session_history = _append_open_context_row(signal_completed_session_history, signal_history, 'timestamp', completed_session_start_ny)
trade_session_history = _append_open_context_row(trade_completed_session_history, trade_history, 'trade_ts', completed_session_start_ny)

perf_window, perf_window_start, perf_window_mode = select_fixed_monitoring_window(perf_history, 'rollup_ts', MONITORING_START_TS)
regime_since_start_window, regime_since_start_start, _ = select_fixed_monitoring_window(regime_history, 'timestamp', MONITORING_START_TS)
regime_rolling_window, regime_rolling_start, _ = select_rolling_window(regime_history, 'timestamp', ROLLING_WINDOW_HOURS)
probability_since_start_window, probability_since_start_start, _ = select_fixed_monitoring_window(probability_history, 'timestamp', MONITORING_START_TS)
probability_rolling_window, probability_rolling_start, _ = select_rolling_window(probability_history, 'timestamp', ROLLING_WINDOW_HOURS)
feature_since_start_window, feature_since_start_start, _ = select_fixed_monitoring_window(feature_history, 'timestamp', MONITORING_START_TS)
feature_rolling_window, feature_rolling_start, _ = select_rolling_window(feature_history, 'timestamp', ROLLING_WINDOW_HOURS)
health_window, health_window_start, health_window_mode = select_live_monitoring_window(health_history, 'rollup_ts', HEALTH_WINDOW_FALLBACK_HOURS)
perf_recent_48h_window, perf_recent_48h_start, _ = select_rolling_window(perf_history, 'rollup_ts', 48)
perf_window_note = format_fixed_monitoring_note(perf_window_start)
regime_window_note = format_fixed_monitoring_note(regime_since_start_start)
probability_since_start_note = format_fixed_monitoring_note(probability_since_start_start)
probability_rolling_note = format_rolling_window_note(probability_rolling_start, ROLLING_WINDOW_HOURS)
feature_since_start_note = format_fixed_monitoring_note(feature_since_start_start)
feature_rolling_note = format_rolling_window_note(feature_rolling_start, ROLLING_WINDOW_HOURS)
health_window_note = format_monitoring_window_note(health_window_mode, health_window_start)
perf_recent_48h_note = format_rolling_window_note(perf_recent_48h_start, 48)

displayHTML("""
<style>
h2 { font-size: 30px !important; font-weight: 700 !important; margin: 18px 0 10px 0 !important; }
h3 { font-size: 24px !important; font-weight: 700 !important; margin: 16px 0 8px 0 !important; }
h4 { font-size: 20px !important; font-weight: 650 !important; margin: 12px 0 6px 0 !important; }
h5 { font-size: 17px !important; font-weight: 650 !important; margin: 10px 0 6px 0 !important; }
</style>
""")

displayHTML('<h2>1. Executive decision header</h2>')
displayHTML(f'<p><b>Production state:</b> Improved ALFRED logic is live in production. Current promoted model: <code>{_as_text(model_version, "N/A")}</code>. Strategy and feature monitoring now use a fixed post-release window from <code>{MONITORING_START_DATE}</code> onward. Historical PIT-join review remains open for CPIAUCSL / GDPC1 / INDPRO.</p>')
prob_label, prob_value = top_named_value_from_row(latest_probability, PROBABILITY_LABELS)
current_probability_value = f'{prob_label}: {fmt_pct(prob_value)}' if pd.notna(_to_float(prob_value)) else None

def _first_present_value(*values):
    for value in values:
        if not _is_missing(value):
            numeric = _to_float(value)
            if pd.notna(numeric):
                return numeric
            return value
    return np.nan


def build_report_account_snapshot(latest_feed_row, latest_perf_row):
    # Account equity is the live marked account value used by the dashboard/performance
    # rollup. Do not silently substitute cash/balance for equity unless equity is
    # unavailable in both dashboard/performance and feed sources.
    equity = _first_present_value(latest_perf_row.get('account_equity'), latest_feed_row.get('account_equity'))
    cash = _first_present_value(latest_feed_row.get('account_cash'), latest_perf_row.get('account_cash'))
    balance = _first_present_value(latest_feed_row.get('account_balance'), latest_perf_row.get('account_balance'))
    equity_source = 'performance_rollup.account_equity' if not _is_missing(latest_perf_row.get('account_equity')) else 'dashboard_feed.account_equity'
    if _is_missing(equity):
        equity = _first_present_value(balance, cash)
        equity_source = 'explicit fallback to account_balance/account_cash because account_equity is unavailable'
    return {
        'account_equity': equity,
        'account_cash': cash,
        'account_balance': balance,
        'account_equity_source': equity_source,
    }


report_account_snapshot = build_report_account_snapshot(latest_feed, latest_perf)



def _parse_report_json_payload(value):
    if _is_missing(value):
        return None
    if isinstance(value, (dict, list)):
        return value
    if isinstance(value, str):
        text = value.strip()
        if not text or text.upper() == 'N/A':
            return None
        try:
            return json.loads(text)
        except Exception:
            return None
    return None


def _first_text_value(row, columns, default='N/A'):
    for column in columns:
        value = row.get(column) if hasattr(row, 'get') else None
        if not _is_missing(value):
            text = str(value).strip()
            if text and text.upper() != 'N/A':
                return text
    return default


def _first_numeric_value(row, columns):
    for column in columns:
        value = row.get(column) if hasattr(row, 'get') else None
        numeric = _to_float(value)
        if pd.notna(numeric):
            return numeric
    return np.nan


def _extract_position_items(positions_payload):
    parsed = _parse_report_json_payload(positions_payload)
    if parsed is None:
        return []
    if isinstance(parsed, dict):
        for key in ['positions', 'Positions', 'PositionView', 'Data', 'items']:
            candidate = parsed.get(key)
            if isinstance(candidate, list):
                return candidate
        return [parsed]
    if isinstance(parsed, list):
        return parsed
    return []


def _position_symbol(position, fallback_symbol='position'):
    for key in ['symbol', 'Symbol', 'ticker', 'Ticker', 'DisplayAndFormat', 'Description', 'AssetType']:
        value = position.get(key) if isinstance(position, dict) else None
        if isinstance(value, dict):
            value = value.get('Symbol') or value.get('Description') or value.get('Uic')
        if not _is_missing(value) and str(value).strip():
            text = str(value).strip()
            if not text.isdigit():
                return text
    return fallback_symbol or 'position'


def _position_quantity(position):
    direct_keys = [
        'qty', 'Qty', 'quantity', 'Quantity', 'amount', 'Amount', 'shares', 'Shares',
        'NetPosition', 'NetPositionAmount', 'OpenQuantity', 'PositionAmount', 'Holding',
    ]
    if isinstance(position, dict):
        for key in direct_keys:
            numeric = _to_float(position.get(key))
            if pd.notna(numeric):
                return numeric

    def _walk_quantity(value):
        if isinstance(value, dict):
            for key, nested in value.items():
                lowered = str(key).lower()
                if any(token in lowered for token in ['qty', 'quantity', 'amount', 'shares', 'netposition', 'positionamount']):
                    numeric = _to_float(nested)
                    if pd.notna(numeric):
                        return numeric
                found = _walk_quantity(nested)
                if pd.notna(found):
                    return found
        elif isinstance(value, list):
            for item in value:
                found = _walk_quantity(item)
                if pd.notna(found):
                    return found
        return np.nan

    return _walk_quantity(position)


def _format_share_quantity(quantity):
    quantity_text = f'{quantity:,.0f}' if abs(quantity - round(quantity)) < 1e-9 else f'{quantity:,.4f}'.rstrip('0').rstrip('.')
    share_label = 'share' if abs(abs(quantity) - 1) < 1e-9 else 'shares'
    return quantity_text, share_label


def _first_detected_position_symbol(row):
    fallback_symbol = _first_text_value(row, ['target_ticker', 'actual_ticker', 'target_symbol'], default='')
    positions = _extract_position_items(row.get('positions_json') if hasattr(row, 'get') else None)
    for position in positions:
        if not isinstance(position, dict):
            continue
        quantity = _position_quantity(position)
        if pd.notna(quantity) and abs(quantity) <= 1e-12:
            continue
        symbol = _position_symbol(position, fallback_symbol=fallback_symbol or 'Saxo position')
        if symbol:
            return symbol
    return fallback_symbol


def format_actual_saxo_holding(row):
    fallback_symbol = _first_text_value(row, ['target_ticker', 'actual_ticker', 'target_symbol'], default='')
    positions = _extract_position_items(row.get('positions_json') if hasattr(row, 'get') else None)
    summaries = []
    for position in positions:
        if not isinstance(position, dict):
            continue
        quantity = _position_quantity(position)
        if pd.notna(quantity) and abs(quantity) <= 1e-12:
            continue
        symbol = _position_symbol(position, fallback_symbol=fallback_symbol or 'Saxo position')
        if pd.notna(quantity):
            quantity_text, share_label = _format_share_quantity(quantity)
            summaries.append(f'{symbol} — {quantity_text} {share_label}')
        else:
            summaries.append(f'{symbol} position detected')
    if summaries:
        return ', '.join(summaries[:3]) + (' +' if len(summaries) > 3 else '')

    count = _first_numeric_value(row, ['positions_count', 'position_count', 'open_positions_count'])
    if pd.notna(count) and count <= 0:
        return 'CASH / no open ETF position'
    if pd.notna(count) and count > 0:
        symbol = fallback_symbol or 'Saxo'
        return f'{symbol} position detected'
    return 'N/A'


def format_position_market_value(row):
    value = _first_numeric_value(row, ['positions_market_value', 'position_market_value', 'holdings_market_value'])
    return fmt_money(value)


def format_estimated_uninvested_equity(account_snapshot, row):
    equity = _to_float(account_snapshot.get('account_equity') if isinstance(account_snapshot, dict) else np.nan)
    market_value = _first_numeric_value(row, ['positions_market_value', 'position_market_value', 'holdings_market_value'])
    if pd.isna(equity) or pd.isna(market_value):
        return 'N/A'
    return fmt_money(equity - market_value)


def format_model_target_allocation(row):
    target_position = _first_text_value(row, ['target_position', 'position', 'model_target_position'], default='N/A')
    target_ticker = _first_text_value(row, ['target_ticker', 'actual_ticker', 'target_symbol'], default='')
    if target_position == 'N/A' and target_ticker:
        return target_ticker
    if target_ticker and target_ticker not in target_position:
        return f'{target_position} {target_ticker}'.strip()
    return target_position


def summarize_execution_status(row):
    error_reason = _first_text_value(row, ['last_error_reason', 'error_reason'], default='')
    if error_reason:
        return f'Error: {error_reason}'

    skip_reason = _first_text_value(row, ['last_skip_reason', 'skip_reason'], default='')
    orders_text = _first_text_value(row, ['orders_placed', 'order_summary', 'last_order_status'], default='')
    action = _first_text_value(row, ['last_execution_action', 'execution_action'], default='')
    combined = ' | '.join([part for part in [action, orders_text, skip_reason] if part]).strip()
    lowered = combined.lower()

    has_filled = any(token in lowered for token in ['filled', 'fill', 'executed'])
    has_skip = any(token in lowered for token in ['skip', 'skipped', 'minimum', 'min order', 'cash precheck', 'insufficient cash'])
    holding_symbol = _first_detected_position_symbol(row)
    holding_detected = bool(holding_symbol) and holding_symbol.upper() not in {'CASH', 'N/A'}
    if has_skip and holding_detected:
        return f'Holding {holding_symbol}; later top-up skipped due to Saxo minimum-order / cash precheck.'
    if has_filled and has_skip:
        return 'Initial target trade filled; subsequent top-up skipped due to Saxo minimum-order / cash precheck constraints.'
    if has_skip:
        return f'Skipped: {skip_reason or orders_text or action}'
    if has_filled:
        return 'Target trade filled.'
    if combined:
        return combined
    return 'N/A'


report_model_target_allocation = format_model_target_allocation(latest_feed)
report_actual_saxo_holding = format_actual_saxo_holding(latest_feed)
report_position_market_value = format_position_market_value(latest_feed)
report_estimated_uninvested_equity = format_estimated_uninvested_equity(report_account_snapshot, latest_feed)
report_execution_status = summarize_execution_status(latest_feed)
kpi = [
    ('Current regime', _as_text(latest_feed.get('regime_name', 'N/A'))),
    ('Current model probability', current_probability_value or 'Unavailable from dashboard/model history'),
    ('Historical regime mix (Since 20-Apr-2026)', historical_mix_text(latest_feed)),
    ('Model target allocation', report_model_target_allocation),
    ('Actual Saxo holding', report_actual_saxo_holding),
    ('Position market value', report_position_market_value),
    ('Estimated uninvested equity', report_estimated_uninvested_equity),
    ('Execution mode', _as_text(latest_feed.get('execution_mode', latest_perf.get('execution_mode', 'N/A')))),
    ('Execution status', report_execution_status),
    ('Account equity', fmt_money(report_account_snapshot.get('account_equity'))),
    ('1h strategy', fmt_pct(latest_perf.get('strategy_return_1h'))),
    ('1h alpha', fmt_pct(latest_perf.get('alpha_1h'))),
    ('1d strategy', fmt_pct(latest_perf.get('strategy_return_1d'))),
    ('1d alpha', fmt_pct(latest_perf.get('alpha_1d'))),
    ('Current drawdown (Since 20-Apr-2026)', fmt_pct(latest_feed.get('drawdown_since_start_pct', latest_feed.get('drawdown_pct')))),
    ('Freshness vs latest market / macro / audit source timestamps', fmt_freshness_summary(latest_feed)),
    ('Audit (latest Saxo model decision and trade/bias audit record)', fmt_audit_summary(latest_feed)),
    ('Last execution action', _as_text(latest_feed.get('last_execution_action'))),
    ('Skip reason', _as_text(latest_feed.get('last_skip_reason'))),
    ('Error reason', _as_text(latest_feed.get('last_error_reason'))),
]
display(pd.DataFrame(kpi, columns=['Metric', 'Value']))

displayHTML('<h3>1b. Clocks and market status</h3>')
market_status_df = pd.DataFrame(build_market_status_snapshot().items(), columns=['Metric', 'Value'])
display(market_status_df)
displayHTML(f'<p><i>All dashboard timestamps are shown in {DISPLAY_TZ_LABEL} unless explicitly labeled as New York time. Market status is based on regular US equity session hours and weekends only.</i></p>')

displayHTML('<h2>2. Strategy vs SPY performance</h2>')
displayHTML(f'<p><i>{perf_window_note}. Strategy and SPY are rebased to the first available observation on or after {MONITORING_START_DATE}.</i></p>')
if perf_window.empty:
    print('No performance history available for the fixed post-release monitoring window.')
else:
    aligned = perf_window.copy()
    aligned = aligned.dropna(subset=['rollup_ts', 'account_equity'])
    aligned = aligned.sort_values('rollup_ts').reset_index(drop=True)

    if aligned.empty:
        print('No account equity history available for the fixed post-release monitoring window.')
    else:
        first_equity = float(pd.to_numeric(aligned['account_equity'], errors='coerce').dropna().iloc[0])
        aligned['strategy_equity_index'] = pd.to_numeric(aligned['account_equity'], errors='coerce') / first_equity
        aligned['strategy_return'] = aligned['strategy_equity_index'] - 1.0
        aligned['strategy_drawdown'] = compute_running_drawdown(aligned['strategy_equity_index'])

        spy_df = pd.DataFrame()
        if 'spy_close' in trade_history.columns and 'trade_ts' in trade_history.columns:
            spy_df = trade_history[['trade_ts', 'spy_close']].copy()
            spy_df['spy_close'] = pd.to_numeric(spy_df['spy_close'], errors='coerce')
            spy_df = spy_df.dropna(subset=['trade_ts', 'spy_close']).sort_values('trade_ts')

        if not spy_df.empty:
            aligned = pd.merge_asof(
                aligned.sort_values('rollup_ts'),
                spy_df.sort_values('trade_ts'),
                left_on='rollup_ts',
                right_on='trade_ts',
                direction='backward'
            )
            if aligned['spy_close'].notna().any():
                first_spy = float(aligned['spy_close'].dropna().iloc[0])
                aligned['spy_equity_index'] = aligned['spy_close'] / first_spy
                aligned['spy_return'] = aligned['spy_equity_index'] - 1.0
                aligned['spy_drawdown'] = compute_running_drawdown(aligned['spy_equity_index'])

        fig = plt.figure(figsize=(16, 8.6), constrained_layout=False)
        fig.subplots_adjust(top=0.90, bottom=0.12, hspace=0.32)
        gs = fig.add_gridspec(2, 1, height_ratios=[3.2, 1.5])
        ax_perf = fig.add_subplot(gs[0, 0])
        ax_dd = fig.add_subplot(gs[1, 0], sharex=ax_perf)

        if not regime_intervals.empty:
            window_start = aligned['rollup_ts'].iloc[0]
            window_end = aligned['rollup_ts'].iloc[-1]
            for idx, row in regime_intervals.iterrows():
                start_t = row['start_ts']
                end_t = row['end_ts'] if pd.notna(row.get('end_ts')) else window_end
                if pd.isna(start_t) or pd.isna(end_t):
                    continue
                if idx == 0 and start_t > window_start:
                    start_t = window_start
                if start_t < window_start:
                    start_t = window_start
                if start_t >= window_end or end_t <= start_t:
                    continue
                color = REGIME_COLOR.get(_as_text(row['regime_name'], 'Unknown'), '#9E9E9E')
                ax_perf.axvspan(start_t, min(end_t, window_end), color=color, alpha=0.12)

        ax_perf.plot(aligned['rollup_ts'], aligned['strategy_return'] * 100, linewidth=2.3, label='Strategy')
        if 'spy_return' in aligned.columns and aligned['spy_return'].notna().any():
            ax_perf.plot(aligned['rollup_ts'], aligned['spy_return'] * 100, linewidth=1.7, linestyle='--', label='SPY proxy', alpha=0.85)

        latest_trade_marker = None
        if 'trade_ts' in aligned.columns and aligned['trade_ts'].notna().any():
            latest_trade_marker = aligned.dropna(subset=['trade_ts']).tail(1)
        if latest_trade_marker is not None and not latest_trade_marker.empty:
            trade_x = latest_trade_marker['rollup_ts'].iloc[-1]
            trade_y = latest_trade_marker['strategy_return'].iloc[-1] * 100
            ax_perf.scatter(trade_x, trade_y, s=34, color='#1F4E79', alpha=0.85, zorder=5)

        ax_perf.set_title(f'Saxo strategy vs SPY | {perf_window_note}', fontsize=17, fontweight='bold', loc='left')
        ax_perf.set_ylabel('Return from start (%)')
        apply_dual_time_axes(ax_perf, bottom_label=f'Timestamp ({DISPLAY_TZ_LABEL})')
        ax_perf.grid(alpha=0.3)
        ax_perf.legend(loc='upper left')

        ax_dd.plot(aligned['rollup_ts'], aligned['strategy_drawdown'] * 100, linewidth=1.9, color='#B03A2E', label='Strategy drawdown')
        if 'spy_drawdown' in aligned.columns and aligned['spy_drawdown'].notna().any():
            ax_dd.plot(aligned['rollup_ts'], aligned['spy_drawdown'] * 100, linewidth=1.5, color='#1F4E79', linestyle='--', label='SPY drawdown')
        ax_dd.axhline(0, color='black', linestyle='--', alpha=0.5)
        ax_dd.set_ylabel('Drawdown (%)')
        ax_dd.set_title(f'Drawdown from {MONITORING_START_DATE} start', fontsize=16, fontweight='bold', loc='left')
        apply_dual_time_axes(ax_dd, bottom_label=f'Timestamp ({DISPLAY_TZ_LABEL})')
        ax_dd.grid(alpha=0.3)
        ax_dd.legend(loc='lower right')
        plt.show()


displayHTML('<h2>3. Historical regime / signal monitoring</h2>')
displayHTML(f'<p><i>Model probabilities below are computed on the fly from the current promoted model using feature history. Regime history remains sourced from bronze_rt_regime_signal. Fixed monitoring scope: {regime_window_note}. Rolling monitoring scope: {format_rolling_window_note(regime_rolling_start, ROLLING_WINDOW_HOURS)}.</i></p>')
if not regime_history.empty:
    regime_head_cols = [c for c in ['timestamp', 'regime_name', 'regime_int', 'position'] if c in regime_history.columns]
    regime_head_source = regime_since_start_window if not regime_since_start_window.empty else regime_history
    regime_head = regime_head_source[regime_head_cols].copy()
    regime_head['timestamp'] = regime_head['timestamp'].apply(fmt_timestamp)
    regime_head = regime_head.sort_values('timestamp', ascending=False).head(24)
    regime_head = regime_head.rename(columns={'timestamp': 'Timestamp', 'regime_name': 'Regime', 'regime_int': 'Regime code', 'position': 'Target position'})
    display(regime_head)

    render_probability_and_regime_window(
        probability_since_start_window,
        regime_since_start_window,
        f'Historical Model Probabilities — Since 20-Apr-2026',
        f'Hourly Regime History — Since 20-Apr-2026',
    )
    render_probability_and_regime_window(
        probability_rolling_window,
        regime_rolling_window,
        f'Historical Model Probabilities — Last 48 Hours',
        f'Hourly Regime History — Last 48 Hours',
    )
else:
    print('No regime signal history available from bronze_rt_regime_signal.')


displayHTML('<h2 style="margin-bottom: 0.35rem;">4. Current feature state</h2>')
current_features = pd.DataFrame(columns=['feature', 'label', 'model_importance_pct', 'current_z_score', 'delta_vs_prior', 'weighted_move_proxy'])
if feature_history.empty:
    print('No feature history available in silver.')
else:
    latest = feature_history.tail(1).iloc[0]
    prev = feature_history.tail(2).iloc[0] if len(feature_history) > 1 else pd.Series(dtype=float)
    rows = []
    for feature in FEATURES:
        if feature not in feature_history.columns:
            continue
        cur = _to_float(latest.get(feature))
        p = _to_float(prev.get(feature)) if not prev.empty else np.nan
        delta_z = cur - p if not pd.isna(cur) and not pd.isna(p) else np.nan
        importance_pct = _to_float(importance_pct_map.get(feature, np.nan))
        rows.append((
            feature,
            FEATURE_LABELS.get(feature, feature),
            importance_pct,
            _to_float(cur),
            delta_z,
            abs(importance_pct * delta_z) if not pd.isna(delta_z) and not pd.isna(importance_pct) else np.nan,
        ))

    current_features = pd.DataFrame(rows, columns=['feature', 'label', 'model_importance_pct', 'current_z_score', 'delta_vs_prior', 'weighted_move_proxy'])
    current_features = current_features.sort_values(['weighted_move_proxy', 'model_importance_pct'], ascending=[False, False]).reset_index(drop=True)
    displayHTML('<p style="margin-top: 0; margin-bottom: 0.85rem; color: #4a4a4a;"><i>Current feature state ranked for institutional monitoring on the basis of the largest model-weighted hourly move proxy, so the top rows surface the features most likely to matter for the next portfolio review.</i></p>')
    feature_display_columns = {
        'label': 'Feature',
        'model_importance_pct': 'Model Importance (%)',
        'current_z_score': 'Current z-score',
        'delta_vs_prior': 'Delta vs prev hour',
        'weighted_move_proxy': 'Weighted move proxy',
    }
    current_feature_display = current_features.drop(columns=['feature']).rename(columns=feature_display_columns)
    current_feature_styler = (
        current_feature_display.style
        .format({
            'Model Importance (%)': '{:.2%}',
            'Current z-score': '{:+.3f}',
            'Delta vs prev hour': '{:+.3f}',
            'Weighted move proxy': '{:.4f}',
        }, na_rep='N/A')
        .hide(axis='index')
        .set_properties(**{
            'text-align': 'left',
            'font-size': '12px',
            'padding': '8px 10px',
            'border': '1px solid #D9E2F3',
            'white-space': 'nowrap',
        })
        .set_table_styles([
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%'), ('margin', '0 0 1.1rem 0'), ('font-family', 'Times New Roman, Times, serif'), ('border', '1px solid #D9E2F3')]},
            {'selector': 'th', 'props': [('background-color', '#DCE6F1'), ('color', '#1F1F1F'), ('font-size', '12px'), ('font-weight', '700'), ('text-align', 'left'), ('padding', '9px 10px'), ('border', '1px solid #B4C7E7')]},
            {'selector': 'td', 'props': [('background-color', '#FFFFFF')]},
            {'selector': 'tbody tr:nth-child(even) td', 'props': [('background-color', '#F7F9FC')]},
            {'selector': 'caption', 'props': [('caption-side', 'top'), ('text-align', 'left'), ('font-size', '14px'), ('font-weight', '700'), ('padding', '0 0 0.4rem 0'), ('color', '#1F1F1F')]},
        ])
        .set_caption('Current feature state')
    )
    display(current_feature_styler)
    displayHTML(f'<p><i>Z-scores use a rolling 1-year lookback of 8760 hourly rows. Delta is the change versus the previous hourly bar. Weighted move proxy is model importance multiplied by the absolute latest hourly z-score move. Ranking is ordered for institutional triage so the most decision-relevant features appear first.</i></p>')
    displayHTML('<div style="height: 0.35rem;"></div>')

displayHTML('<h3 style="margin-top: 0.25rem;">4b. Actual Macro Inputs Used by Model</h3>')
displayHTML('<p><i>Institutional audit of the actual macro inputs in the latest silver_rt_1h model row, reconciled to the latest ALFRED/FRED public vintage reference. This is diagnostic display logic only; production feeds and model inference are unchanged.</i></p>')
displayHTML('<p><i>Raw macro values are reconstructed for display from bronze_bt_macro_events using the Historical timing rule for historical rows. Validated live overrides use the Validated live-source gate. The model itself consumes z-scored features from silver_rt_1h. Historical macro features use a conservative release-timing buffer in the training grid. Validated live overrides can be reflected by the next scheduled RT enrichment run when available.</i></p>')
displayHTML('<p><i>For live macro overrides, the model may use a validated vendor live release before FRED/ALFRED/public reference tables expose the same observation period. Public comparison is only valid when release stage and observation period align.</i></p>')

MODEL_MACRO_INPUT_SPECS = [
    {'label': 'CPI', 'feature': 'CPIAUCSL_z', 'series_id': 'CPIAUCSL'},
    {'label': 'Core PCE', 'feature': 'PCEPI_z', 'series_id': 'PCEPI'},
    {'label': 'PCE', 'feature': 'PCE_z', 'series_id': 'PCE'},
    {'label': 'Payrolls', 'feature': 'PAYEMS_z', 'series_id': 'PAYEMS'},
    {'label': 'Unemployment', 'feature': 'UNRATE_z', 'series_id': 'UNRATE'},
    {'label': 'GDP', 'feature': 'GDPC1_z', 'series_id': 'GDPC1'},
    {'label': 'Industrial Production', 'feature': 'INDPRO_z', 'series_id': 'INDPRO'},
    {'label': 'Sentiment', 'feature': 'UMCSENT_z', 'series_id': 'UMCSENT'},
]
MODEL_MACRO_MATCH_TOLERANCE = 1e-6
MODEL_MACRO_LIVE_OVERRIDE_SERIES = set(LIVE_MACRO_OVERRIDE_INDICATORS)


def _latest_public_macro_reference(series_id):
    public_rows = fetch_alfred_public_lookup(series_id, lookback_days=ALFRED_EXPORT_LOOKBACK_DAYS)
    if public_rows.empty:
        return pd.Series(dtype='object')
    public_rows = public_rows.copy()
    public_rows['public_value'] = pd.to_numeric(public_rows['public_value'], errors='coerce')
    public_rows['realtime_start'] = _to_naive_utc(public_rows['realtime_start'])
    public_rows['observation_date'] = _to_naive_utc(public_rows['observation_date'])
    public_rows = public_rows.dropna(subset=['realtime_start'])
    if public_rows.empty:
        return pd.Series(dtype='object')
    public_rows = public_rows.sort_values(['realtime_start', 'observation_date'], ascending=[False, False])
    return public_rows.iloc[0]


def _load_bronze_macro_pit_raw_value(series_id, feature_ts):
    feature_ts_sql = sql_timestamp(feature_ts)
    if feature_ts_sql is None or not table_exists(BT_MACRO_TABLE):
        return pd.Series(dtype='object')
    bt_columns = set(table_columns(BT_MACRO_TABLE))
    value_field = 'actual' if 'actual' in bt_columns else 'observed_value' if 'observed_value' in bt_columns else None
    if value_field is None or not {'indicator_id', 'timestamp'}.issubset(bt_columns):
        return pd.Series(dtype='object')

    bronze_pit_sql = f"""
        SELECT
            indicator_id,
            timestamp AS bronze_release_ts,
            {value_field} AS resolved_raw_value
        FROM {BT_MACRO_TABLE}
        WHERE indicator_id = '{series_id}'
          AND {value_field} IS NOT NULL
          AND timestamp + INTERVAL 2 HOURS <= TIMESTAMP('{feature_ts_sql}')
        ORDER BY timestamp DESC
        LIMIT 1
    """
    return first_row(query_to_pandas(bronze_pit_sql, empty_columns=['indicator_id', 'bronze_release_ts', 'resolved_raw_value']))


def _lag_hours(later_ts, earlier_ts):
    later = _to_naive_utc(later_ts)
    earlier = _to_naive_utc(earlier_ts)
    if pd.isna(later) or pd.isna(earlier):
        return np.nan
    return (later - earlier).total_seconds() / 3600.0


def _same_observation_period(public_observation_date, live_release_ts):
    public_obs = pd.to_datetime(public_observation_date, errors='coerce')
    live_ts = pd.to_datetime(live_release_ts, errors='coerce')
    if pd.isna(public_obs) or pd.isna(live_ts):
        return False
    return public_obs.year == live_ts.year and public_obs.month == live_ts.month


def _is_validated_live_override(live_row, model_ts):
    if getattr(live_row, 'empty', True):
        return False
    release_ts = pd.to_datetime(live_row.get('source_ts'), errors='coerce')
    model_ts = pd.to_datetime(model_ts, errors='coerce')
    if pd.isna(release_ts) or pd.isna(model_ts) or release_ts > model_ts:
        return False
    stage = _as_text(live_row.get('release_stage', ''), '').lower()
    validation = _as_text(live_row.get('validation_status', ''), '').lower()
    return stage == 'final' and validation == 'validated'


def _macro_audit_match_status(series_id, source_label, resolved_raw_value, public_value, model_z_score, public_reference=None, live_row=None, model_ts=None):
    if pd.isna(resolved_raw_value):
        return 'RAW_UNRESOLVED' if pd.notna(_to_float(model_z_score)) else 'RAW_UNRESOLVED'
    if pd.isna(public_value):
        return 'PUBLIC_REFERENCE_MISSING'
    values_match = abs(resolved_raw_value - public_value) <= MODEL_MACRO_MATCH_TOLERANCE
    if values_match:
        return 'MATCH'
    if source_label == 'Live override (Finnhub)':
        public_observation_date = public_reference.get('observation_date') if public_reference is not None else pd.NaT
        live_release_ts = live_row.get('source_ts') if live_row is not None and not getattr(live_row, 'empty', True) else pd.NaT
        if not _same_observation_period(public_observation_date, live_release_ts):
            return 'LIVE_OVERRIDE_ACCEPTED' if _is_validated_live_override(live_row, model_ts) else 'PUBLIC_REFERENCE_PERIOD_MISMATCH_OR_NOT_UPDATED'
        if _is_validated_live_override(live_row, model_ts):
            return 'PUBLIC_REFERENCE_PERIOD_MISMATCH_OR_NOT_UPDATED'
    return 'MISMATCH'


latest_model_input_df = query_to_pandas(f"""
    SELECT *
    FROM {SILVER_RT_TABLE}
    ORDER BY timestamp DESC
    LIMIT 1
""", empty_columns=['timestamp'] + [spec['feature'] for spec in MODEL_MACRO_INPUT_SPECS])

if latest_model_input_df.empty:
    print(f'No latest model input row available from {SILVER_RT_TABLE}.')
else:
    latest_model_input_row = latest_model_input_df.iloc[0]
    model_input_ts = latest_model_input_row.get('timestamp')
    model_input_source = _as_text(latest_model_input_row.get('source', ''), '')
    source_indicates_live_override = bool(model_input_source) and ('live' in model_input_source.lower())
    audit_rows = []
    for spec in MODEL_MACRO_INPUT_SPECS:
        feature = spec['feature']
        series_id = spec['series_id']
        model_z_score = _to_float(latest_model_input_row.get(feature, np.nan))
        bronze_pit_row = _load_bronze_macro_pit_raw_value(series_id, model_input_ts)
        live_override_row = pd.Series(dtype='object')
        source_label = 'Carry-forward from silver'
        availability_rule = 'Historical timing rule'
        raw_value_evidence = 'Historical macro table'
        resolved_raw_value = _to_float(bronze_pit_row.get('resolved_raw_value'))

        if series_id in MODEL_MACRO_LIVE_OVERRIDE_SERIES and source_indicates_live_override:
            live_override_row = load_live_macro_source(series_id, model_input_ts)
            live_raw_value = _to_float(live_override_row.get('raw_value')) if not live_override_row.empty else np.nan
            if pd.notna(live_raw_value):
                resolved_raw_value = live_raw_value
                source_label = 'Live override (Finnhub)'
                availability_rule = 'Validated live-source gate'
                raw_value_evidence = 'Live source'

        if pd.isna(resolved_raw_value):
            source_label = 'Unresolved' if source_label != 'Live override (Finnhub)' else source_label
            availability_rule = 'unresolved' if source_label != 'Live override (Finnhub)' else availability_rule
            raw_value_evidence = 'N/A'

        public_reference = _latest_public_macro_reference(series_id)
        public_value = _to_float(public_reference.get('public_value'))
        live_release_time = live_override_row.get('source_ts') if source_label == 'Live override (Finnhub)' else pd.NaT
        release_time = live_release_time if pd.notna(pd.to_datetime(live_release_time, errors='coerce')) else public_reference.get('realtime_start')
        lag_hours = _lag_hours(model_input_ts, release_time)
        match_status = _macro_audit_match_status(series_id, source_label, resolved_raw_value, public_value, model_z_score, public_reference=public_reference, live_row=live_override_row, model_ts=model_input_ts)

        notes = []
        if source_label == 'Live override (Finnhub)':
            notes.append('Live override (Finnhub)')
            if _is_validated_live_override(live_override_row, model_input_ts):
                notes.append('LIVE_OVERRIDE_ACCEPTED')
            if match_status != 'MATCH' and not _same_observation_period(public_reference.get('observation_date'), live_override_row.get('source_ts')):
                notes.append('PUBLIC_REFERENCE_PERIOD_MISMATCH_OR_NOT_UPDATED')
        elif source_label == 'Unresolved':
            notes.append('Raw value unresolved from bronze macro PIT reconstruction')
        else:
            notes.append('Carry-forward from silver')
        if series_id == 'GDPC1':
            notes.append('Quarterly release (GDP)')
        if series_id == 'UMCSENT':
            notes.append('Final-only override (UMCSENT)')

        audit_rows.append({
            'Feature': spec['label'],
            'Model feature': feature,
            'Observation date': fmt_date(public_reference.get('observation_date')),
            'Release time': fmt_timestamp_pair(release_time),
            'Resolved raw value used': fmt_raw_snapshot_value(resolved_raw_value),
            'Model z-score': fmt_model_input_value(model_z_score),
            'Model input value': fmt_model_input_value(model_z_score),
            'Source': source_label,
            'Availability rule': availability_rule,
            'Raw value evidence': raw_value_evidence,
            'Evidence source': raw_value_evidence,
            'Model timestamp': fmt_timestamp_pair(model_input_ts),
            'Public latest value': fmt_raw_snapshot_value(public_value),
            'Match status': match_status,
            'Lag (hours)': 'N/A' if pd.isna(lag_hours) else f'{lag_hours:.1f}',
            'Notes': '; '.join(notes),
        })

    model_macro_input_audit_df = pd.DataFrame(audit_rows)
    if len(model_macro_input_audit_df) != len(MODEL_MACRO_INPUT_SPECS) or model_macro_input_audit_df['Model feature'].duplicated().any():
        print('Model macro input audit table validation warning: expected exactly one row per macro feature with no duplicates.')

    model_macro_input_audit_styler = (
        model_macro_input_audit_df.style
        .hide(axis='index')
        .set_properties(**{
            'text-align': 'left',
            'font-size': '12px',
            'padding': '8px 10px',
            'border': '1px solid #D9E2F3',
            'white-space': 'nowrap',
        })
        .set_table_styles([
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%'), ('margin', '0 0 1.1rem 0'), ('font-family', 'Times New Roman, Times, serif'), ('border', '1px solid #D9E2F3')]},
            {'selector': 'th', 'props': [('background-color', '#DCE6F1'), ('color', '#1F1F1F'), ('font-size', '12px'), ('font-weight', '700'), ('text-align', 'left'), ('padding', '9px 10px'), ('border', '1px solid #B4C7E7')]},
            {'selector': 'td', 'props': [('background-color', '#FFFFFF')]},
            {'selector': 'tbody tr:nth-child(even) td', 'props': [('background-color', '#F7F9FC')]},
            {'selector': 'caption', 'props': [('caption-side', 'top'), ('text-align', 'left'), ('font-size', '14px'), ('font-weight', '700'), ('padding', '0 0 0.4rem 0'), ('color', '#1F1F1F')]},
        ])
        .set_caption('Actual macro inputs used by model')
    )
    display(model_macro_input_audit_styler)

displayHTML('<div style="height: 0.35rem;"></div>')
displayHTML('<h3 style="margin-top: 0.25rem;">4c. ALFRED/FRED public vintage reference</h3>')
displayHTML('<p><i>This table is a diagnostic public-vintage reference only. It is not the direct model input row. The model consumes z-scored features from silver_rt_1h. Multiple realtime_start rows for the same observation_date are expected because ALFRED preserves vintages and revisions.</i></p>')
displayHTML('<p><i>GDP is quarterly, so its latest observation_date can lag monthly indicators. The dashboard does not infer unreleased quarters, and export_run_ts is the notebook extraction timestamp for this pull.</i></p>')

macro_export_run_ts = pd.Timestamp.utcnow()
macro_pull_frames = []
macro_source_status = []
macro_export_series = list(dict.fromkeys(REQUIRED_PUBLIC_MACRO_SERIES + OPTIONAL_PUBLIC_MACRO_SERIES))
macro_unresolved_series = []

for series_id in macro_export_series:
    feature_label = PUBLIC_MACRO_SERIES_FEATURE_MAP.get(series_id, series_id)
    public_rows = fetch_alfred_public_lookup(series_id, lookback_days=ALFRED_EXPORT_LOOKBACK_DAYS)
    if public_rows.empty:
        macro_unresolved_series.append({'series_id': series_id, 'feature': feature_label, 'public_source': 'UNRESOLVED', 'row_count': 0, 'min_observation_date': pd.NaT, 'max_observation_date': pd.NaT})
        continue

    public_rows = public_rows.copy()
    public_rows['public_value'] = pd.to_numeric(public_rows['public_value'], errors='coerce')
    public_rows = public_rows.dropna(subset=['observation_date', 'public_value'])
    if public_rows.empty:
        macro_unresolved_series.append({'series_id': series_id, 'feature': feature_label, 'public_source': 'UNRESOLVED', 'row_count': 0, 'min_observation_date': pd.NaT, 'max_observation_date': pd.NaT})
        continue

    public_rows['series_id'] = series_id
    public_rows['feature'] = feature_label
    public_rows['export_run_ts'] = macro_export_run_ts
    macro_pull_frames.append(public_rows[['series_id', 'feature', 'observation_date', 'realtime_start', 'realtime_end', 'public_value', 'public_source', 'export_run_ts']])

    source_counts = sorted(public_rows['public_source'].dropna().astype(str).unique().tolist())
    source_label = ','.join(source_counts) if source_counts else 'UNRESOLVED'
    obs_dates = pd.to_datetime(public_rows['observation_date'], errors='coerce')
    macro_source_status.append({
        'series_id': series_id,
        'feature': feature_label,
        'public_source': source_label,
        'row_count': int(public_rows.shape[0]),
        'min_observation_date': obs_dates.min(),
        'max_observation_date': obs_dates.max(),
    })

macro_export_df = pd.concat(macro_pull_frames, ignore_index=True) if macro_pull_frames else pd.DataFrame(columns=['series_id', 'feature', 'observation_date', 'realtime_start', 'realtime_end', 'public_value', 'public_source', 'export_run_ts'])

if macro_export_df.empty:
    print('No ALFRED/FRED rows were pulled for the configured series.')
else:
    export_columns = ['series_id', 'feature', 'observation_date', 'realtime_start', 'public_value', 'public_source', 'export_run_ts']
    macro_export_df['observation_date'] = _to_naive_utc(macro_export_df['observation_date'])
    macro_export_df['realtime_start'] = _to_naive_utc(macro_export_df['realtime_start'])
    macro_export_df['realtime_end'] = _to_naive_utc(macro_export_df['realtime_end'])

    displayHTML('<h4>Latest exported rows</h4>')
    latest_rows = macro_export_df.sort_values(['series_id', 'observation_date', 'realtime_start']).groupby('series_id').tail(3)
    latest_view = latest_rows.copy()
    latest_view['observation_date'] = pd.to_datetime(latest_view['observation_date'], errors='coerce').apply(fmt_date)
    latest_view['realtime_start'] = latest_view['realtime_start'].apply(fmt_timestamp)
    latest_view['export_run_ts'] = latest_view['export_run_ts'].apply(fmt_timestamp)
    latest_view['public_value'] = pd.to_numeric(latest_view['public_value'], errors='coerce')
    display(latest_view[export_columns].sort_values(['series_id', 'observation_date', 'realtime_start'], ascending=[True, False, False]))
    displayHTML('<p><i>GDP rows reflect the latest released quarterly observation available from ALFRED/FRED. The dashboard does not infer a value for an unreleased current quarter.</i></p>')

    try:
        macro_export_df.to_csv(ALFRED_PUBLIC_EXPORT_CSV_PATH, index=False)
        print(f'ALFRED/FRED macro public CSV written to: {ALFRED_PUBLIC_EXPORT_CSV_PATH}')
    except Exception as exc:
        print(f'ALFRED/FRED macro public CSV write failed: {exc}')



displayHTML('<h3 style="margin-top: 0.25rem;">4d. Upcoming economic events / model catalyst watch</h3>')
displayHTML('<p><i>This section is read-only. It uses a forward-looking economic calendar source for visibility and uses bronze_rt_macro_events_live only to annotate live capture / override status when a matching row exists. It does not alter silver_rt_1h, model inference, signals, or trading.</i></p>')
displayHTML('<p><i>External calendars may show Core PCE YoY/MoM or GDP QoQ percent-change values. The model features PCEPI_z, PCE_z, and GDPC1_z are based on raw level/index series. Calendar percent-change values must not be used as raw model overrides unless explicitly mapped.</i></p>')

MODEL_CATALYST_LOOKAHEAD_DAYS = 14
MODEL_CATALYST_MAX_ROWS = 12
FINNHUB_ECONOMIC_CALENDAR_URL = 'https://finnhub.io/api/v1/calendar/economic'
FINNHUB_HAYSTACK_CALENDAR_PATH = '/Volumes/pawan_quant_research_1/v4/market_data/raw_finnhub_haystack.csv'

MODEL_CATALYST_EVENT_SPECS = {
    'CPIAUCSL': {
        'label': 'CPI',
        'aliases': ['consumer price index', 'cpi', 'cpi mm', 'cpi mom', 'cpi yy', 'cpi yoy'],
        'source_status': 'live override enabled',
        'can_affect': 'True',
        'risk_note': 'Confirm live value is compatible with CPIAUCSL level semantics before interpreting public-reference comparisons.',
    },
    'PCEPI': {
        'label': 'Core PCE / PCE Price Index',
        'aliases': ['core pce price index', 'pce price index', 'personal consumption expenditures price index', 'core pce'],
        'source_status': 'Phase 1 live override enabled; unit/series validation required',
        'can_affect': 'Conditional',
        'risk_note': 'Calendar values may be YoY/MoM percent changes; model feature expects a raw PCEPI level/index series.',
    },
    'PCE': {
        'label': 'Personal Consumption Expenditures',
        'aliases': ['personal consumption expenditures', 'personal spending', 'consumer spending'],
        'source_status': 'Phase 1 live override enabled; unit/series validation required',
        'can_affect': 'Conditional',
        'risk_note': 'Calendar values may be percent changes; model feature expects a raw PCE level series.',
    },
    'GDPC1': {
        'label': 'GDP / Real GDP',
        'aliases': ['real gdp', 'gdp growth rate', 'gross domestic product', 'gdp qoq', 'gdp growth'],
        'source_status': 'Phase 1 live override enabled; unit/series validation required',
        'can_affect': 'Conditional',
        'risk_note': 'GDP calendars often publish QoQ percent-change values; model feature expects raw GDPC1 level/index semantics.',
    },
    'PAYEMS': {
        'label': 'Payrolls / NFP',
        'aliases': ['nonfarm payroll', 'non farm payroll', 'nfp', 'payrolls', 'payroll'],
        'source_status': 'carry-forward only / deferred; NFP_Event_Active design required before live override',
        'can_affect': 'False / deferred',
        'risk_note': 'Do not change PAYEMS or NFP_Event_Active semantics from this dashboard view.',
    },
    'NFP_Event_Active': {
        'label': 'NFP event pulse',
        'aliases': ['nfp event active'],
        'source_status': 'event flag / no raw value override from calendar',
        'can_affect': 'False / mapped event flag only',
        'risk_note': 'Calendar visibility only; event flag semantics are controlled by RT enrichment, not this report.',
    },

    'UNRATE': {
        'label': 'Unemployment Rate',
        'aliases': ['unemployment rate', 'jobless rate'],
        'source_status': 'carry-forward only / deferred; NFP_Event_Active design required before live override',
        'can_affect': 'False / deferred',
        'risk_note': 'Usually released with payrolls; no live model override is enabled in Phase 1.',
    },
    'INDPRO': {
        'label': 'Industrial Production',
        'aliases': ['industrial production'],
        'source_status': 'carry-forward only / future candidate',
        'can_affect': 'False / deferred',
        'risk_note': 'Future candidate only; this dashboard section is not a production ingestion change.',
    },
    'UMCSENT': {
        'label': 'University of Michigan Sentiment',
        'aliases': ['michigan consumer sentiment', 'u mich sentiment', 'consumer sentiment', 'u mich'],
        'source_status': 'live override enabled, final-only',
        'can_affect': 'True, final-only',
        'risk_note': 'Preliminary Michigan sentiment remains display-only; only final validated rows can override RT silver.',
    },
}

KNOWN_WEEKLY_CATALYST_EVENTS = [
    {'event_ts': '2026-05-03 18:30:00', 'event_name': 'Holiday - China Labor Day', 'currency_country': 'CN', 'importance': '-', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '04-May-26', 'day': 'Mon', 'time_ist': 'All Day', 'source_note': 'Holiday watchlist item; lower-priority market context.'},
    {'event_ts': '2026-05-03 18:30:00', 'event_name': 'Holiday - Greenery Day', 'currency_country': 'JP', 'importance': '-', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '04-May-26', 'day': 'Mon', 'time_ist': 'All Day', 'source_note': 'Holiday watchlist item; lower-priority market context.'},
    {'event_ts': '2026-05-03 18:30:00', 'event_name': 'Holiday - Early May Bank Holiday', 'currency_country': 'UK', 'importance': '-', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '04-May-26', 'day': 'Mon', 'time_ist': 'All Day', 'source_note': 'Holiday watchlist item; lower-priority market context.'},
    {'event_ts': '2026-05-04 18:30:00', 'event_name': 'Holiday - China Labor Day', 'currency_country': 'CN', 'importance': '-', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '05-May-26', 'day': 'Tue', 'time_ist': 'All Day', 'source_note': 'Holiday watchlist item; lower-priority market context.'},
    {'event_ts': '2026-05-04 18:30:00', 'event_name': "Holiday - Children's Day", 'currency_country': 'JP', 'importance': '-', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '05-May-26', 'day': 'Tue', 'time_ist': 'All Day', 'source_note': 'Holiday watchlist item; lower-priority market context.'},
    {'event_ts': '2026-05-04 18:30:00', 'event_name': "Holiday - Children's Day", 'currency_country': 'KR', 'importance': '-', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '05-May-26', 'day': 'Tue', 'time_ist': 'All Day', 'source_note': 'Holiday watchlist item; lower-priority market context.'},
    {'event_ts': '2026-05-05 04:30:00', 'event_name': 'RBA Interest Rate Decision (May)', 'currency_country': 'AU', 'importance': '3', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '05-May-26', 'day': 'Tue', 'time_ist': '10:00', 'source_note': 'Central-bank catalyst watch item; no direct model raw override.'},
    {'event_ts': '2026-05-05 13:45:00', 'event_name': 'S&P Global Services PMI (Apr)', 'currency_country': 'US', 'importance': '2', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '05-May-26', 'day': 'Tue', 'time_ist': '19:15', 'source_note': 'Services/PMI watchlist catalyst; no direct model raw override.'},
    {'event_ts': '2026-05-05 14:00:00', 'event_name': 'New Home Sales (Mar)', 'currency_country': 'US', 'importance': '3', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '05-May-26', 'day': 'Tue', 'time_ist': '19:30', 'source_note': 'Housing watchlist catalyst; no direct model raw override.'},
    {'event_ts': '2026-05-05 14:00:00', 'event_name': 'JOLTS Job Openings (Mar)', 'currency_country': 'US', 'importance': '3', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '05-May-26', 'day': 'Tue', 'time_ist': '19:30', 'mapped_model_features': 'PAYEMS, UNRATE, NFP_Event_Active', 'source_note': 'Labor catalyst watch item; no direct raw model override unless explicitly mapped.'},
    {'event_ts': '2026-05-05 14:00:00', 'event_name': 'ISM Non-Manufacturing Prices (Apr)', 'currency_country': 'US', 'importance': '3', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '05-May-26', 'day': 'Tue', 'time_ist': '19:30', 'source_note': 'ISM/services watchlist catalyst; no direct model raw override.'},
    {'event_ts': '2026-05-05 14:00:00', 'event_name': 'ISM Non-Manufacturing PMI (Apr)', 'currency_country': 'US', 'importance': '3', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '05-May-26', 'day': 'Tue', 'time_ist': '19:30', 'source_note': 'ISM/services watchlist catalyst; no direct model raw override.'},
    {'event_ts': '2026-05-05 18:30:00', 'event_name': 'Holiday - Constitution Day', 'currency_country': 'JP', 'importance': '-', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '06-May-26', 'day': 'Wed', 'time_ist': 'All Day', 'source_note': 'Holiday watchlist item; lower-priority market context.'},
    {'event_ts': '2026-05-06 12:15:00', 'event_name': 'ADP Nonfarm Employment Change (Apr)', 'currency_country': 'US', 'importance': '3', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '06-May-26', 'day': 'Wed', 'time_ist': '17:45', 'mapped_model_features': 'PAYEMS, NFP_Event_Active', 'source_note': 'Labor catalyst; PAYEMS_z / NFP_Event_Active watch where applicable.'},
    {'event_ts': '2026-05-07 12:30:00', 'event_name': 'Initial Jobless Claims', 'currency_country': 'US', 'importance': '3', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '07-May-26', 'day': 'Thu', 'time_ist': '18:00', 'source_note': 'Labor-market catalyst; no direct raw model override unless explicitly mapped.'},
    {'event_ts': '2026-05-08 12:30:00', 'event_name': 'Average Hourly Earnings (MoM) (Apr)', 'currency_country': 'US', 'importance': '3', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '08-May-26', 'day': 'Fri', 'time_ist': '18:00', 'mapped_model_features': 'PAYEMS, UNRATE, NFP_Event_Active', 'source_note': 'Labor catalyst released with payrolls; PAYEMS_z / UNRATE_z / NFP_Event_Active watch where applicable.'},
    {'event_ts': '2026-05-08 12:30:00', 'event_name': 'Nonfarm Payrolls (Apr)', 'currency_country': 'US', 'importance': '3', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '08-May-26', 'day': 'Fri', 'time_ist': '18:00', 'mapped_model_features': 'PAYEMS, NFP_Event_Active', 'source_note': 'Labor catalyst; PAYEMS_z / NFP_Event_Active watch where applicable.'},
    {'event_ts': '2026-05-08 12:30:00', 'event_name': 'Unemployment Rate (Apr)', 'currency_country': 'US', 'importance': '3', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '08-May-26', 'day': 'Fri', 'time_ist': '18:00', 'mapped_model_features': 'UNRATE, NFP_Event_Active', 'source_note': 'Labor catalyst; UNRATE_z / NFP_Event_Active watch where applicable.'},
    {'event_ts': '2026-05-08 18:30:00', 'event_name': 'Holiday - Victory Day', 'currency_country': 'RU', 'importance': '-', 'calendar_source': 'Hardcoded stakeholder catalyst watchlist', 'date': '09-May-26', 'day': 'Sat', 'time_ist': 'All Day', 'source_note': 'Holiday watchlist item; lower-priority market context.'},
]



def _optional_event_column(columns, column_name, sql_type='STRING'):
    return column_name if column_name in columns else f'CAST(NULL AS {sql_type}) AS {column_name}'

def _calendar_text(value):
    if _is_missing(value):
        return ''
    return str(value)


def _normalize_calendar_text(value):
    return re.sub(r'[^a-z0-9]+', ' ', _calendar_text(value).lower()).strip()


def _calendar_event_ts(value):
    parsed = pd.to_datetime(value, errors='coerce')
    if pd.isna(parsed):
        return pd.NaT
    if getattr(parsed, 'tzinfo', None) is None:
        return parsed.tz_localize('UTC')
    return parsed.tz_convert('UTC')


def _calendar_event_ts_naive(value):
    parsed = _calendar_event_ts(value)
    if pd.isna(parsed):
        return pd.NaT
    return parsed.tz_localize(None)


def _mapped_model_features(event_name):
    normalized = _normalize_calendar_text(event_name)
    matches = []
    for feature, spec in MODEL_CATALYST_EVENT_SPECS.items():
        if any(_normalize_calendar_text(alias) in normalized for alias in spec['aliases']):
            matches.append(feature)
    if 'PCEPI' in matches and 'PCE' in matches and 'price index' in normalized:
        matches = [feature for feature in matches if feature != 'PCE']
    if 'GDPC1' in matches and 'price index' in normalized:
        matches = [feature for feature in matches if feature != 'GDPC1']
    return matches


def _combined_event_status(mapped_features):
    if not mapped_features:
        return 'not a model feature'
    return '; '.join(dict.fromkeys(MODEL_CATALYST_EVENT_SPECS[feature]['source_status'] for feature in mapped_features))


def _combined_can_affect(mapped_features):
    if not mapped_features:
        return 'False'
    if any(MODEL_CATALYST_EVENT_SPECS[feature]['can_affect'] == 'Conditional' for feature in mapped_features):
        return 'Conditional'
    if any(MODEL_CATALYST_EVENT_SPECS[feature]['can_affect'].startswith('False') for feature in mapped_features):
        return 'False / deferred'
    return '; '.join(dict.fromkeys(MODEL_CATALYST_EVENT_SPECS[feature]['can_affect'] for feature in mapped_features))


def _combined_unit_stage_notes(mapped_features, source_note=''):
    notes = []
    if mapped_features:
        notes.extend(dict.fromkeys(MODEL_CATALYST_EVENT_SPECS[feature]['risk_note'] for feature in mapped_features))
    else:
        notes.append('Unmapped catalyst; display-only.')
    if source_note:
        notes.append(source_note)
    return ' | '.join(notes)


def _event_time_to_release(event_ts_utc):
    event_ts = to_display_ist_timestamp(event_ts_utc)
    if pd.isna(event_ts):
        return 'N/A'
    now_display = pd.Timestamp.now(tz=DISPLAY_TZ)
    delta = event_ts - now_display
    prefix = 'in ' if delta.total_seconds() >= 0 else ''
    suffix = '' if delta.total_seconds() >= 0 else ' ago'
    return f'{prefix}{format_duration(abs(delta))}{suffix}'


def _event_time_ist(event_ts_utc):
    event_ts = pd.to_datetime(event_ts_utc, errors='coerce')
    if pd.isna(event_ts):
        return 'N/A'
    if getattr(event_ts, 'tzinfo', None) is None:
        event_ts = event_ts.tz_localize('UTC')
    return fmt_timestamp_long(event_ts, DISPLAY_TZ, DISPLAY_TZ_LABEL)


def _event_time_edt(event_ts_utc):
    event_ts = pd.to_datetime(event_ts_utc, errors='coerce')
    if pd.isna(event_ts):
        return 'N/A'
    if getattr(event_ts, 'tzinfo', None) is None:
        event_ts = event_ts.tz_localize('UTC')
    event_ny = event_ts.tz_convert(NEW_YORK_TZ)
    return event_ny.strftime('%d-%b %H:%M %Z')


def _event_time_compact_ist(event_ts_utc):
    event_ts = pd.to_datetime(event_ts_utc, errors='coerce')
    if pd.isna(event_ts):
        return 'N/A'
    if getattr(event_ts, 'tzinfo', None) is None:
        event_ts = event_ts.tz_localize('UTC')
    return event_ts.tz_convert(DISPLAY_TZ).strftime('%H:%M')


def _compact_model_relevance(event_name, mapped_features):
    event_text = _calendar_text(event_name).lower()
    mapped = ', '.join(mapped_features or [])
    mapped_text = mapped.lower()
    if 'holiday' in event_text:
        return 'Holiday'
    if any(token in event_text for token in ['nonfarm', 'payroll', 'unemployment', 'hourly earnings', 'adp']) or any(token in mapped_text for token in ['payems', 'unrate', 'nfp']):
        return 'Labor catalyst'
    if 'jobless' in event_text:
        return 'Labor watch'
    if any(token in event_text for token in ['ism', 'services pmi', 'new home sales', 'home sales']):
        return 'Macro watch'
    if mapped_features:
        return 'Mapped catalyst'
    return 'Watchlist'


def _get_finnhub_calendar_key():
    env_key = os.environ.get('FINNHUB_API_KEY') or os.environ.get('FINNHUB_TOKEN')
    if env_key:
        return env_key
    if 'dbutils' in globals():
        for scope, key in [('alpaca', 'finnhub_api_key'), ('pawan_quant', 'finnhub_api_key')]:
            try:
                secret_value = dbutils.secrets.get(scope=scope, key=key)
                if secret_value:
                    return secret_value
            except Exception:
                pass
    return ''


def _shape_calendar_rows(frame, calendar_source):
    if frame is None or frame.empty:
        return pd.DataFrame()
    rows = []
    for _, row in frame.iterrows():
        event_ts_utc = _calendar_event_ts_naive(row.get('event_ts'))
        if pd.isna(event_ts_utc):
            continue
        event_name = _calendar_text(row.get('event_name'))
        explicit_mapped = _calendar_text(row.get('mapped_model_features'))
        mapped_features = [item.strip() for item in explicit_mapped.split(',') if item.strip()] if explicit_mapped else _mapped_model_features(event_name)
        source_note = _calendar_text(row.get('source_note'))
        rows.append({
            'event_ts_utc': event_ts_utc,
            'Date': _calendar_text(row.get('date')) or fmt_date(event_ts_utc),
            'Day': _calendar_text(row.get('day')),
            'Time IST': _calendar_text(row.get('time_ist')) or _event_time_compact_ist(event_ts_utc),
            'Time EDT': _event_time_edt(event_ts_utc),
            'Currency': _calendar_text(row.get('currency_country')).replace('USD / ', '') or 'US',
            'Event': event_name or 'N/A',
            'Event name': event_name or 'N/A',
            'Scheduled release time': fmt_timestamp_pair(event_ts_utc),
            'Release time (IST)': _event_time_ist(event_ts_utc),
            'Time to release': _event_time_to_release(event_ts_utc),
            'Currency / country': _calendar_text(row.get('currency_country')) or 'USD / US',
            'Importance': _calendar_text(row.get('importance')) or 'N/A',
            'Mapped model feature(s)': ', '.join(mapped_features) if mapped_features else 'Unmapped',
            'Model relevance': _compact_model_relevance(event_name, mapped_features),
            'Source status': _combined_event_status(mapped_features),
            'Can affect next RT silver row?': _combined_can_affect(mapped_features),
            'Catalyst note': _combined_unit_stage_notes(mapped_features, source_note=source_note),
            'Unit/stage caution': _combined_unit_stage_notes(mapped_features, source_note=source_note),
            'Model usage note': 'Stakeholder visibility only; no change to silver_rt_1h, inference, signals, or trading.',
            'Calendar source': _calendar_text(row.get('calendar_source')) or calendar_source,
        })
    return pd.DataFrame(rows)


def _load_finnhub_api_calendar(now_utc, end_utc):
    api_key = _get_finnhub_calendar_key()
    if not api_key:
        return pd.DataFrame()
    try:
        response = requests.get(
            FINNHUB_ECONOMIC_CALENDAR_URL,
            params={'from': pd.Timestamp(now_utc).date().isoformat(), 'to': pd.Timestamp(end_utc).date().isoformat(), 'token': api_key},
            timeout=FRED_API_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        payload = response.json() or {}
    except Exception as exc:
        print(f'Finnhub upcoming calendar fetch failed: {exc}')
        return pd.DataFrame()
    rows = []
    for record in payload.get('economicCalendar', []):
        country = _calendar_text(record.get('country'))
        if country and country.upper() != 'US':
            continue
        importance = _calendar_text(record.get('impact')) or _calendar_text(record.get('importance'))
        if importance and importance.lower() not in {'high', 'medium'}:
            continue
        event_ts_utc = _calendar_event_ts_naive(record.get('time'))
        if pd.isna(event_ts_utc) or event_ts_utc < now_utc or event_ts_utc > end_utc:
            continue
        rows.append({
            'event_ts': event_ts_utc,
            'event_name': record.get('event'),
            'currency_country': f"{_calendar_text(record.get('currency')) or 'USD'} / {country or 'US'}",
            'importance': importance or 'N/A',
            'calendar_source': 'Finnhub economic calendar API',
        })
    return _shape_calendar_rows(pd.DataFrame(rows), 'Finnhub economic calendar API')


def _load_finnhub_haystack_calendar(now_utc, end_utc):
    try:
        haystack_df = spark.read.option('header', True).csv(FINNHUB_HAYSTACK_CALENDAR_PATH)
    except Exception as exc:
        print(f'Finnhub haystack calendar read failed: {exc}')
        return pd.DataFrame()
    haystack_columns = set(haystack_df.columns)
    if not {'time', 'event'}.issubset(haystack_columns):
        return pd.DataFrame()
    temp_view_name = 'dashboard_finnhub_haystack_calendar'
    haystack_df.createOrReplaceTempView(temp_view_name)
    now_sql = sql_timestamp(now_utc)
    end_sql = sql_timestamp(end_utc)
    if now_sql is None or end_sql is None:
        return pd.DataFrame()
    country_expr = 'country' if 'country' in haystack_columns else "'US'"
    impact_expr = 'impact' if 'impact' in haystack_columns else "'N/A'"
    rows = query_to_pandas(f"""
        SELECT
            timestamp(time) AS event_ts,
            event AS event_name,
            CONCAT('USD / ', COALESCE({country_expr}, 'US')) AS currency_country,
            {impact_expr} AS importance,
            '{FINNHUB_HAYSTACK_CALENDAR_PATH}' AS calendar_source
        FROM {temp_view_name}
        WHERE timestamp(time) >= TIMESTAMP('{now_sql}')
          AND timestamp(time) <= TIMESTAMP('{end_sql}')
          AND ({country_expr} IS NULL OR UPPER({country_expr}) = 'US')
          AND (LOWER({impact_expr}) IN ('high', 'medium') OR {impact_expr} IS NULL)
        ORDER BY timestamp(time) ASC, event ASC
        LIMIT 50
    """, empty_columns=['event_ts', 'event_name', 'currency_country', 'importance', 'calendar_source'])
    return _shape_calendar_rows(rows, FINNHUB_HAYSTACK_CALENDAR_PATH)


def _load_known_weekly_calendar(now_utc, end_utc):
    frame = pd.DataFrame(KNOWN_WEEKLY_CATALYST_EVENTS)
    if frame.empty:
        return pd.DataFrame()
    frame['event_ts_filter'] = frame['event_ts'].apply(_calendar_event_ts_naive)
    frame = frame[(frame['event_ts_filter'] >= now_utc) & (frame['event_ts_filter'] <= end_utc)].copy()
    return _shape_calendar_rows(frame.drop(columns=['event_ts_filter']), 'Known weekly catalyst watchlist fallback')


def _load_live_macro_status_annotations(calendar_frame):
    if calendar_frame is None or calendar_frame.empty or not table_exists(LIVE_MACRO_TABLE):
        return calendar_frame
    live_columns = set(table_columns(LIVE_MACRO_TABLE))
    if not {'indicator_id', 'scheduled_release_ts'}.issubset(live_columns):
        calendar_frame['Live capture status'] = 'no live status row'
        return calendar_frame
    now_utc = pd.Timestamp.utcnow().tz_localize(None)
    end_utc = now_utc + pd.Timedelta(days=MODEL_CATALYST_LOOKAHEAD_DAYS)
    now_sql = sql_timestamp(now_utc - pd.Timedelta(hours=24))
    end_sql = sql_timestamp(end_utc)
    optional_columns = [
        _optional_event_column(live_columns, 'status'),
        _optional_event_column(live_columns, 'validation_status'),
        _optional_event_column(live_columns, 'release_stage'),
        _optional_event_column(live_columns, 'source_provider'),
        _optional_event_column(live_columns, 'observed_value', 'DOUBLE'),
    ]
    order_column = 'ingestion_ts DESC' if 'ingestion_ts' in live_columns else 'scheduled_release_ts DESC'
    live_rows = query_to_pandas(f"""
        SELECT indicator_id,
               scheduled_release_ts,
               status,
               validation_status,
               release_stage,
               source_provider,
               observed_value
        FROM (
            SELECT indicator_id,
                   scheduled_release_ts,
                   {', '.join(optional_columns)},
                   ROW_NUMBER() OVER (PARTITION BY indicator_id, scheduled_release_ts ORDER BY {order_column}) AS rn
            FROM {LIVE_MACRO_TABLE}
            WHERE scheduled_release_ts >= TIMESTAMP('{now_sql}')
              AND scheduled_release_ts <= TIMESTAMP('{end_sql}')
        ) t
        WHERE rn = 1
    """, empty_columns=['indicator_id', 'scheduled_release_ts', 'status', 'validation_status', 'release_stage', 'source_provider', 'observed_value'])
    if live_rows.empty:
        calendar_frame['Live capture status'] = 'no live status row'
        return calendar_frame
    live_rows = live_rows.copy()
    live_rows['scheduled_release_ts'] = pd.to_datetime(live_rows['scheduled_release_ts'], errors='coerce')

    def _match_live_status(row):
        mapped = [feature.strip() for feature in _as_text(row.get('Mapped model feature(s)'), '').split(',') if feature.strip() and feature.strip() != 'Unmapped']
        event_ts = pd.to_datetime(row.get('event_ts_utc'), errors='coerce')
        if not mapped or pd.isna(event_ts):
            return 'not applicable'
        candidates = live_rows[live_rows['indicator_id'].isin(mapped)].copy()
        if candidates.empty:
            return 'no live status row'
        candidates['time_diff_minutes'] = (candidates['scheduled_release_ts'] - event_ts).abs().dt.total_seconds() / 60.0
        candidates = candidates[candidates['time_diff_minutes'] <= 180]
        if candidates.empty:
            return 'no live status row'
        parts = []
        for _, live in candidates.sort_values(['time_diff_minutes', 'indicator_id']).iterrows():
            value = fmt_raw_snapshot_value(live.get('observed_value'))
            parts.append(
                f"{live.get('indicator_id')}: status={_as_text(live.get('status'))}, "
                f"validation={_as_text(live.get('validation_status'))}, "
                f"stage={_as_text(live.get('release_stage'))}, value={value}"
            )
        return ' | '.join(parts)

    calendar_frame['Live capture status'] = calendar_frame.apply(_match_live_status, axis=1)
    return calendar_frame


report_anchor_display = pd.Timestamp.now(tz=REPORT_TZ).normalize()
now_utc = report_anchor_display.tz_convert('UTC').tz_localize(None)
calendar_end_utc = now_utc + pd.Timedelta(days=MODEL_CATALYST_LOOKAHEAD_DAYS)
calendar_sources = [
    _load_known_weekly_calendar(now_utc, calendar_end_utc),
]
calendar_frames = [frame for frame in calendar_sources if frame is not None and not frame.empty]
upcoming_events_calendar = pd.concat(calendar_frames, ignore_index=True) if calendar_frames else pd.DataFrame()

if upcoming_events_calendar.empty:
    upcoming_events_display = pd.DataFrame([{
        'Event name': 'No upcoming high importance economic calendar rows available',
        'Scheduled release time': 'N/A',
        'Release time (IST)': 'N/A',
        'Mapped model feature(s)': 'N/A',
        'Source status': 'Dashboard calendar watch',
        'Can affect next RT silver row?': 'No model override is applied.',
        'Unit/stage caution': 'No upcoming event source row available; no model override is applied.',
        'Live capture status': 'No matching live capture row',
        'Calendar source': 'Dashboard calendar watch',
        'Indicator': 'N/A',
        'Importance': 'N/A',
        'Source': 'Dashboard calendar watch',
        'Model usage note': 'No model override is applied.',
    }])
    displayHTML('<p><i>No upcoming high importance economic calendar rows were available from dashboard-accessible calendar sources for the configured watch window.</i></p>')
    display(upcoming_events_display)
else:
    source_priority = {
        'Hardcoded stakeholder catalyst watchlist': 0,
        'Finnhub economic calendar API': 1,
        FINNHUB_HAYSTACK_CALENDAR_PATH: 2,
        'Known weekly catalyst watchlist fallback': 3,
    }
    upcoming_events_calendar['source_priority'] = upcoming_events_calendar['Calendar source'].map(source_priority).fillna(9)
    upcoming_events_calendar = upcoming_events_calendar.sort_values(['event_ts_utc', 'Event name', 'source_priority'])
    upcoming_events_calendar = upcoming_events_calendar.drop_duplicates(subset=['event_ts_utc', 'Event name'], keep='first')
    upcoming_events_calendar = _load_live_macro_status_annotations(upcoming_events_calendar)
    upcoming_events_detail_display = upcoming_events_calendar.head(MODEL_CATALYST_MAX_ROWS)[[
        'Date',
        'Day',
        'Time IST',
        'Time EDT',
        'Currency',
        'Importance',
        'Event',
        'Mapped model feature(s)',
        'Model relevance',
        'Catalyst note',
        'Model usage note',
        'Scheduled release time',
        'Release time (IST)',
        'Source status',
        'Can affect next RT silver row?',
        'Unit/stage caution',
        'Live capture status',
        'Calendar source',
    ]].copy()
    compact_columns = ['Date', 'Time IST', 'Time EDT', 'Currency', 'Importance', 'Event', 'Model relevance']
    upcoming_events_compact_display = upcoming_events_detail_display[compact_columns].copy()
    upcoming_events_display = upcoming_events_detail_display
    upcoming_events_styler = (
        upcoming_events_compact_display.style
        .set_table_styles([
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%'), ('font-size', '12px'), ('table-layout', 'fixed')]},
            {'selector': 'th', 'props': [('background-color', '#DCE6F1'), ('color', '#1F1F1F'), ('font-size', '12px'), ('font-weight', '700'), ('text-align', 'left'), ('padding', '7px 8px'), ('border', '1px solid #B4C7E7')]},
            {'selector': 'td', 'props': [('padding', '6px 8px'), ('border', '1px solid #D9E2F3'), ('vertical-align', 'top'), ('white-space', 'normal')]},
            {'selector': 'caption', 'props': [('caption-side', 'top'), ('text-align', 'left'), ('font-size', '14px'), ('font-weight', '700'), ('padding', '0 0 0.4rem 0'), ('color', '#1F1F1F')]},
        ])
        .set_caption('Upcoming economic events / model catalyst watch - compact stakeholder view')
    )
    display(upcoming_events_styler)
    if SHOW_CATALYST_DEBUG and REPORT_MODE != 'daily_pdf':
        displayHTML('<details><summary>Detailed catalyst diagnostics</summary>')
        display(upcoming_events_detail_display)
        displayHTML('</details>')

displayHTML('<p><i>This watchlist is for stakeholder visibility only and does not alter silver_rt_1h, model inference, signals, or trading.</i></p>')


displayHTML('<h2>5. Historical feature monitoring</h2>')
displayHTML(f'<p><i>Historical feature monitoring is shown in two scopes: {feature_since_start_note} and {feature_rolling_note}. Panels are grouped for institutional review across inflation, growth, employment, sentiment, volatility, price action, and event flags.</i></p>')

if feature_history.empty or current_features.empty:
    print('No feature monitoring data available.')
else:
    render_feature_monitoring_scope(feature_since_start_window, f'Historical Feature Monitoring — Since {MONITORING_START_DATE}')
    render_feature_monitoring_scope(feature_rolling_window, 'Historical Feature Monitoring — Last 48 Hours')


displayHTML('<h2>6. Attribution</h2>')
displayHTML('<p><i>Top weighted movers vs previous bar. Descriptive proxy only; not causal and not a SHAP explanation.</i></p>')
if feature_history.empty or current_features.empty:
    print('No attribution history available.')
else:
    probability_values_available = any(pd.notna(_to_float(latest_probability.get(column))) for column in PROBABILITY_LABELS.keys())
    if not probability_values_available:
        print('No probability history available for this window.')
    else:
        probability_context_df = pd.DataFrame(
            [
                {
                    'Top probability': f'{prob_label}: {fmt_pct(prob_value)}' if pd.notna(_to_float(prob_value)) else 'N/A',
                    'Expansion': fmt_pct(latest_probability.get('prob_expansion')),
                    'Slowdown': fmt_pct(latest_probability.get('prob_slowdown')),
                    'Recession': fmt_pct(latest_probability.get('prob_recession')),
                }
            ]
        )
        display(probability_context_df)


displayHTML('<h2>7. Historical signals and trades</h2>')
sig_cols = [
    'timestamp', 'regime_name', 'regime_int', 'position'
]
if not signal_history.empty:
    available_sig = [c for c in sig_cols if c in signal_history.columns]
    print('Latest 24 hourly signals:')
    signal_tail = signal_history[available_sig].sort_values('timestamp', ascending=False).head(24).copy()
    if 'timestamp' in signal_tail.columns:
        signal_tail['timestamp'] = signal_tail['timestamp'].apply(fmt_timestamp)
    if 'regime_name' in signal_tail.columns:
        signal_tail['regime_name'] = signal_tail['regime_name'].apply(_as_text)
    if 'position' in signal_tail.columns:
        signal_tail['position'] = signal_tail['position'].apply(_as_text)
    signal_tail = signal_tail.rename(columns={'timestamp': 'Timestamp', 'regime_name': 'Regime', 'regime_int': 'Regime code', 'position': 'Target position'})
    display(signal_tail)

if trade_history.empty:
    print('No trade history available in window.')
else:
    trade_cols = ['trade_ts', 'signal_ts', 'regime_name', 'target_position', 'switched', 'target_ticker', 'account_balance', 'account_equity', 'hourly_pnl', 'hourly_pnl_pct', 'day_pnl', 'day_pnl_pct', 'execution_mode']
    available_cols = [c for c in trade_cols if c in trade_history.columns]
    trade_tail = trade_history[available_cols].sort_values('trade_ts', ascending=False).head(100).copy()
    for ts_col in ['trade_ts', 'signal_ts']:
        if ts_col in trade_tail.columns:
            trade_tail[ts_col] = trade_tail[ts_col].apply(fmt_timestamp)
    for text_col in ['regime_name', 'target_position', 'target_ticker', 'execution_mode']:
        if text_col in trade_tail.columns:
            trade_tail[text_col] = trade_tail[text_col].apply(_as_text)
    if 'switched' in trade_tail.columns:
        trade_tail['switched'] = trade_tail['switched'].apply(fmt_flag)
    for money_col in ['account_balance', 'account_equity']:
        if money_col in trade_tail.columns:
            trade_tail[money_col] = trade_tail[money_col].apply(fmt_money)
    for pnl_col in ['hourly_pnl', 'day_pnl']:
        if pnl_col in trade_tail.columns:
            trade_tail[pnl_col] = trade_tail[pnl_col].apply(fmt_signed_money)
    for pct_col in ['hourly_pnl_pct', 'day_pnl_pct']:
        if pct_col in trade_tail.columns:
            trade_tail[pct_col] = trade_tail[pct_col].apply(fmt_optional_percent)
    trade_tail = trade_tail.rename(columns={'trade_ts': 'Trade timestamp', 'signal_ts': 'Signal timestamp', 'regime_name': 'Regime', 'target_position': 'Target position', 'switched': 'Switched', 'target_ticker': 'Target ticker', 'account_balance': 'Account balance', 'account_equity': 'Account equity', 'hourly_pnl': 'Hourly pnl', 'hourly_pnl_pct': 'Hourly pnl %', 'day_pnl': 'Day pnl', 'day_pnl_pct': 'Day pnl %', 'execution_mode': 'Execution mode'})
    display(trade_tail)


displayHTML('<h2>8. Suggested future additions</h2>')
displayHTML('''<ul>
<li>Trade attribution panel linking realized and unrealized PnL to regime calls and execution decisions.</li>
<li>Regime transition log with timestamped triggers, prior regime, new regime, and action summary.</li>
<li>Feature contribution over time panel for model-driven diagnostics beyond simple weighted-move proxies.</li>
<li>Risk metrics panel covering exposure, volatility, drawdown context, and benchmark-relative risk.</li>
<li>Stale data warnings and source health panel for market, macro, and audit feeds.</li>
</ul>''')





def _safe_report_text(value):
    return fmt_report_text(value)


def _wrap_report_cell_text(value, max_chars=28, preserve_newlines=False):
    text = _as_text(value)
    if not text:
        return 'N/A'
    text = text.replace('\r', '')
    parts = text.split('\n') if preserve_newlines else [text.replace('\n', ' ').strip()]
    wrapped_parts = []
    for part in parts:
        wrapped_parts.extend(textwrap.wrap(part.strip() or ' ', width=max_chars, break_long_words=True, break_on_hyphens=True) or [''])
    return '\n'.join(wrapped_parts).strip() or 'N/A'


def _report_variant_label():
    return 'Stakeholder circulation' if REPORT_VARIANT == 'circulation' else 'Overall institutional'


def _report_pdf_basename(report_date):
    suffix = 'circulation' if REPORT_VARIANT == 'circulation' else 'dashboard'
    return f'saxo_institutional_{suffix}_{report_date}_SGT.pdf'


def _format_metric_value(value, column_name=''):
    lowered = str(column_name).lower()
    if _is_missing(value):
        return 'N/A'
    if isinstance(value, str):
        return value
    if 'timestamp' in lowered or 'release time' in lowered or lowered in {'new york time', DISPLAY_TZ_LABEL.lower()}:
        return fmt_timestamp_pair(value)
    return fmt_report_number(value, column_name)


def _draw_text_page(pdf, title, lines, footer=None):
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis('off')
    ax.text(0.04, 0.96, title, fontsize=18, fontweight='bold', va='top')
    y = 0.9
    for line in (lines or []):
        wrapped_lines = []
        for part in str(line).split('\n'):
            wrapped_lines.extend(textwrap.wrap(part.strip() or ' ', width=95, break_long_words=False, break_on_hyphens=False) or [''])
        for wrapped_line in wrapped_lines:
            ax.text(0.04, y, wrapped_line, fontsize=10.5, va='top', wrap=True)
            y -= 0.032
            if y < 0.06:
                if footer:
                    ax.text(0.04, 0.025, footer, fontsize=7.5, va='bottom', color='#6f6f6f')
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
                fig, ax = plt.subplots(figsize=(11, 8.5))
                ax.axis('off')
                ax.text(0.04, 0.96, title, fontsize=18, fontweight='bold', va='top')
                y = 0.90
    if footer:
        ax.text(0.04, 0.025, footer, fontsize=7.5, va='bottom', color='#6f6f6f')
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)



def _draw_cover_page(pdf, title, subtitle, summary_lines, kpi_pairs, footer=None, kpi_cols=3):
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis('off')
    ax.add_patch(plt.Rectangle((0, 0.905), 1, 0.095, transform=ax.transAxes, facecolor='#17324D', edgecolor='none'))
    title_len = len(_as_text(title))
    title_font_size = 15 if title_len > 58 else 17 if title_len > 42 else 24
    ax.text(0.045, 0.952, title, fontsize=title_font_size, fontweight='bold', va='center', color='white')
    ax.text(0.045, 0.875, subtitle, fontsize=11.4, color='#404040', va='top')

    label_x = 0.045
    value_x = 0.325
    y = 0.825
    for line in summary_lines:
        line = _as_text(line)
        if ':' in line:
            label_text, value_text = line.split(':', 1)
            value_text = _safe_report_text(value_text.strip())
            value_parts = []
            for part in value_text.replace('\r', '').split('\n'):
                value_parts.extend(textwrap.wrap(part.strip(), width=76, break_long_words=False, break_on_hyphens=False) or [''])
            value_parts = value_parts or ['N/A']
            ax.text(label_x, y, f'{label_text.strip()}:', fontsize=9.8, fontweight='bold', va='top')
            for line_idx, value_line in enumerate(value_parts[:3]):
                ax.text(value_x, y - line_idx * 0.021, value_line, fontsize=9.3, va='top')
            y -= 0.028 + 0.021 * (min(len(value_parts), 3) - 1)
        else:
            ax.text(label_x, y, line, fontsize=10.2, va='top')
            y -= 0.032

    executive_y = min(0.545, y - 0.020)
    executive_y = max(executive_y, 0.455)
    ax.text(0.045, executive_y, 'Executive KPIs', fontsize=14.8, fontweight='bold', va='top')
    grid_top = executive_y - 0.038
    kpi_col_count = max(1, min(3, int(kpi_cols)))
    box_w = 0.445 if kpi_col_count == 2 else 0.292
    box_h = 0.070 if kpi_col_count == 2 else 0.054
    gap_x = 0.022
    gap_y = 0.008
    max_kpis = 10 if kpi_col_count == 2 else 12
    visible_kpis = list(kpi_pairs[:max_kpis])
    for i, (metric, value) in enumerate(visible_kpis):
        row = i // kpi_col_count
        col = i % kpi_col_count
        x = 0.045 + col * (box_w + gap_x)
        y0 = grid_top - row * (box_h + gap_y)
        rect = plt.Rectangle((x, y0 - box_h), box_w, box_h, facecolor='#F7F9FC', edgecolor='#C7D0DA', linewidth=0.8)
        ax.add_patch(rect)
        metric_text = _safe_report_text(metric)
        ax.text(x + 0.012, y0 - 0.012, metric_text, fontsize=7.8, color='#555555', va='top', fontweight='bold')
        value_text = fmt_report_text(value, max_chars=110)
        wrap_width = 42 if kpi_col_count == 2 else 27
        value_lines = textwrap.wrap(value_text, width=wrap_width, break_long_words=False, break_on_hyphens=False) or ['N/A']
        value_font = 8.0 if len(value_lines) > 1 else 8.8
        value_y = y0 - 0.036
        line_step = 0.019
        for line_idx, value_line in enumerate(value_lines[:2]):
            ax.text(x + 0.012, value_y - line_idx * line_step, value_line, fontsize=value_font, color='#1F1F1F', va='top')
    if footer:
        ax.text(0.045, 0.025, footer, fontsize=7.5, va='bottom', color='#6f6f6f')
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)


def _is_no_event_placeholder(frame):
    if frame is None or getattr(frame, 'empty', True) or 'Event name' not in frame.columns:
        return False
    event_name = _as_text(frame.iloc[0].get('Event name'), '').lower()
    return event_name.startswith('no upcoming high importance economic calendar rows')


def _draw_message_panel_page(pdf, title, message, footer=None, subtitle=None):
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis('off')
    ax.text(0.045, 0.94, title, fontsize=16, fontweight='bold', va='top')
    if subtitle:
        ax.text(0.045, 0.885, _wrap_report_cell_text(subtitle, max_chars=120), fontsize=10.2, color='#5A5A5A', va='top', linespacing=1.22)
    panel_top = 0.735
    panel_h = 0.155
    ax.add_patch(plt.Rectangle((0.045, panel_top - panel_h), 0.91, panel_h, facecolor='#F7F9FC', edgecolor='#C7D0DA', linewidth=0.9))
    ax.text(0.065, panel_top - 0.035, _wrap_report_cell_text(message, max_chars=115), fontsize=12.0, color='#1F1F1F', va='top', linespacing=1.28)
    if footer:
        ax.text(0.045, 0.025, footer, fontsize=7.5, va='bottom', color='#6f6f6f')
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)


def _source_table_alias(table_name):
    short_name = _as_text(table_name).split('.')[-1]
    aliases = {
        'gold_rt_saxo_dashboard_feed': 'Dashboard feed',
        'gold_rt_saxo_dashboard_health_rollup': 'Health rollup',
        'gold_rt_saxo_dashboard_performance_rollup': 'Performance rollup',
        'gold_saxo_v4_strategy_equity_curve': 'V4 strategy equity curve',
        'gold_saxo_v4_strategy_trade_ledger': 'V4 strategy trade ledger',
        'bronze_rt_regime_signal': 'RT signal',
        'silver_rt_1h': 'RT silver',
        'bronze_rt_saxo_trade_log': 'Saxo trade log',
        'bronze_rt_macro_events_live': 'Live macro events',
        'model_registry_current': 'Model registry',
    }
    return aliases.get(short_name, short_name)


def _draw_table_page(pdf, title, frame, max_rows=24, max_cols=8, footer=None, subtitle=None, center_cols=None, bold_cols=None, long_ts_cols=None, force_landscape=False, col_width_map=None, cell_wrap_map=None, font_size_override=None, header_wrap_chars=16):
    center_cols = set(center_cols or [])
    bold_cols = set(bold_cols or [])
    long_ts_cols = set(long_ts_cols or [])
    col_width_map = col_width_map or {}
    cell_wrap_map = cell_wrap_map or {}
    if frame is None or getattr(frame, 'empty', True):
        fig, ax = plt.subplots(figsize=(12.8, 8.75) if force_landscape else (11, 8.5))
        ax.axis('off')
        ax.text(0.04, 0.96, title, fontsize=16, fontweight='bold', va='top')
        if subtitle:
            ax.text(0.04, 0.895, _wrap_report_cell_text(subtitle, max_chars=135), fontsize=10.5, color='#5A5A5A', va='top', linespacing=1.22)
        ax.text(0.04, 0.84, 'No rows available.', fontsize=11, va='top')
        if footer:
            ax.text(0.04, 0.025, footer, fontsize=7.5, va='bottom', color='#6f6f6f')
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        return

    view = frame.copy().head(max_rows).iloc[:, :max_cols]
    for col in view.columns:
        if col in long_ts_cols:
            view[col] = view[col].map(lambda v: _wrap_report_cell_text(_format_paired_timestamp_cell(v), max_chars=32, preserve_newlines=True))
        elif pd.api.types.is_datetime64_any_dtype(view[col]):
            view[col] = view[col].apply(fmt_timestamp)
        elif pd.api.types.is_numeric_dtype(view[col]):
            view[col] = view[col].apply(lambda v, c=col: _wrap_report_cell_text(_format_metric_value(v, c), max_chars=22))
        else:
            view[col] = view[col].map(lambda v, c=col: _wrap_report_cell_text(_format_metric_value(v, c), max_chars=cell_wrap_map.get(c, 26)))

    col_count = max(len(view.columns), 1)
    landscape = force_landscape or col_count >= 5
    fig, ax = plt.subplots(figsize=(12.8, 8.75) if landscape else (11, 8.5))
    ax.axis('off')
    title_x = 0.03 if landscape else 0.04
    ax.text(title_x, 0.965, title, fontsize=15.5, fontweight='bold', va='top')
    if subtitle:
        ax.text(title_x, 0.900, _wrap_report_cell_text(subtitle, max_chars=145 if landscape else 115), fontsize=10.2, color='#5A5A5A', va='top', linespacing=1.22)

    font_size = 7.5 if landscape else 8.2
    if col_count >= 9:
        font_size = 6.6
    if len(view) > 18:
        font_size = min(font_size, 6.9)
    if font_size_override is not None:
        font_size = font_size_override
    table_left = 0.018 if landscape else 0.04
    table_width = 0.965 if landscape else 0.92
    table_height = 0.755 if subtitle else 0.825
    max_cell_lines = max((str(v).count('\n') + 1 for v in view.to_numpy().flatten()), default=1)
    row_height_scale = 1.55 if len(view) <= 16 else 1.28
    if long_ts_cols:
        row_height_scale += 0.36
    row_height_scale = max(row_height_scale, min(2.55, 1.08 + 0.28 * max_cell_lines))

    width_weights = []
    for col in view.columns:
        col_text = str(col).lower()
        if col in col_width_map:
            width_weights.append(float(col_width_map[col]))
        elif any(token in col_text for token in ('timestamp', 'release', 'time', 'scheduled')):
            width_weights.append(2.05)
        elif 'mapped model feature' in col_text:
            width_weights.append(1.9)
        elif any(token in col_text for token in ('feature', 'raw value evidence', 'evidence', 'notes', 'observation')):
            width_weights.append(1.8)
        elif any(token in col_text for token in ('importance', 'z-score', 'score', 'probability', 'pnl', 'return', 'equity', 'balance')):
            width_weights.append(1.1)
        elif 'regime' in col_text:
            width_weights.append(0.9)
        else:
            width_weights.append(1.0)
    total_weight = max(sum(width_weights), 1)
    col_widths = [(w / total_weight) * table_width for w in width_weights]

    table = ax.table(
        cellText=view.values,
        colLabels=[_wrap_report_cell_text(c, max_chars=header_wrap_chars) for c in view.columns],
        loc='upper left',
        bbox=[table_left, 0.065, table_width, table_height],
        cellLoc='center' if center_cols else 'left',
        colWidths=col_widths,
    )
    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.scale(1, row_height_scale)

    for (row, col), cell in table.get_celld().items():
        cell.PAD = 0.085
        cell.set_edgecolor('#C7D0DA')
        text = cell.get_text()
        text.set_va('center')
        text.set_linespacing(1.30 if (col < len(view.columns) and view.columns[col] in long_ts_cols) else 1.20)
        if row == 0:
            cell.set_facecolor('#DCE6F1')
            cell.set_text_props(weight='bold', color='#1F1F1F', ha='center', va='center')
            continue
        column_name = view.columns[col] if col < len(view.columns) else None
        if column_name in bold_cols:
            cell.set_text_props(weight='bold')
        else:
            cell.set_text_props(weight='normal')
        if column_name in center_cols:
            cell.set_text_props(ha='center')
        if (row + 1) % 2 == 0:
            cell.set_facecolor('#F7F9FC')

    if footer:
        ax.text(title_x, 0.025, footer, fontsize=7.5, va='bottom', color='#6f6f6f')
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

def _probability_plot_columns(frame):
    if frame is None or getattr(frame, 'empty', True):
        return []
    return [col for col in PROBABILITY_LABELS.keys() if col in frame.columns and pd.to_numeric(frame[col], errors='coerce').notna().any()]


def _draw_probability_chart_page(pdf, title, probability_frame, footer=None, subtitle=None, dual_time_axis=False, latest_snapshot_ok=False):
    probability_cols = _probability_plot_columns(probability_frame)
    if not probability_cols:
        _draw_message_panel_page(
            pdf,
            title,
            'No probability history available for this window.',
            footer=footer,
            subtitle=subtitle,
        )
        return

    plot_frame = probability_frame.dropna(subset=probability_cols, how='all').sort_values('timestamp').copy()
    if plot_frame.empty:
        _draw_message_panel_page(
            pdf,
            title,
            'No probability history available for this window.',
            footer=footer,
            subtitle=subtitle,
        )
        return

    fig, ax = plt.subplots(figsize=(9.6, 5.7))
    fig.subplots_adjust(bottom=0.24, top=0.86)
    if len(plot_frame) >= 2:
        for col in probability_cols:
            ax.plot(plot_frame['timestamp'], pd.to_numeric(plot_frame[col], errors='coerce') * 100, label=PROBABILITY_LABELS[col], linewidth=1.8)
        ax.set_title(title, loc='left', fontweight='bold')
        ax.set_ylabel('Probability (%)')
        if dual_time_axis:
            apply_dual_time_axes(ax, bottom_label=f'Timestamp ({DISPLAY_TZ_LABEL})')
        else:
            apply_ist_axis(ax)
        ax.grid(alpha=0.25)
        ax.legend(loc='best')
    elif latest_snapshot_ok:
        latest_prob = plot_frame.iloc[-1]
        labels = [PROBABILITY_LABELS[col] for col in probability_cols]
        values = [float(pd.to_numeric(pd.Series([latest_prob.get(col)]), errors='coerce').iloc[0]) * 100 for col in probability_cols]
        bars = ax.bar(labels, values, color=['#2E8B57', '#D4A017', '#C0392B'][:len(labels)], alpha=0.88)
        ax.set_title(title, loc='left', fontweight='bold')
        ax.set_ylabel('Probability (%)')
        ax.set_ylim(0, 100)
        ax.grid(axis='y', alpha=0.22)
        for bar, value in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, min(value + 2, 96), f'{value:.1f}%', ha='center', va='bottom', fontsize=9.2)
        ax.text(0.01, -0.13, f"Probability timestamp: {fmt_timestamp_pair(latest_prob.get('timestamp'))}", fontsize=8.2, transform=ax.transAxes, color='#4A4A4A', va='top')
    else:
        plt.close(fig)
        _draw_message_panel_page(
            pdf,
            title,
            'Probability history has only one timestamp in this window. See the latest snapshot page for current model probabilities.',
            footer=footer,
            subtitle=subtitle,
        )
        return
    if footer:
        fig.text(0.12, 0.035, footer, fontsize=7.5, color='#6f6f6f', va='bottom')
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)


def _v4_strategy_report_frames():
    trades = v4_strategy_trade_history.copy() if 'v4_strategy_trade_history' in globals() else pd.DataFrame(columns=['broker_event_ts'])
    equity = v4_strategy_equity_history.copy() if 'v4_strategy_equity_history' in globals() else pd.DataFrame(columns=['timestamp'])

    if not trades.empty:
        trades = trades[trades.get('symbol', pd.Series(dtype='object')).astype(str).str.upper().isin(['SPY', 'QQQ'])].copy()
        if 'broker_event_ts' in trades.columns:
            trades = trades.dropna(subset=['broker_event_ts']).sort_values('broker_event_ts').reset_index(drop=True)
        for c in ['quantity', 'price', 'gross_notional', 'total_transaction_cost', 'modeled_5bps_cost']:
            if c in trades.columns:
                trades[c] = pd.to_numeric(trades[c], errors='coerce')
        if 'modeled_5bps_cost' not in trades.columns and 'gross_notional' in trades.columns:
            trades['modeled_5bps_cost'] = pd.to_numeric(trades['gross_notional'], errors='coerce') * 0.0005
        elif 'gross_notional' in trades.columns:
            trades['modeled_5bps_cost'] = pd.to_numeric(trades['gross_notional'], errors='coerce') * 0.0005

    trade_table = trades.rename(columns={'total_transaction_cost': 'captured_conversion_cost'}).copy()
    trade_columns = [
        'broker_event_ts',
        'strategy_stance_before',
        'strategy_stance_after',
        'symbol',
        'side',
        'quantity',
        'price',
        'gross_notional',
        'captured_conversion_cost',
        'modeled_5bps_cost',
        'cost_observation_quality',
        'all_in_cost_confirmed',
        'trade_id',
        'position_id',
    ]
    trade_table = _select_existing_columns(trade_table, trade_columns)

    perf = equity.copy()
    if perf.empty or 'timestamp' not in perf.columns or 'strategy_equity_gross' not in perf.columns:
        return perf, trade_table, [], {
            'v4_strategy_chart_start_date': V4_STRATEGY_PERFORMANCE_START_DATE,
            'v4_strategy_trade_rows': int(len(trade_table)),
            'v4_strategy_symbols': sorted([str(v) for v in trade_table.get('symbol', pd.Series(dtype='object')).dropna().unique()]),
            'v4_non_strategy_activity_excluded': True,
            'v4_modeled_5bps_formula_ok': True,
        }

    perf = perf.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    for c in ['strategy_equity_gross', 'strategy_equity_net', 'cumulative_transaction_cost']:
        if c in perf.columns:
            perf[c] = pd.to_numeric(perf[c], errors='coerce')
    perf = perf.dropna(subset=['strategy_equity_gross']).reset_index(drop=True)
    if perf.empty:
        return perf, trade_table, [], {
            'v4_strategy_chart_start_date': V4_STRATEGY_PERFORMANCE_START_DATE,
            'v4_strategy_trade_rows': int(len(trade_table)),
            'v4_strategy_symbols': sorted([str(v) for v in trade_table.get('symbol', pd.Series(dtype='object')).dropna().unique()]),
            'v4_non_strategy_activity_excluded': True,
            'v4_modeled_5bps_formula_ok': True,
        }

    start_equity = float(perf['strategy_equity_gross'].iloc[0])
    cumulative_captured = pd.to_numeric(perf.get('cumulative_transaction_cost', 0.0), errors='coerce').ffill().fillna(0.0)
    captured_start = float(cumulative_captured.iloc[0]) if len(cumulative_captured) else 0.0
    perf['captured_conversion_cost_since_start'] = (cumulative_captured - captured_start).clip(lower=0.0)

    start_display_ts = pd.Timestamp(V4_STRATEGY_PERFORMANCE_START_DATE, tz=DISPLAY_TZ)
    window_trades = trades.copy()
    if not window_trades.empty and 'broker_event_ts' in window_trades.columns:
        window_trades = window_trades[window_trades['broker_event_ts'] >= start_display_ts].copy()

    if not window_trades.empty and {'broker_event_ts', 'modeled_5bps_cost'}.issubset(window_trades.columns):
        cost_curve = window_trades[['broker_event_ts', 'modeled_5bps_cost']].dropna(subset=['broker_event_ts']).copy()
        cost_curve['modeled_5bps_cost'] = pd.to_numeric(cost_curve['modeled_5bps_cost'], errors='coerce').fillna(0.0)
        cost_curve = cost_curve.sort_values('broker_event_ts').reset_index(drop=True)
        cost_curve['cumulative_modeled_5bps_cost'] = cost_curve['modeled_5bps_cost'].cumsum()
        if not cost_curve.empty:
            perf = pd.merge_asof(
                perf.sort_values('timestamp'),
                cost_curve[['broker_event_ts', 'cumulative_modeled_5bps_cost']],
                left_on='timestamp',
                right_on='broker_event_ts',
                direction='backward',
            ).drop(columns=['broker_event_ts'], errors='ignore')
    if 'cumulative_modeled_5bps_cost' not in perf.columns:
        perf['cumulative_modeled_5bps_cost'] = 0.0
    perf['cumulative_modeled_5bps_cost'] = pd.to_numeric(perf['cumulative_modeled_5bps_cost'], errors='coerce').fillna(0.0)

    perf['captured_cost_net_equity'] = perf['strategy_equity_gross'] - perf['captured_conversion_cost_since_start']
    perf['modeled_5bps_net_equity'] = perf['strategy_equity_gross'] - perf['cumulative_modeled_5bps_cost']
    if start_equity and not np.isclose(start_equity, 0.0):
        perf['gross_v4_return_pct'] = (perf['strategy_equity_gross'] / start_equity - 1.0) * 100.0
        perf['captured_cost_net_return_pct'] = (perf['captured_cost_net_equity'] / start_equity - 1.0) * 100.0
        perf['modeled_5bps_net_return_pct'] = (perf['modeled_5bps_net_equity'] / start_equity - 1.0) * 100.0
    else:
        perf['gross_v4_return_pct'] = np.nan
        perf['captured_cost_net_return_pct'] = np.nan
        perf['modeled_5bps_net_return_pct'] = np.nan

    ending = perf.iloc[-1]
    modeled_net_returns = pd.to_numeric(perf['modeled_5bps_net_equity'], errors='coerce').pct_change().replace([np.inf, -np.inf], np.nan).dropna()
    sharpe = np.nan
    if len(modeled_net_returns) >= 2 and modeled_net_returns.std(ddof=1) > 0:
        sharpe = float(modeled_net_returns.mean() / modeled_net_returns.std(ddof=1) * np.sqrt(252 * 6.5))

    captured_total = float(ending.get('captured_conversion_cost_since_start', np.nan))
    modeled_total = float(ending.get('cumulative_modeled_5bps_cost', np.nan))
    summary_pairs = [
        ('Start equity', fmt_money(start_equity)),
        ('Ending gross equity', fmt_money(ending.get('strategy_equity_gross'))),
        ('Ending captured-cost net equity', fmt_money(ending.get('captured_cost_net_equity'))),
        ('Ending 5bps-modeled net equity', fmt_money(ending.get('modeled_5bps_net_equity'))),
        ('Gross return %', fmt_pct((ending.get('strategy_equity_gross') / start_equity - 1.0) if start_equity else np.nan)),
        ('Captured-cost net return %', fmt_pct((ending.get('captured_cost_net_equity') / start_equity - 1.0) if start_equity else np.nan)),
        ('5bps-modeled net return %', fmt_pct((ending.get('modeled_5bps_net_equity') / start_equity - 1.0) if start_equity else np.nan)),
        ('Captured broker conversion cost $', fmt_money(captured_total)),
        ('Modeled 5bps cost $', fmt_money(modeled_total)),
        ('Number of V4 ETF fills', f'{len(trade_table):,}'),
        ('Annualized hourly Sharpe estimate for 5bps-modeled net return', 'N/A' if pd.isna(sharpe) else f'{sharpe:+.2f} short-sample estimate'),
    ]

    formula_ok = True
    if not trade_table.empty and {'gross_notional', 'modeled_5bps_cost'}.issubset(trade_table.columns):
        expected = pd.to_numeric(trade_table['gross_notional'], errors='coerce') * 0.0005
        observed = pd.to_numeric(trade_table['modeled_5bps_cost'], errors='coerce')
        formula_ok = bool(np.allclose(expected.fillna(0.0), observed.fillna(0.0), rtol=1e-9, atol=1e-9))

    validation = {
        'v4_strategy_chart_start_date': V4_STRATEGY_PERFORMANCE_START_DATE,
        'v4_strategy_chart_min_timestamp': fmt_timestamp(perf['timestamp'].min()) if not perf.empty else 'N/A',
        'v4_strategy_trade_rows': int(len(trade_table)),
        'v4_strategy_window_trade_rows': int(len(window_trades)) if 'window_trades' in locals() else 0,
        'v4_strategy_symbols': sorted([str(v) for v in trade_table.get('symbol', pd.Series(dtype='object')).dropna().unique()]),
        'v4_non_strategy_activity_excluded': True,
        'v4_modeled_5bps_formula_ok': formula_ok,
        'v4_strategy_source_tables': [V4_STRATEGY_EQUITY_TABLE, V4_STRATEGY_LEDGER_TABLE],
    }
    return perf, trade_table, summary_pairs, validation


def _draw_v4_strategy_performance_page(pdf, perf_frame, summary_pairs, footer=None):
    if perf_frame is None or getattr(perf_frame, 'empty', True):
        _draw_message_panel_page(
            pdf,
            '3b. V4 Strategy-Only Performance - SPY / QQQ / CASH',
            'No V4 strategy-only equity curve rows are available from the gold strategy equity table.',
            footer=footer,
            subtitle=V4_STRATEGY_COST_FOOTNOTE,
        )
        return

    fig = plt.figure(figsize=(12.8, 8.75))
    gs = fig.add_gridspec(1, 2, width_ratios=[2.35, 1.05], wspace=0.18)
    ax = fig.add_subplot(gs[0, 0])
    ax_summary = fig.add_subplot(gs[0, 1])
    ax_summary.axis('off')
    plot_frame = perf_frame.dropna(subset=['timestamp']).sort_values('timestamp').copy()
    ax.plot(plot_frame['timestamp'], plot_frame['gross_v4_return_pct'], linewidth=2.1, label='Gross V4 return', color='#17324D')
    ax.plot(plot_frame['timestamp'], plot_frame['captured_cost_net_return_pct'], linewidth=1.9, label='Net V4 return - captured conversion cost', color='#2E8B57')
    ax.plot(plot_frame['timestamp'], plot_frame['modeled_5bps_net_return_pct'], linewidth=1.9, linestyle='--', label='Net V4 return - modeled 5 bps per ETF fill', color='#B03A2E')
    ax.set_title('3b. V4 Strategy-Only Performance - SPY / QQQ / CASH', loc='left', fontweight='bold')
    ax.set_ylabel('Return since 2026-04-20 (%)')
    ax.set_xlim(left=pd.Timestamp(V4_STRATEGY_PERFORMANCE_START_DATE, tz=DISPLAY_TZ))
    apply_dual_time_axes(ax, bottom_label=f'Timestamp ({DISPLAY_TZ_LABEL})')
    ax.grid(alpha=0.3)
    ax.legend(loc='best', fontsize=8.5)

    ax_summary.text(0.0, 0.98, 'Summary metrics', fontsize=13.0, fontweight='bold', va='top')
    y = 0.925
    for metric, value in summary_pairs:
        metric_text = _wrap_report_cell_text(metric, max_chars=34)
        value_text = _wrap_report_cell_text(value, max_chars=30)
        ax_summary.text(0.0, y, metric_text, fontsize=8.1, color='#555555', fontweight='bold', va='top')
        ax_summary.text(0.0, y - 0.030, value_text, fontsize=8.9, color='#1F1F1F', va='top')
        y -= 0.074
        if y < 0.12:
            break

    fig.subplots_adjust(bottom=0.20, top=0.90)
    fig.text(0.055, 0.080, _wrap_report_cell_text(V4_STRATEGY_COST_FOOTNOTE, max_chars=165), fontsize=8.2, color='#4A4A4A', va='bottom')
    if footer:
        fig.text(0.055, 0.035, footer, fontsize=7.5, color='#6f6f6f', va='bottom')
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)


def _get_databricks_run_id():
    if 'dbutils' not in globals():
        return None
    candidate_keys = ['jobRunId', 'rootRunId', 'currentRunId', 'runId', 'taskRunId', 'jobId']
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        for attr_name in ['currentRunId', 'rootRunId']:
            try:
                option_value = getattr(context, attr_name)()
                if option_value.isDefined():
                    return str(option_value.get())
            except Exception:
                pass
        try:
            tags = context.tags()
            for key in candidate_keys:
                for accessor in ['get', 'apply']:
                    try:
                        value = getattr(tags, accessor)(key)
                        if hasattr(value, 'isDefined'):
                            if value.isDefined():
                                return str(value.get())
                        elif value is not None:
                            text = str(value)
                            if text and text.lower() not in {'none', 'null'}:
                                return text
                    except Exception:
                        pass
        except Exception:
            pass
        try:
            payload = json.loads(context.toJson())
            tags_payload = payload.get('tags', {}) if isinstance(payload, dict) else {}
            for key in candidate_keys:
                value = tags_payload.get(key) or payload.get(key)
                if value:
                    return str(value)
        except Exception:
            pass
    except Exception:
        return None
    return None

def _latest_table_timestamp(table_name, timestamp_candidates):
    if not table_exists(table_name):
        return None
    present = set(table_columns(table_name))
    columns = [col for col in timestamp_candidates if col in present]
    if not columns:
        return None
    expressions = ', '.join([f'MAX({col}) AS {col}' for col in columns])
    latest_ts = pd.NaT
    rows = query_to_pandas(f'SELECT {expressions} FROM {table_name}', empty_columns=columns)
    if not rows.empty:
        for col in columns:
            value = pd.to_datetime(rows.iloc[0].get(col), errors='coerce')
            if pd.notna(value) and (pd.isna(latest_ts) or value > latest_ts):
                latest_ts = value
    return None if pd.isna(latest_ts) else fmt_timestamp(latest_ts)


def _build_report_manifest(pdf_path, report_date):
    timestamp_candidates = {
        FEED_TABLE: ['feed_ts'],
        HEALTH_TABLE: ['rollup_ts'],
        PERF_TABLE: ['rollup_ts'],
        REGIME_SIGNAL_TABLE: ['timestamp'],
        SILVER_RT_TABLE: ['timestamp'],
        TRADE_LOG_TABLE: ['trade_ts', 'signal_ts'],
        LIVE_MACRO_TABLE: ['ingestion_ts', 'scheduled_release_ts', 'observed_timestamp'],
        MODEL_REGISTRY_TABLE: ['registered_at', 'updated_at', 'created_at'],
        V4_STRATEGY_EQUITY_TABLE: ['timestamp'],
        V4_STRATEGY_LEDGER_TABLE: ['broker_event_ts', 'event_ts', 'ingestion_ts'],
    }
    latest_source_timestamps = {
        table_name: _latest_table_timestamp(table_name, candidates)
        for table_name, candidates in timestamp_candidates.items()
    }
    return {
        'report_date_sgt': report_date,
        'generated_at': fmt_timestamp(pd.Timestamp.utcnow()),
        'databricks_run_id': _get_databricks_run_id(),
        'source_tables_used': list(timestamp_candidates.keys()),
        'latest_source_timestamps': latest_source_timestamps,
        'output_pdf_path': pdf_path,
        'report_mode': REPORT_MODE,
        'report_variant': REPORT_VARIANT,
        'timezone': REPORT_TIMEZONE,
    }


def _select_existing_columns(frame, columns):
    return frame[[c for c in columns if c in frame.columns]].copy()


def generate_daily_pdf_report():
    report_date = pd.Timestamp.now(tz=REPORT_TZ).strftime('%Y-%m-%d')
    is_circulation = REPORT_VARIANT == 'circulation'
    output_dir = REPORT_OUTPUT_PATH.rstrip('/')
    if is_circulation and os.path.basename(output_dir).lower().replace(' ', '_') != 'circulation_report':
        output_dir = f"{output_dir}/circulation_report"
    os.makedirs(output_dir, exist_ok=True)
    pdf_path = f"{output_dir}/{_report_pdf_basename(report_date)}"
    sidecar_path = pdf_path.replace('.pdf', '.json')
    manifest = _build_report_manifest(pdf_path, report_date)
    report_footer = f"Generated: {manifest['generated_at']} | run: {manifest['databricks_run_id'] or 'N/A'}"
    execution_mode = _as_text(latest_feed.get('execution_mode', latest_perf.get('execution_mode', 'N/A')))
    current_regime = _as_text(latest_feed.get('regime_name', 'N/A'))
    regime_probability_summary = probability_summary_text(latest_probability)
    probability_summary_line = f'Regime probabilities: {regime_probability_summary}' if regime_probability_summary else 'Regime probabilities: unavailable from dashboard/model history'
    current_probability_value = f'{prob_label}: {fmt_pct(prob_value)}' if pd.notna(_to_float(prob_value)) else None
    overall_cover_summary = [
        f'Model version: {_as_text(model_version, "N/A")}',
        f'Current regime: {current_regime}',
        probability_summary_line,
        f'Execution mode: {execution_mode}',
        f'Execution status: {report_execution_status}',
        f'Historical regime mix: Expansion {fmt_percent_points(latest_feed.get("pct_expansion"))}; Slowdown {fmt_percent_points(latest_feed.get("pct_slowdown"))}; Recession {fmt_percent_points(latest_feed.get("pct_recession"))}',
        f'Model target allocation: {report_model_target_allocation}',
        f'Actual Saxo holding: {report_actual_saxo_holding}',
        f'Position market value: {report_position_market_value}',
        f'Account equity: {fmt_money(report_account_snapshot.get("account_equity"))}',
        f'Freshness: {fmt_freshness_summary(latest_feed)}',
        f'Audit basis: latest model decision {fmt_timestamp_long(latest_feed.get("latest_audit_ts"))}',
    ]
    circulation_cover_summary = [
        f'Model version: {_as_text(model_version, "N/A")}',
        f'Current regime: {current_regime}',
        f'Target allocation: {report_model_target_allocation}',
        f'Actual holding: {report_actual_saxo_holding}',
        f'Account equity: {fmt_money(report_account_snapshot.get("account_equity"))}',
        f'Freshness: {fmt_freshness_summary(latest_feed)}',
        f'Execution note: {fmt_report_text(report_execution_status, max_chars=140)}',
    ]
    if regime_probability_summary:
        circulation_cover_summary.insert(2, f'Regime probabilities: {regime_probability_summary}')
    cover_summary = circulation_cover_summary if is_circulation else overall_cover_summary
    overall_kpi = [
        ('Current regime', current_regime),
        ('Current model probability', current_probability_value or 'Unavailable from dashboard/model history'),
        ('Model target allocation', report_model_target_allocation),
        ('Actual Saxo holding', report_actual_saxo_holding),
        ('Position market value', report_position_market_value),
        ('Account equity', fmt_money(report_account_snapshot.get('account_equity'))),
        ('Estimated uninvested equity', report_estimated_uninvested_equity),
        ('Execution mode', execution_mode),
    ]
    circulation_kpi = [
        ('Model version', _as_text(model_version, 'N/A')),
        ('Current regime', current_regime),
        ('Model target allocation', report_model_target_allocation),
        ('Actual Saxo holding', report_actual_saxo_holding),
        ('Account equity', fmt_money(report_account_snapshot.get('account_equity'))),
        ('Freshness', fmt_freshness_summary(latest_feed)),
    ]
    if current_probability_value:
        circulation_kpi.insert(2, ('Current probability', current_probability_value))
    market_status_subtitle = f"All timestamps shown in long-form {DISPLAY_TZ_LABEL} and EDT. Status is based on regular US equity session hours."

    with PdfPages(pdf_path) as pdf:
        _draw_cover_page(
            pdf,
            'V4 Strategy Institutional Report' + (' - Stakeholder Circulation' if is_circulation else ''),
            f'{_report_variant_label()} PDF | Report date (SGT): {report_date}',
            cover_summary,
            circulation_kpi if is_circulation else overall_kpi,
            footer=report_footer,
            kpi_cols=2,
        )

        _draw_table_page(
            pdf,
            '1. Market Status',
            market_status_df,
            max_rows=12,
            max_cols=2,
            footer=report_footer,
            subtitle=market_status_subtitle,
            center_cols={'Metric', 'Value'},
            bold_cols={'Metric'},
        )

        if 'upcoming_events_display' in globals() and upcoming_events_display is not None and not upcoming_events_display.empty:
            no_event_placeholder = _is_no_event_placeholder(upcoming_events_display)
            catalyst_columns = [
                'Date',
                'Time IST',
                'Time EDT',
                'Currency',
                'Importance',
                'Event',
                'Model relevance',
            ] if is_circulation else [
                'Date',
                'Time IST',
                'Time EDT',
                'Event name',
                'Mapped model feature(s)',
                'Source status',
                'Live capture status',
            ]
            if no_event_placeholder:
                _draw_message_panel_page(
                    pdf,
                    '2. Upcoming Economic Events / Model Catalyst Watch',
                    'No upcoming high importance economic calendar rows were available from dashboard-accessible calendar sources for the configured watch window. No model override is applied.',
                    footer=report_footer,
                    subtitle='This section is read-only and does not alter silver_rt_1h, model inference, signals, or trading.',
                )
            else:
                _draw_table_page(
                    pdf,
                    '2. Upcoming Economic Events / Model Catalyst Watch',
                    _select_existing_columns(globals().get('upcoming_events_compact_display', upcoming_events_display), catalyst_columns),
                    max_rows=10,
                    max_cols=7,
                    footer=report_footer,
                    subtitle='Hardcoded stakeholder watchlist. Stakeholder visibility only; no change to silver_rt_1h, model inference, signals, or trading.',
                    center_cols={'Date', 'Time IST', 'Time EDT', 'Currency', 'Importance'},
                    col_width_map={'Date': 0.9, 'Time IST': 0.72, 'Time EDT': 1.0, 'Currency': 0.65, 'Importance': 0.75, 'Event': 2.25, 'Model relevance': 1.55},
                    cell_wrap_map={'Event': 24, 'Model relevance': 22},
                    header_wrap_chars=14,
                    force_landscape=is_circulation,
                )
            if not is_circulation and not no_event_placeholder:
                _draw_table_page(
                    pdf,
                    '2a. Catalyst Detail - Unit/stage and Live Capture',
                    _select_existing_columns(upcoming_events_display, [
                        'Event name',
                        'Live capture status',
                        'Calendar source',
                    ]),
                    max_rows=10,
                    max_cols=3,
                    footer=report_footer,
                )

        if not perf_window.empty and {'rollup_ts', 'account_equity'}.issubset(perf_window.columns):
            perf_plot = perf_window.dropna(subset=['rollup_ts', 'account_equity']).sort_values('rollup_ts').copy()
            if not perf_plot.empty:
                perf_plot['account_equity'] = pd.to_numeric(perf_plot['account_equity'], errors='coerce')
                base = perf_plot['account_equity'].dropna().iloc[0]
                perf_plot['account_return'] = perf_plot['account_equity'] / base - 1.0 if base else np.nan
                fig, ax = plt.subplots(figsize=(9.6, 5.7))
                ax.plot(perf_plot['rollup_ts'], perf_plot['account_return'] * 100, linewidth=2.0, label='Actual Saxo account')
                ax.set_title('3. Actual Saxo Account Performance - may include account-level costs and non-V4 activity', loc='left', fontweight='bold')
                ax.set_ylabel('Return (%)')
                apply_dual_time_axes(ax, bottom_label=f'Timestamp ({DISPLAY_TZ_LABEL})')
                ax.grid(alpha=0.3)
                ax.legend(loc='best')
                fig.subplots_adjust(bottom=0.20, top=0.82)
                fig.text(0.12, 0.075, 'Actual Saxo Account Performance - may include account-level costs and non-V4 activity.', fontsize=9.2, color='#4A4A4A', va='bottom')
                fig.text(0.12, 0.035, report_footer, fontsize=7.5, color='#6f6f6f', va='bottom')
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)

        if is_circulation and not perf_recent_48h_window.empty and {'rollup_ts', 'account_equity'}.issubset(perf_recent_48h_window.columns):
            perf_48h_plot = perf_recent_48h_window.dropna(subset=['rollup_ts', 'account_equity']).sort_values('rollup_ts').copy()
            if not perf_48h_plot.empty:
                perf_48h_plot['account_equity'] = pd.to_numeric(perf_48h_plot['account_equity'], errors='coerce')
                base_48h = perf_48h_plot['account_equity'].dropna().iloc[0]
                perf_48h_plot['account_return'] = perf_48h_plot['account_equity'] / base_48h - 1.0 if base_48h else np.nan
                fig, ax = plt.subplots(figsize=(9.6, 5.7))
                ax.plot(perf_48h_plot['rollup_ts'], perf_48h_plot['account_return'] * 100, linewidth=2.0, color='#17324D')
                ax.set_title('3a. Actual Saxo Account Performance - Last 48 Hours', loc='left', fontweight='bold')
                ax.set_ylabel('Return (%)')
                apply_dual_time_axes(ax, bottom_label=f'Timestamp ({DISPLAY_TZ_LABEL})')
                ax.grid(alpha=0.3)
                fig.subplots_adjust(bottom=0.20)
                fig.text(0.12, 0.075, f'{perf_recent_48h_note}. May include account-level costs and non-V4 activity.', fontsize=9.0, color='#4A4A4A', va='bottom')
                fig.text(0.12, 0.035, report_footer, fontsize=7.5, color='#6f6f6f', va='bottom')
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)

        if is_circulation:
            v4_perf_frame, v4_trade_table, v4_summary_pairs, v4_validation = _v4_strategy_report_frames()
            manifest['v4_strategy_only_validation'] = v4_validation
            _draw_v4_strategy_performance_page(
                pdf,
                v4_perf_frame,
                v4_summary_pairs,
                footer=report_footer,
            )
            _draw_table_page(
                pdf,
                '3b. V4 Strategy-Only ETF Fill Ledger - SPY / QQQ only',
                v4_trade_table,
                max_rows=28,
                max_cols=14,
                footer=report_footer,
                subtitle=V4_STRATEGY_COST_FOOTNOTE,
                center_cols={'symbol', 'side', 'all_in_cost_confirmed'},
                long_ts_cols={'broker_event_ts'},
                force_landscape=True,
                font_size_override=5.2,
                header_wrap_chars=12,
                col_width_map={
                    'broker_event_ts': 2.15,
                    'strategy_stance_before': 1.05,
                    'strategy_stance_after': 1.05,
                    'symbol': 0.65,
                    'side': 0.65,
                    'quantity': 0.70,
                    'price': 0.70,
                    'gross_notional': 0.90,
                    'captured_conversion_cost': 1.05,
                    'modeled_5bps_cost': 1.00,
                    'cost_observation_quality': 1.65,
                    'all_in_cost_confirmed': 0.95,
                    'trade_id': 0.95,
                    'position_id': 0.95,
                },
                cell_wrap_map={'cost_observation_quality': 24, 'trade_id': 18, 'position_id': 18},
            )

        probability_history_subtitle = 'Probabilities are computed from the promoted model predict_proba over silver_rt_1h feature history.'
        _draw_probability_chart_page(
            pdf,
            '4. Historical Model Probabilities - Since 20-Apr-2026',
            probability_since_start_window,
            footer=report_footer,
            subtitle=probability_history_subtitle,
            dual_time_axis=is_circulation,
        )
        _draw_probability_chart_page(
            pdf,
            '4a. Historical Model Probabilities - Last 48 Hours',
            probability_rolling_window,
            footer=report_footer,
            subtitle=probability_rolling_note,
            dual_time_axis=is_circulation,
        )
        _draw_probability_chart_page(
            pdf,
            '4b. Model Probabilities - Latest Snapshot',
            latest_probability_rows.tail(1) if 'latest_probability_rows' in globals() else pd.DataFrame(),
            footer=report_footer,
            subtitle='Latest non-null model probability row.',
            dual_time_axis=False,
            latest_snapshot_ok=True,
        )

        if 'model_macro_input_audit_df' in globals() and not model_macro_input_audit_df.empty:
            if is_circulation:
                radar_specs = [spec for spec in MODEL_MACRO_INPUT_SPECS if spec['feature'] in latest_model_input_row.index]
                radar_labels = [spec['label'] for spec in radar_specs]
                radar_values = [max(min(_to_float(latest_model_input_row.get(spec['feature'])), 3.0), -3.0) if pd.notna(_to_float(latest_model_input_row.get(spec['feature']))) else 0.0 for spec in radar_specs]
                if radar_labels:
                    angles = np.linspace(0, 2 * np.pi, len(radar_labels), endpoint=False).tolist()
                    values = radar_values + radar_values[:1]
                    angles = angles + angles[:1]
                    fig = plt.figure(figsize=(8.5, 11.0))
                    ax = fig.add_axes([0.22, 0.27, 0.56, 0.50], polar=True)
                    ax.plot(angles, values, linewidth=2.0, color='#17324D')
                    ax.fill(angles, values, color='#A9BFCE', alpha=0.35)
                    ax.set_xticks(angles[:-1])
                    ax.set_xticklabels(radar_labels, fontsize=7.2)
                    ax.tick_params(axis='x', pad=5)
                    fig.text(0.075, 0.955, '5. Actual Macro Inputs Used By Model', ha='left', va='top', fontweight='bold', fontsize=14.0)
                    ax.set_ylim(-3, 3)
                    ax.set_yticks([-3, -1.5, 0, 1.5, 3])
                    ax.set_yticklabels(['-3', '-1.5', '0', '1.5', '3'])
                    zero_theta = np.linspace(0, 2 * np.pi, 361)
                    ax.plot(zero_theta, np.zeros_like(zero_theta), color='#566573', linewidth=1.15, linestyle='--', alpha=0.85)
                    fig.text(0.22, 0.185, f'Model timestamp: {fmt_timestamp_pair(model_input_ts)}', fontsize=8.2, color='#6f6f6f', ha='left')
                    fig.text(0.075, 0.065, report_footer, fontsize=7.5, color='#6f6f6f', ha='left')
                    pdf.savefig(fig)
                    plt.close(fig)
            macro_cols_primary = [
                'Feature',
                'Model feature',
                'Model z-score',
                'Model timestamp',
            ]
            macro_cols_values = [
                'Feature',
                'Model input value',
                'Resolved raw value used',
                'Public latest value',
                'Match status',
            ]
            macro_cols_detail = [
                'Feature',
                'Source',
                'Availability rule',
                'Observation date',
                'Release time',
            ] if is_circulation else [
                'Feature',
                'Source',
                'Availability rule',
                'Observation date',
                'Release time',
                'Evidence source',
                'Lag (hours)',
            ]
            _draw_table_page(
                pdf,
                '5. Actual Macro Inputs Used By Model - Features and Timestamps',
                _select_existing_columns(model_macro_input_audit_df, macro_cols_primary),
                max_rows=12,
                max_cols=4,
                footer=report_footer,
                subtitle='The model consumes z-scored silver inputs. Source and availability detail is separated from this page to keep model feature, z-score, and model timestamp readable.',
                center_cols={'Model z-score'},
                long_ts_cols={'Model timestamp'},
                force_landscape=is_circulation,
                col_width_map={'Feature': 1.35, 'Model feature': 1.45, 'Model z-score': 0.85, 'Model timestamp': 2.25},
                cell_wrap_map={'Feature': 20, 'Model feature': 20},
            )
            _draw_table_page(
                pdf,
                '5a. Actual Macro Inputs Used By Model - Values and Match Status',
                _select_existing_columns(model_macro_input_audit_df, macro_cols_values),
                max_rows=12,
                max_cols=5,
                footer=report_footer,
                center_cols={'Model input value', 'Resolved raw value used', 'Public latest value', 'Match status'},
            )
            _draw_table_page(
                pdf,
                '5b. Macro Input Detail',
                _select_existing_columns(model_macro_input_audit_df, macro_cols_detail),
                max_rows=12,
                max_cols=5 if is_circulation else 7,
                footer=report_footer,
                subtitle='Historical macro features use a conservative release-timing buffer in the training grid. Validated live overrides can be reflected by the next scheduled RT enrichment run when available.',
                center_cols={'Observation date', 'Release time', 'Lag (hours)'},
                long_ts_cols={'Release time'},
                col_width_map={'Feature': 1.25, 'Source': 1.35, 'Availability rule': 1.55, 'Observation date': 1.10, 'Release time': 1.95, 'Evidence source': 1.20, 'Lag (hours)': 0.70},
                cell_wrap_map={'Source': 20, 'Availability rule': 22, 'Evidence source': 18},
            )

        if 'current_feature_display' in globals():
            feature_page = current_feature_display.copy()
            feature_subtitle = 'Z-scores use a rolling 1-year lookback of 8760 hourly rows, approximately 365 days.'
            if is_circulation:
                feature_page = feature_page.rename(columns={'Model Importance (%)': 'Importance', 'Current z-score': 'Z-score'})
                for internal_col in ['Weighted move proxy', 'Delta vs prev hour']:
                    if internal_col in feature_page.columns:
                        feature_page = feature_page.drop(columns=[internal_col])
                feature_subtitle += ' Hourly delta and weighted move proxy columns are omitted in circulation format to keep the table readable.'
            else:
                feature_subtitle += ' Weighted move proxy equals model importance times the absolute latest hourly z-score move.'
            _draw_table_page(
                pdf,
                '6. Current Feature State',
                feature_page,
                max_rows=16,
                max_cols=5,
                footer=report_footer,
                subtitle=feature_subtitle,
                force_landscape=is_circulation,
                col_width_map={'Feature': 1.85, 'Model Importance (%)': 1.10, 'Importance': 1.0, 'Current z-score': 1.0, 'Z-score': 0.95, 'Delta vs prev hour': 1.10, 'Weighted move proxy': 1.10},
                header_wrap_chars=18,
            )

        if not is_circulation:
            _draw_table_page(
                pdf,
                '7. Latest Source Timestamps',
                pd.DataFrame([(_source_table_alias(name), ts) for name, ts in manifest['latest_source_timestamps'].items()], columns=['Source table', 'Latest timestamp']),
                max_rows=20,
                max_cols=2,
                footer=report_footer,
            )

        if not signal_session_history.empty:
            signal_cols = [c for c in ['Context', 'timestamp', 'regime_name', 'regime_int', 'position'] if c in signal_session_history.columns]
            signal_appendix = signal_session_history[signal_cols].sort_values('timestamp', ascending=False).head(16).copy()
            if 'timestamp' in signal_appendix.columns:
                split_ts = signal_appendix['timestamp'].apply(_safe_split_session_timestamp)
                signal_appendix['Timestamp (SGT)'] = [item[0] for item in split_ts]
                signal_appendix['Timestamp (EDT)'] = [item[1] for item in split_ts]
                signal_appendix = signal_appendix.drop(columns=['timestamp'])
            signal_appendix = signal_appendix.rename(columns={'regime_name': 'Regime', 'regime_int': 'Regime code', 'position': 'Model target allocation'})
            ordered_signal_cols = [c for c in ['Context', 'Timestamp (SGT)', 'Timestamp (EDT)', 'Regime', 'Regime code', 'Model target allocation'] if c in signal_appendix.columns]
            _draw_table_page(
                pdf,
                'Appendix A - Signals For Last Complete Trading Session',
                signal_appendix[ordered_signal_cols],
                max_rows=8,
                max_cols=6,
                footer=report_footer,
                subtitle=f'Completed New York session: {completed_session_label}. Open context targets one hour before the first completed-session row when available.',
                center_cols=set(ordered_signal_cols),
                col_width_map={'Context': 1.45, 'Timestamp (SGT)': 2.10, 'Timestamp (EDT)': 2.10, 'Regime': 0.95, 'Regime code': 0.95, 'Model target allocation': 1.05},
                cell_wrap_map={'Context': 18},
            )

        if not trade_session_history.empty:
            trade_cols = [c for c in ['Context', 'trade_ts', 'regime_name', 'target_position', 'target_ticker', 'hourly_pnl', 'execution_mode', 'orders_placed', 'last_skip_reason', 'last_error_reason'] if c in trade_session_history.columns]
            session_day_pnl = pd.to_numeric(trade_completed_session_history.get('hourly_pnl'), errors='coerce').fillna(0).sum() if 'hourly_pnl' in trade_completed_session_history.columns else np.nan
            session_day_pnl_pct = pd.to_numeric(trade_completed_session_history.get('hourly_pnl_pct'), errors='coerce').fillna(0).sum() if 'hourly_pnl_pct' in trade_completed_session_history.columns else np.nan
            trade_appendix = trade_session_history[trade_cols].sort_values('trade_ts', ascending=False).head(16).copy()
            if 'trade_ts' in trade_appendix.columns:
                split_ts = trade_appendix['trade_ts'].apply(_safe_split_session_timestamp)
                trade_appendix['Trade timestamp (SGT)'] = [item[0] for item in split_ts]
                trade_appendix['Trade timestamp (EDT)'] = [item[1] for item in split_ts]
                trade_appendix = trade_appendix.drop(columns=['trade_ts'])
            
            if any(c in trade_appendix.columns for c in ['orders_placed', 'last_skip_reason', 'last_error_reason']):
                trade_appendix['Execution status'] = trade_appendix.apply(summarize_execution_status, axis=1)
                trade_appendix = trade_appendix.drop(columns=[c for c in ['orders_placed', 'last_skip_reason', 'last_error_reason'] if c in trade_appendix.columns])
            trade_appendix = trade_appendix.rename(columns={'regime_name': 'Regime', 'target_position': 'Model target allocation', 'target_ticker': 'Target ticker', 'hourly_pnl': 'Hourly P&L', 'execution_mode': 'Execution mode'})
            ordered_trade_cols = [c for c in ['Context', 'Trade timestamp (SGT)', 'Trade timestamp (EDT)', 'Regime', 'Model target allocation', 'Target ticker', 'Hourly P&L', 'Execution mode', 'Execution status'] if c in trade_appendix.columns]
            _draw_table_page(
                pdf,
                'Appendix B - Trades For Last Complete Trading Session',
                trade_appendix[ordered_trade_cols],
                max_rows=8,
                max_cols=8,
                footer=report_footer,
                subtitle=f'Completed New York session: {completed_session_label}. Open context targets one hour before the first completed-session row when available. Calculated session day P&L: {fmt_signed_money(session_day_pnl)} ({fmt_optional_percent(session_day_pnl_pct)})',
                center_cols=set(ordered_trade_cols),
                col_width_map={'Context': 1.55, 'Trade timestamp (SGT)': 2.05, 'Trade timestamp (EDT)': 2.05, 'Regime': 0.85, 'Model target allocation': 0.95, 'Target ticker': 0.85, 'Hourly P&L': 0.85, 'Execution mode': 0.85},
                cell_wrap_map={'Context': 18},
                header_wrap_chars=12,
            )

    with open(sidecar_path, 'w') as handle:
        json.dump(manifest, handle, indent=2, sort_keys=True)
    print(f'Daily PDF written to: {pdf_path}')
    print(f'Daily PDF sidecar written to: {sidecar_path}')
    if 'v4_strategy_only_validation' in manifest:
        print('V4 strategy-only validation:')
        print(json.dumps(manifest['v4_strategy_only_validation'], indent=2, sort_keys=True))
    return pdf_path, sidecar_path, manifest


if REPORT_MODE == 'daily_pdf':
    DAILY_PDF_OUTPUT_PATH, DAILY_PDF_SIDECAR_PATH, DAILY_PDF_MANIFEST = generate_daily_pdf_report()
